# Solar Wind Prediction — V3.5 11-Member Focused Elite Ensemble

**목표:** V8의 실제 champion 학습 순서를 그대로 복원하고, V3.3에서 실제 앙상블 기여가 확인된 세 specialist만 추가해 **더 강하고 더 재현 가능한 정예 앙상블**을 만든다.

- 대회 제공 Train만 학습에 사용, Validation은 모델/앙상블 선택에만 사용
- 외부 데이터 / pretrained weight 미사용
- 첫 4개 member는 V8 원래 순서 `physics777 → physics778 → wide779 → wide780`를 그대로 학습
- V8 champion은 원래와 동일하게 `physics778 + wide780` 1:1
- 추가 specialist는 V3.3에서 실제 선택된 `Bright-Dark Boundary`, `Physics LT`, `CH Topology`
- 2개 완료 시 응급 milestone 저장, 4개 V8 anchor 완료 시 V8 champion milestone 저장
- 이후 specialist 하나가 끝날 때마다 앙상블 재탐색 후 같은 외부 경로에 덮어쓰기
- 새 앙상블이 현재 V8 champion보다 충분히 좋아지지 않으면 자동으로 V8 champion 유지
- `train_state.pth` / `current_member.pth`는 rolling overwrite
- 모든 결과는 `outputs/v3.5`에만 저장하며 `/submission`은 자동으로 건드리지 않음

> V3.3의 11-member 실험은 신호 탐색용이었다면, V3.5는 **실전 점수용 정예 압축 버전**이다.

> **V3.5 sprint:** V8 원본 4개를 재현한 뒤 V3.3에서 의미가 확인된 7개 specialist만 추가하여 2시간 안에 탐색합니다.


In [ ]:
from pathlib import Path
import gc
import json
import math
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import copy
import itertools
import random
import time
import traceback
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

# ============================================================
# V3.5 execution / safety
# ============================================================
RUN_STARTED_AT = time.time()

TRAIN_MODEL = True
# V3.5은 외부 작업 경로에만 저장합니다. /submission은 자동으로 건드리지 않습니다.
VERIFY_EXISTING_OUTPUT = True
RESUME_PARTIAL = True

EXPECTED_COUNTS = {"train": 9607, "validation": 1199, "test": 3868}
ARCHITECTURE_VERSION = "v3.5_11member_2hour_elite_sprint_r1"
DELIVERY_FORMAT_VERSION = 1

# 2시간 sprint 보호:
# 전체 wallclock 1.92 h 안에서 끝내고 마지막 10분은
# ensemble/Test/파일 저장에 강제로 예약합니다.
MAX_TOTAL_RUNTIME_HOURS = 1.92
FINALIZATION_RESERVE_MINUTES = 10
TRAINING_CUTOFF_AT = (
    RUN_STARTED_AT
    + MAX_TOTAL_RUNTIME_HOURS * 3600
    - FINALIZATION_RESERVE_MINUTES * 60
)
ABSOLUTE_FINISH_TARGET_AT = RUN_STARTED_AT + MAX_TOTAL_RUNTIME_HOURS * 3600
MIN_PARTIAL_MEMBER_EPOCHS = 8
CURRENT_CHECKPOINT_EVERY_EPOCHS = 2
DELIVERY_MILESTONES = (2, 4, 5, 6, 7, 8, 9, 10, 11)

SEED = 777

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

def resolve_data_root():
    candidates = [
        Path("public_dataset/competition_dataset_6h"),
        Path("../public_dataset/competition_dataset_6h"),
        Path("data/competition_dataset_6h"),
    ]
    for candidate in candidates:
        if (candidate / "dataset_info.json").exists():
            return candidate
    raise FileNotFoundError(
        "competition_dataset_6h 폴더를 찾지 못했습니다. DATA_ROOT를 직접 지정하세요."
    )

DATA_ROOT = resolve_data_root()

TEAM_MEMBER = "sunyeok"
VERSION = "v3.5"
# 데이터셋/제출 폴더와 분리된 외부 작업 경로.
OUTPUT_DIR = Path.home() / TEAM_MEMBER / VERSION
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CHECKPOINT_PATH = OUTPUT_DIR / "model.pth"          # 항상 추론 가능한 최신 milestone/final
OUTPUT_SUBMISSION_PATH = OUTPUT_DIR / "submission.csv"    # 위 model.pth와 항상 짝
TRAIN_STATE_PATH = OUTPUT_DIR / "train_state.pth"         # 완료 member 누적, 덮어쓰기
CURRENT_MEMBER_PATH = OUTPUT_DIR / "current_member.pth"   # 현재 member epoch resume, 덮어쓰기
OUTPUT_LOG_PATH = OUTPUT_DIR / "run_status.json"

# inference-only 재현 검증도 외부 작업 경로의 최신 pair만 사용합니다.
ACTIVE_CHECKPOINT_PATH = OUTPUT_CHECKPOINT_PATH
ACTIVE_CSV_PATH = OUTPUT_SUBMISSION_PATH

CACHE_ROOT = Path.home() / "solar_wind_resized_cache"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

# ============================================================
# proven v8 base settings
# ============================================================
IMAGE_SIZE = 64
CHANNELS = ("193", "211")
RMSE_EPSILON = 1e-8
BATCH_SIZE = 256
NUM_WORKERS = 2  # 4보다 약간 보수적으로: shared server 안정성 우선
WIND_FEATURE_DIM = 39

TARGET_HORIZONS = 12
HORIZON_EMBED_DIM = 32
IMAGE_ATTENTION_DIM = 64
WIND_GRU_HIDDEN = 32
WIND_ATTENTION_DIM = 32
HORIZON_DECODER_HIDDEN = 128

CORONAL_FEATURES_PER_BAND = len(CHANNELS) * 3 * 2  # channel×threshold×zone = 12
CORONAL_BAND_TOKEN_DIM = 24
CORONAL_BAND_ATTENTION_DIM = 32
ROTATION_TRACK_DIM = 64
AUX_CONTEXT_DIM = 64
REGIME_CONTEXT_DIM = 48

BASELINE_V8_VALIDATION_RMSE = 63.50648601752991
BASELINE_V8_PUBLIC_RMSE = 56.5385

# ============================================================
# regularization
# ============================================================
DROPOUT = 0.3
CONV_DROPOUT = 0.1
WEIGHT_DECAY = 1e-2
MIXUP_ALPHA = 0.2
BRIGHTNESS_JITTER = 0.2
FRAME_DROP_PROBABILITY = 0.1

# 동-서 반전은 solar rotation 방향을 뒤집으므로 사용하지 않음
FLIP_PROBABILITY = 0.0
VFLIP_PROBABILITY = 0.5

# ============================================================
# coronal-hole physics features (official images only)
# ============================================================
CH_BANDS = 6
CH_THRESHOLDS = (0.25, 0.40, 0.55)
CH_EQUATOR_FRACTION = 0.5
DISK_SHRINK = 0.90
CH_ZONES = 2
CH_FEATURE_PER_STEP = (
    len(CHANNELS) * len(CH_THRESHOLDS) * CH_ZONES * CH_BANDS
)
POOL_HEIGHT = 2
POOL_WIDTH = 4
RECENT_STEPS = 8  # 48 h

# ============================================================
# 11-member two-hour elite sprint
#
# 0~3은 V8 원래 학습 순서를 그대로 재현한다.
# 그 뒤 V3.3 결과에서 단독 성능/상위 앙상블 기여가 확인된
# specialist 7개만 중요도 순서로 추가한다.
# ============================================================
ENSEMBLE_VARIANTS = (
    "physics",                 # 0 V8 Physics 1, seed 777
    "physics",                 # 1 V8 Physics 2, seed 778 <- champion
    "wide",                    # 2 V8 Wide 1, seed 779
    "wide",                    # 3 V8 Wide 2, seed 780    <- champion
    "ch_topology",             # 4 strongest new single
    "bright_dark_boundary",    # 5 strongest complement
    "hss_peak",                # 6 near-tied top ensemble alternative
    "physics_lt",              # 7 longitude x time complement
    "ratio_temporal",          # 8 193/211 temporal ratio
    "east_west",               # 9 strong individual
    "trajectory",              # 10 strong individual / curve prior
)
MEMBER_LABELS = (
    "V8 Physics 1",
    "V8 Physics 2",
    "V8 Wide 1",
    "V8 Wide 2",
    "CH Topology",
    "Bright-Dark Boundary",
    "HSS Peak",
    "Physics LT",
    "193/211 Ratio Temporal",
    "East-West",
    "Trajectory",
)
ENSEMBLE_SIZE = len(ENSEMBLE_VARIANTS)
# Specialist seeds are exactly the seeds used in the successful V3.3 run.
MEMBER_SEEDS = (777, 778, 779, 780, 789, 784, 792, 787, 785, 786, 798)
assert len(MEMBER_SEEDS) == ENSEMBLE_SIZE

# Historical V8 Public-best champion pair.
CHAMPION_MEMBERS = (1, 3)
SPECIALIST_MEMBERS = tuple(range(4, ENSEMBLE_SIZE))

# 338 planned epochs. At the observed ~17 s/epoch this is about 96 min,
# leaving a protected finalization margin inside the 2-hour sprint.
MEMBER_EPOCHS = (
    40, 40, 40, 40,   # exact V8 replay order/budget
    24,                # CH Topology
    26,                # Bright-Dark Boundary
    24,                # HSS Peak
    26,                # Physics LT
    26,                # 193/211 Ratio Temporal
    26,                # East-West
    26,                # Trajectory
)
assert len(MEMBER_EPOCHS) == ENSEMBLE_SIZE

WARMUP_EPOCHS = 3
GRAD_CLIP_NORM = 5.0
EMA_DECAY = 0.995

# ============================================================
# domain-specialist losses
# normalized wind units: 0.500 == 500 km/s
# ============================================================
NEAR_CONTINUITY_HORIZON_WEIGHTS = np.asarray(
    [1.80, 1.65, 1.45, 1.20, 0.95, 0.80, 0.70, 0.65, 0.60, 0.55, 0.55, 0.55],
    dtype=np.float32,
)
NEAR_CONTINUITY_HORIZON_WEIGHTS /= NEAR_CONTINUITY_HORIZON_WEIGHTS.mean()

SOLAR_LONG_HORIZON_WEIGHTS = np.asarray(
    [0.50, 0.55, 0.60, 0.70, 0.80, 0.90, 1.00, 1.15, 1.30, 1.50, 1.70, 1.90],
    dtype=np.float32,
)
SOLAR_LONG_HORIZON_WEIGHTS /= SOLAR_LONG_HORIZON_WEIGHTS.mean()

# Regime-aware gates
ONSET_DELTA_CENTER = 0.080     # +80 km/s
ONSET_DELTA_WIDTH = 0.040
HSS_SPEED_CENTER = 0.500
HSS_SPEED_WIDTH = 0.045
RECOVERY_DELTA_CENTER = 0.080
RECOVERY_DELTA_WIDTH = 0.040
SLOW_WIND_CENTER = 0.450
SLOW_WIND_WIDTH = 0.050
REGIME_CHANGE_CENTER = 0.100   # |change| ≈ 100 km/s
REGIME_CHANGE_WIDTH = 0.050

# 형태/상태 specialist는 sample mixing을 약화/제거해 물리적 패턴을 보존
MIXUP_ALPHA_BY_VARIANT = {
    "active_region_edge": 0.10,
    "bright_dark_boundary": 0.10,
    "ratio_temporal": 0.10,
    "ch_topology": 0.10,
    "stream_onset": 0.0,
    "hss_peak": 0.0,
    "hss_duration": 0.0,
    "hss_recovery": 0.0,
    "slow_wind": 0.0,
    "regime_change": 0.0,
}

# ============================================================
# finite ensemble search
#
# V8 champion 근처의 작은 specialist 가중치도 놓치지 않도록
# pair는 0.05 간격, triple/quad는 0.10 simplex로 탐색한다.
# Validation overfit 방지를 위해 최종 support는 최대 4개,
# 두 validation half에서 한쪽만 크게 악화되는 후보는 배제한다.
# ============================================================
PAIR_WEIGHT_GRID = tuple(np.round(np.arange(0.05, 1.00, 0.05), 2))
SIMPLEX_UNITS = 10  # 0.10 step for triple / quad
CHAMPION_SPECIALIST_MASS_GRID = tuple(
    np.round(np.arange(0.05, 0.36, 0.05), 2)
)
CHAMPION_TWO_SPECIALIST_MASS_GRID = (0.10, 0.20, 0.30, 0.40)

MIN_ENSEMBLE_GAIN_KM_S = 0.08
HALF_SPLIT_WORSE_TOLERANCE_KM_S = 0.20
MAX_SELECTED_MEMBERS = 4
USE_TTA = False

REPRODUCTION_RTOL = 1e-5
REPRODUCTION_ATOL_KM_S = 1e-4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
PIN_MEMORY = DEVICE.type == "cuda"

if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = bool(TRAIN_MODEL)
    torch.backends.cudnn.deterministic = not TRAIN_MODEL
else:
    print("WARNING: CUDA unavailable — full training is intended for A100.")

if not TRAIN_MODEL and hasattr(torch, "use_deterministic_algorithms"):
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except TypeError:
        torch.use_deterministic_algorithms(True)

print("PyTorch:", torch.__version__)
print("device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("data:", DATA_ROOT.resolve())
print("version:", VERSION)
print("output:", OUTPUT_DIR.resolve())
print("members:", ENSEMBLE_SIZE)
print("planned epochs:", sum(MEMBER_EPOCHS), "| per member:", MEMBER_EPOCHS)
print("workers:", NUM_WORKERS, "| batch:", BATCH_SIZE)
print("training cutoff:", time.strftime("%H:%M:%S", time.localtime(TRAINING_CUTOFF_AT)))
print("target max wallclock:", MAX_TOTAL_RUNTIME_HOURS, "hours")
print("baseline v8 val/public:", BASELINE_V8_VALIDATION_RMSE, BASELINE_V8_PUBLIC_RMSE)

def safe_torch_load(path, map_location="cpu"):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"checkpoint가 없습니다: {path}")
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except Exception:
        return torch.load(path, map_location=map_location, weights_only=False)

def atomic_torch_save(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)

def atomic_json_save(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=2)
    os.replace(temporary, path)


def elapsed_seconds():
    return time.time() - RUN_STARTED_AT

def training_seconds_left():
    return TRAINING_CUTOFF_AT - time.time()

def format_duration(seconds):
    seconds = max(0.0, float(seconds))
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"

def validate_delivery_checkpoint(checkpoint):
    required = {
        "architecture_version", "delivery_format_version",
        "model_state_dicts", "variants", "labels", "seeds",
        "original_member_indices", "ensemble_weights",
        "validation_rmse_km_s", "ensemble_strategy",
        "ensemble_search_count", "completed_member_count",
        "planned_member_count", "member_epochs_actual",
        "channels", "wind_feature_dim", "dropout", "conv_dropout",
        "image_size", "input_steps", "target_horizons",
        "pool_height", "pool_width", "decoder_config",
        "coronal_feature_config", "coronal_geometry",
        "prediction_mode", "use_tta", "initialization",
        "external_data_used", "pretrained_weights_used",
        "train_validation_combined",
    }
    missing = sorted(required - set(checkpoint))
    assert not missing, f"delivery checkpoint metadata 누락: {missing}"
    assert checkpoint["architecture_version"] == ARCHITECTURE_VERSION
    assert int(checkpoint["delivery_format_version"]) == DELIVERY_FORMAT_VERSION
    assert int(checkpoint["planned_member_count"]) == ENSEMBLE_SIZE
    assert 2 <= int(checkpoint["completed_member_count"]) <= ENSEMBLE_SIZE
    assert tuple(checkpoint["channels"]) == CHANNELS
    assert int(checkpoint["wind_feature_dim"]) == WIND_FEATURE_DIM
    assert int(checkpoint["image_size"]) == IMAGE_SIZE
    assert int(checkpoint["input_steps"]) == 20
    assert int(checkpoint["target_horizons"]) == TARGET_HORIZONS
    assert int(checkpoint["pool_height"]) == POOL_HEIGHT
    assert int(checkpoint["pool_width"]) == POOL_WIDTH
    assert bool(checkpoint["use_tta"]) == USE_TTA
    assert checkpoint["initialization"] == "random_from_scratch_v34_v8_replay_elite"
    assert checkpoint["prediction_mode"] == (
        "persistence_plus_linear_ar_plus_specialist_horizon_delta_scaled_by_1000"
    )
    assert checkpoint["external_data_used"] is False
    assert checkpoint["pretrained_weights_used"] is False
    assert checkpoint["train_validation_combined"] is False

    states = checkpoint["model_state_dicts"]
    variants = checkpoint["variants"]
    labels = checkpoint["labels"]
    seeds = checkpoint["seeds"]
    original_indices = checkpoint["original_member_indices"]
    weights = np.asarray(checkpoint["ensemble_weights"], dtype=np.float64)
    selected_size = len(states)

    assert selected_size >= 2
    assert len(variants) == selected_size
    assert len(labels) == selected_size
    assert len(seeds) == selected_size
    assert len(original_indices) == selected_size
    assert weights.shape == (selected_size,)
    assert np.isfinite(weights).all()
    assert (weights > 0).all()
    assert np.isclose(weights.sum(), 1.0, rtol=0.0, atol=1e-8)
    assert np.isfinite(float(checkpoint["validation_rmse_km_s"]))
    assert all(isinstance(state, dict) and state for state in states)

    for local_index, original_index in enumerate(original_indices):
        original_index = int(original_index)
        assert 0 <= original_index < int(checkpoint["completed_member_count"])
        assert variants[local_index] == ENSEMBLE_VARIANTS[original_index]
        assert labels[local_index] == MEMBER_LABELS[original_index]
        assert int(seeds[local_index]) == MEMBER_SEEDS[original_index]

    decoder = checkpoint["decoder_config"]
    assert int(decoder["horizon_embed_dim"]) == HORIZON_EMBED_DIM
    assert int(decoder["image_attention_dim"]) == IMAGE_ATTENTION_DIM
    assert int(decoder["wind_gru_hidden"]) == WIND_GRU_HIDDEN
    assert int(decoder["wind_attention_dim"]) == WIND_ATTENTION_DIM
    assert int(decoder["horizon_decoder_hidden"]) == HORIZON_DECODER_HIDDEN
    return checkpoint

loaded_checkpoint = None
if not TRAIN_MODEL:
    loaded_checkpoint = safe_torch_load(ACTIVE_CHECKPOINT_PATH, map_location="cpu")
    validate_delivery_checkpoint(loaded_checkpoint)
    print("TRAIN_MODEL=False: delivery checkpoint loaded:", ACTIVE_CHECKPOINT_PATH)


## 1. Data load + 규정/누수 검증


In [ ]:
IMAGE_COLUMNS = [f"image_{index:02d}" for index in range(20)]
WIND_COLUMNS = [f"wind_{index:02d}" for index in range(20)]
TARGET_COLUMNS = [f"target_{index:02d}" for index in range(12)]

EXPECTED_INPUT_COLUMNS = ["sample_id"] + IMAGE_COLUMNS + WIND_COLUMNS
EXPECTED_TARGET_COLUMNS = ["sample_id"] + TARGET_COLUMNS
EXPECTED_SUBMISSION_COLUMNS = ["sample_id"] + TARGET_COLUMNS


def assert_input_schema(frame, split):
    assert len(frame.columns) == 41, (
        f"{split}/inputs.csv columns={len(frame.columns)}, expected=41"
    )

    assert set(frame.columns) == set(EXPECTED_INPUT_COLUMNS), (
        f"{split}/inputs.csv schema mismatch:\n"
        f"actual={frame.columns.tolist()}\n"
        f"expected={EXPECTED_INPUT_COLUMNS}"
    )

    assert frame["sample_id"].is_unique, (
        f"{split}: sample_id duplicate"
    )

    assert frame[IMAGE_COLUMNS].notna().all().all(), (
        f"{split}: image filename missing"
    )

    assert np.isfinite(
        frame[WIND_COLUMNS].to_numpy(np.float32)
    ).all(), (
        f"{split}: wind contains NaN/Inf"
    )


# ============================================================
# dataset_info.json
# ============================================================
# dataset_info.json의 내부 schema는 대회 배포 버전에 따라
# counts/train/validation/test 형태가 아닐 수 있습니다.
#
# 따라서 metadata는 참고용으로만 읽고,
# 실제 split 크기는 각 CSV의 행 수를 직접 확인합니다.
# ============================================================

info_path = DATA_ROOT / "dataset_info.json"

try:
    info = json.loads(
        info_path.read_text(encoding="utf-8")
    )
except Exception as exc:
    warnings.warn(
        f"dataset_info.json을 읽지 못했습니다: {exc}\n"
        "실제 CSV를 기준으로 데이터 검증을 계속합니다."
    )
    info = {}


info_counts = {}

if isinstance(info, dict):
    candidate_counts = info.get("counts", {})

    if isinstance(candidate_counts, dict):
        info_counts = candidate_counts


# dataset_info.json에 해당 count가 실제로 있을 때만 consistency 확인.
# 없는 key 때문에 학습을 중단하지 않습니다.
for split, expected_count in EXPECTED_COUNTS.items():

    if split not in info_counts:
        print(
            f"[INFO] dataset_info.json에 '{split}' count가 없습니다. "
            f"실제 CSV 행 수로 검증합니다."
        )
        continue

    actual_count = int(info_counts[split])

    assert actual_count == expected_count, (
        f"dataset_info {split}={actual_count}, "
        f"expected={expected_count}"
    )


# ============================================================
# Test
# ============================================================

test_inputs = pd.read_csv(
    DATA_ROOT / "test/inputs.csv"
)

test_ids = pd.read_csv(
    DATA_ROOT / "test/test_ids.csv"
)


assert_input_schema(
    test_inputs,
    "test",
)

assert test_ids.columns.tolist() == ["sample_id"], (
    f"test_ids.csv columns mismatch: {test_ids.columns.tolist()}"
)

# 실제 CSV 기준으로 sample count를 엄격하게 검증
assert len(test_inputs) == EXPECTED_COUNTS["test"], (
    f"test inputs={len(test_inputs)}, "
    f"expected={EXPECTED_COUNTS['test']}"
)

assert len(test_ids) == EXPECTED_COUNTS["test"], (
    f"test ids={len(test_ids)}, "
    f"expected={EXPECTED_COUNTS['test']}"
)

assert (
    test_inputs["sample_id"].tolist()
    == test_ids["sample_id"].tolist()
), "test inputs와 test_ids의 sample_id 순서가 다릅니다."

assert not any(
    column.startswith("target_")
    for column in test_inputs.columns
), "test inputs에 target column이 존재합니다."


test_index = np.arange(
    len(test_inputs),
    dtype=np.int64,
)


# ============================================================
# Train / Validation
# ============================================================

train_inputs = None
train_targets_frame = None
train_targets = None
train_index = None

val_inputs = None
val_targets_frame = None
val_targets = None
val_index = None


if TRAIN_MODEL:

    train_inputs = pd.read_csv(
        DATA_ROOT / "train/inputs.csv"
    )

    train_targets_frame = pd.read_csv(
        DATA_ROOT / "train/targets.csv"
    )

    val_inputs = pd.read_csv(
        DATA_ROOT / "validation/inputs.csv"
    )

    val_targets_frame = pd.read_csv(
        DATA_ROOT / "validation/targets.csv"
    )


    # ----------------------------
    # schema
    # ----------------------------

    assert_input_schema(
        train_inputs,
        "train",
    )

    assert_input_schema(
        val_inputs,
        "validation",
    )


    assert (
        train_targets_frame.columns.tolist()
        == EXPECTED_TARGET_COLUMNS
    ), (
        "train/targets.csv schema mismatch:\n"
        f"{train_targets_frame.columns.tolist()}"
    )

    assert (
        val_targets_frame.columns.tolist()
        == EXPECTED_TARGET_COLUMNS
    ), (
        "validation/targets.csv schema mismatch:\n"
        f"{val_targets_frame.columns.tolist()}"
    )


    # ----------------------------
    # 실제 CSV sample count
    # ----------------------------

    assert len(train_inputs) == EXPECTED_COUNTS["train"], (
        f"train inputs={len(train_inputs)}, "
        f"expected={EXPECTED_COUNTS['train']}"
    )

    assert len(train_targets_frame) == EXPECTED_COUNTS["train"], (
        f"train targets={len(train_targets_frame)}, "
        f"expected={EXPECTED_COUNTS['train']}"
    )

    assert len(val_inputs) == EXPECTED_COUNTS["validation"], (
        f"validation inputs={len(val_inputs)}, "
        f"expected={EXPECTED_COUNTS['validation']}"
    )

    assert len(val_targets_frame) == EXPECTED_COUNTS["validation"], (
        f"validation targets={len(val_targets_frame)}, "
        f"expected={EXPECTED_COUNTS['validation']}"
    )


    # ----------------------------
    # input ↔ target ID alignment
    # ----------------------------

    assert (
        train_inputs["sample_id"].tolist()
        == train_targets_frame["sample_id"].tolist()
    ), "Train inputs/targets sample_id 순서 불일치"

    assert (
        val_inputs["sample_id"].tolist()
        == val_targets_frame["sample_id"].tolist()
    ), "Validation inputs/targets sample_id 순서 불일치"


    # ----------------------------
    # split leakage 방지
    # ----------------------------

    train_ids_set = set(
        train_inputs["sample_id"]
    )

    val_ids_set = set(
        val_inputs["sample_id"]
    )

    test_ids_set = set(
        test_inputs["sample_id"]
    )


    assert train_ids_set.isdisjoint(val_ids_set), (
        "Train / Validation sample_id overlap 발견"
    )

    assert train_ids_set.isdisjoint(test_ids_set), (
        "Train / Test sample_id overlap 발견"
    )

    assert val_ids_set.isdisjoint(test_ids_set), (
        "Validation / Test sample_id overlap 발견"
    )


    # ----------------------------
    # target
    # ----------------------------

    train_targets = (
        train_targets_frame[TARGET_COLUMNS]
        .to_numpy(np.float32)
    )

    val_targets = (
        val_targets_frame[TARGET_COLUMNS]
        .to_numpy(np.float32)
    )


    assert np.isfinite(train_targets).all(), (
        "Train target에 NaN/Inf 존재"
    )

    assert np.isfinite(val_targets).all(), (
        "Validation target에 NaN/Inf 존재"
    )


    train_index = np.arange(
        len(train_inputs),
        dtype=np.int64,
    )

    val_index = np.arange(
        len(val_inputs),
        dtype=np.int64,
    )


    print()
    print("========================================")
    print("DATASET VALIDATION PASSED")
    print("========================================")
    print(
        f"Train      : {len(train_index):,}"
    )
    print(
        f"Validation : {len(val_index):,}"
    )
    print(
        f"Test       : {len(test_index):,}"
    )
    print("Train/Validation/Test ID overlap: NONE")
    print("Input/Target alignment: OK")
    print("NaN/Inf: NONE")
    print("========================================")


else:

    print()
    print("========================================")
    print("INFERENCE-ONLY DATASET VALIDATION PASSED")
    print("========================================")
    print(
        f"Test : {len(test_index):,}"
    )
    print("========================================")


# dataset_info.json은 디버깅 참고용으로만 출력
print()
print("[dataset_info.json counts]")
if info_counts:
    print(
        json.dumps(
            info_counts,
            ensure_ascii=False,
            indent=2,
        )
    )
else:
    print(
        "counts field 없음 "
        "(정상: 실제 CSV row count로 검증 완료)"
    )

## 2. 64×64 image cache + 공식 이미지 기반 coronal feature cache


In [ ]:
def prepare_image_memmap(split, inputs):
    # 같은 inputs.csv로 만든 정상 cache가 있으면 PNG 전처리를 반복하지 않습니다.
    # DATA_ROOT(대회 제공 데이터)는 read-only로 마운트되어 있을 수 있고,
    # /submission과 분리하기 위해 CACHE_ROOT를 따로 씁니다.
    image_root = DATA_ROOT / split
    cache_root = CACHE_ROOT / f"{IMAGE_SIZE}px"
    cache_root.mkdir(parents=True, exist_ok=True)
    array_path = cache_root / f"{split}_images.npy"
    metadata_path = cache_root / f"{split}_metadata.json"
    # 여러 sample에서 반복되는 PNG는 고유 파일당 한 번만 resize해 저장합니다.
    filenames = sorted(pd.unique(inputs[IMAGE_COLUMNS].to_numpy().ravel()).tolist())
    expected = {
        "dataset_root": str(DATA_ROOT.resolve()),
        "split": split,
        "image_size": IMAGE_SIZE,
        "channels": list(CHANNELS),
        "filenames": filenames,
    }
    valid = False
    if array_path.exists() and metadata_path.exists():
        try:
            metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
            cached = np.load(array_path, mmap_mode="r")
            valid = metadata == expected and cached.shape == (
                len(filenames), len(CHANNELS), IMAGE_SIZE, IMAGE_SIZE
            ) and cached.dtype == np.uint8
        except (OSError, ValueError, json.JSONDecodeError):
            valid = False
    if not valid:
        # partial 파일에 먼저 완성한 뒤 이름을 바꿔 중단된 cache의 재사용을 막습니다.
        array_temp = array_path.with_name(array_path.name + f".partial.{os.getpid()}")
        metadata_temp = metadata_path.with_name(
            metadata_path.name + f".partial.{os.getpid()}"
        )
        # uint8 memory-map은 전체 이미지를 RAM에 올리지 않고 필요한 부분만 읽습니다.
        resized_images = np.lib.format.open_memmap(
            array_temp, mode="w+", dtype=np.uint8,
            shape=(len(filenames), len(CHANNELS), IMAGE_SIZE, IMAGE_SIZE),
        )
        resampling = Image.Resampling.BILINEAR
        for index, filename in enumerate(filenames):
            for channel_index, channel in enumerate(CHANNELS):
                with Image.open(image_root / channel / filename) as image:
                    resized_images[index, channel_index] = np.asarray(
                        image.convert("L").resize(
                            (IMAGE_SIZE, IMAGE_SIZE), resampling
                        ), dtype=np.uint8,
                    )
            if (index + 1) % 2000 == 0 or index + 1 == len(filenames):
                print(f"{split} resize: {index + 1}/{len(filenames)}", flush=True)
        resized_images.flush()
        del resized_images
        metadata_temp.write_text(
            json.dumps(expected, ensure_ascii=False) + "\n", encoding="utf-8"
        )
        array_temp.replace(array_path)
        metadata_temp.replace(metadata_path)
        print(f"created resized cache: {array_path.resolve()}")
    else:
        print(f"reusing resized cache: {array_path.resolve()}")
    image_array = np.load(array_path, mmap_mode="r")
    image_index = {filename: index for index, filename in enumerate(filenames)}
    return image_array, image_index


train_image_array = train_image_index = None
val_image_array = val_image_index = None
if TRAIN_MODEL:
    train_image_array, train_image_index = prepare_image_memmap("train", train_inputs)
    val_image_array, val_image_index = prepare_image_memmap("validation", val_inputs)
test_image_array, test_image_index = prepare_image_memmap("test", test_inputs)


def estimate_native_disk_geometry(image_root, filenames, sample_limit=200):
    # 원본 해상도에서 태양 원반의 중심/반지름을 추정합니다. 여러 장을 평균 내면
    # 원반이 밝은 원으로 남아 bounding box로 기하를 잡을 수 있습니다.
    # 학습이 아니라 결정론적 기하 추정이며, 규정 준수를 위해 train 영상만 사용해
    # 계산한 뒤 동일한 기하를 validation/test에 그대로 적용합니다.
    accumulator = None
    used = 0
    for filename in filenames[:sample_limit]:
        with Image.open(image_root / CHANNELS[0] / filename) as image:
            array = np.asarray(image.convert("L"), dtype=np.float32)
        accumulator = array if accumulator is None else accumulator + array
        used += 1
    mean_image = accumulator / max(used, 1)
    threshold = 0.5 * (float(mean_image.max()) + float(mean_image.min()))
    bright = mean_image > threshold
    rows = np.where(bright.any(axis=1))[0]
    columns = np.where(bright.any(axis=0))[0]
    height, width = mean_image.shape
    if len(rows) == 0 or len(columns) == 0:
        return 0.5 * (height - 1), 0.5 * (width - 1), 0.5 * min(height, width), (height, width)
    center_y = 0.5 * float(rows[0] + rows[-1])
    center_x = 0.5 * float(columns[0] + columns[-1])
    radius = 0.5 * float(max(rows[-1] - rows[0], columns[-1] - columns[0]))
    return center_y, center_x, radius, (height, width)


def build_coronal_geometry(shape, center_y, center_x, radius):
    grid_y, grid_x = np.ogrid[: shape[0], : shape[1]]
    distance_squared = (grid_y - center_y) ** 2 + (grid_x - center_x) ** 2
    effective_radius = radius * DISK_SHRINK
    full_disk = distance_squared <= effective_radius ** 2
    # 지구로 오는 태양풍에 주로 기여하는 저위도 벨트만 따로 봅니다.
    equator_belt = full_disk & (
        np.abs(grid_y - center_y) <= CH_EQUATOR_FRACTION * effective_radius
    )
    edges = np.linspace(
        center_x - effective_radius, center_x + effective_radius, CH_BANDS + 1
    )
    band_slices = []
    for band_index in range(CH_BANDS):
        start = int(max(0, math.floor(edges[band_index])))
        stop = int(min(shape[1], math.ceil(edges[band_index + 1])))
        band_slices.append(slice(start, max(stop, start + 1)))
    zones = (("full", full_disk), ("equator", equator_belt))
    band_pixel_counts = {
        name: np.array(
            [max(int(mask[:, band].sum()), 1) for band in band_slices], dtype=np.float64
        )
        for name, mask in zones
    }
    return zones, band_slices, band_pixel_counts


def coronal_features_for_array(array, zones, band_slices, band_pixel_counts, full_disk):
    # 영상마다 밝기 스케일이 달라, 절대값이 아니라 그 영상 원반의 중앙값 대비
    # 상대적으로 어두운 픽셀을 coronal hole 후보로 봅니다.
    median = float(np.median(array[full_disk]))
    values = []
    for threshold_ratio in CH_THRESHOLDS:
        dark = array <= median * threshold_ratio
        for zone_name, zone_mask in zones:
            counts = band_pixel_counts[zone_name]
            for band_index, band in enumerate(band_slices):
                covered = np.count_nonzero(dark[:, band] & zone_mask[:, band])
                values.append(covered / counts[band_index])
    return values


def prepare_coronal_features(split, inputs, zones, band_slices, band_pixel_counts, full_disk):
    # 원본 PNG를 한 번 훑어 시점별 coronal hole 특징만 뽑아 작은 배열로 저장합니다.
    # 원본 이미지를 복사/압축하지 않으며, 결과 배열은 수 MB 수준입니다.
    image_root = DATA_ROOT / split
    cache_root = CACHE_ROOT / "coronal"
    cache_root.mkdir(parents=True, exist_ok=True)
    array_path = cache_root / f"{split}_coronal.npy"
    metadata_path = cache_root / f"{split}_coronal_meta.json"
    filenames = sorted(pd.unique(inputs[IMAGE_COLUMNS].to_numpy().ravel()).tolist())
    per_channel = len(CH_THRESHOLDS) * CH_ZONES * CH_BANDS
    expected = {
        "dataset_root": str(DATA_ROOT.resolve()),
        "split": split,
        "channels": list(CHANNELS),
        "bands": CH_BANDS,
        "thresholds": list(CH_THRESHOLDS),
        "equator_fraction": CH_EQUATOR_FRACTION,
        "disk_shrink": DISK_SHRINK,
        "geometry": {
            "center_y": float(DISK_CENTER_Y),
            "center_x": float(DISK_CENTER_X),
            "radius": float(DISK_RADIUS),
            "native_shape": list(NATIVE_SHAPE),
        },
        "filenames": filenames,
    }
    valid = False
    if array_path.exists() and metadata_path.exists():
        try:
            metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
            cached = np.load(array_path, mmap_mode="r")
            valid = metadata == expected and cached.shape == (
                len(filenames), len(CHANNELS) * per_channel
            )
        except (OSError, ValueError, json.JSONDecodeError):
            valid = False
    if not valid:
        features = np.zeros((len(filenames), len(CHANNELS) * per_channel), dtype=np.float32)
        for index, filename in enumerate(filenames):
            row = []
            for channel in CHANNELS:
                with Image.open(image_root / channel / filename) as image:
                    array = np.asarray(image.convert("L"), dtype=np.float32)
                row.extend(
                    coronal_features_for_array(
                        array, zones, band_slices, band_pixel_counts, full_disk
                    )
                )
            features[index] = row
            if (index + 1) % 2000 == 0 or index + 1 == len(filenames):
                print(f"{split} coronal: {index + 1}/{len(filenames)}", flush=True)
        array_temp = array_path.with_name(array_path.name + f".partial.{os.getpid()}")
        # np.save는 파일명이 .npy로 끝나지 않으면 확장자를 덧붙이므로,
        # 임시 파일에 정확히 쓰기 위해 파일 핸들을 직접 넘깁니다.
        with open(array_temp, "wb") as handle:
            np.save(handle, features)
        metadata_temp = metadata_path.with_name(
            metadata_path.name + f".partial.{os.getpid()}"
        )
        metadata_temp.write_text(
            json.dumps(expected, ensure_ascii=False) + "\n", encoding="utf-8"
        )
        array_temp.replace(array_path)
        metadata_temp.replace(metadata_path)
        print(f"created coronal cache: {array_path.resolve()}")
    else:
        print(f"reusing coronal cache: {array_path.resolve()}")
    coronal_array = np.load(array_path)
    coronal_index = {filename: index for index, filename in enumerate(filenames)}
    return coronal_array, coronal_index


# 원반 기하는 Train 영상에서만 추정합니다. Inference-only에서는 최종 checkpoint의
# Train-derived geometry를 복원하여 Test에 동일하게 적용합니다.
if TRAIN_MODEL:
    _train_filenames = sorted(
        pd.unique(train_inputs[IMAGE_COLUMNS].to_numpy().ravel()).tolist()
    )
    DISK_CENTER_Y, DISK_CENTER_X, DISK_RADIUS, NATIVE_SHAPE = (
        estimate_native_disk_geometry(DATA_ROOT / "train", _train_filenames)
    )
else:
    _geometry = loaded_checkpoint["coronal_geometry"]
    DISK_CENTER_Y = float(_geometry["center_y"])
    DISK_CENTER_X = float(_geometry["center_x"])
    DISK_RADIUS = float(_geometry["radius"])
    NATIVE_SHAPE = tuple(int(value) for value in _geometry["native_shape"])

CORONAL_ZONES, CORONAL_BANDS, CORONAL_BAND_COUNTS = build_coronal_geometry(
    NATIVE_SHAPE, DISK_CENTER_Y, DISK_CENTER_X, DISK_RADIUS
)
FULL_DISK_MASK = CORONAL_ZONES[0][1]
print(
    f"native disk geometry (Train-derived): shape={NATIVE_SHAPE} "
    f"center=({DISK_CENTER_Y:.1f}, {DISK_CENTER_X:.1f}) radius={DISK_RADIUS:.1f} "
    f"on-disk pixels={int(FULL_DISK_MASK.sum())}"
)

train_coronal_array = train_coronal_index = None
val_coronal_array = val_coronal_index = None
if TRAIN_MODEL:
    train_coronal_array, train_coronal_index = prepare_coronal_features(
        "train", train_inputs, CORONAL_ZONES, CORONAL_BANDS,
        CORONAL_BAND_COUNTS, FULL_DISK_MASK,
    )
    val_coronal_array, val_coronal_index = prepare_coronal_features(
        "validation", val_inputs, CORONAL_ZONES, CORONAL_BANDS,
        CORONAL_BAND_COUNTS, FULL_DISK_MASK,
    )
test_coronal_array, test_coronal_index = prepare_coronal_features(
    "test", test_inputs, CORONAL_ZONES, CORONAL_BANDS,
    CORONAL_BAND_COUNTS, FULL_DISK_MASK,
)
if TRAIN_MODEL:
    assert train_coronal_array.shape[1] == CH_FEATURE_PER_STEP
    _band_means = train_coronal_array.mean(axis=0)
    print(
        "coronal feature 차원:", CH_FEATURE_PER_STEP, "| 평균 범위:",
        f"{_band_means.min():.4f} ~ {_band_means.max():.4f}",
    )
    assert _band_means.max() > 0.001, (
        "coronal 특징이 전부 0에 가깝습니다. 임계값/원반 기하를 확인하세요."
    )


class SolarWindDataset(Dataset):
    def __init__(
        self, image_array, image_index, inputs, indexes, targets=None,
        coronal_array=None, coronal_index=None,
    ):
        self.image_array = image_array
        self.coronal_array = coronal_array
        self.coronal_indexes = np.asarray([
            [coronal_index[filename] for filename in row]
            for row in inputs[IMAGE_COLUMNS].itertuples(index=False, name=None)
        ], dtype=np.int32)
        # CSV의 파일명을 cache row 번호로 한 번만 변환해 epoch별 문자열 탐색을 없앱니다.
        self.image_indexes = np.asarray([
            [image_index[filename] for filename in row]
            for row in inputs[IMAGE_COLUMNS].itertuples(index=False, name=None)
        ], dtype=np.int32)
        # wind와 target은 약 0.x 규모로 맞춰 학습을 안정화하고, 평가 때 km/s로 복원합니다.
        self.wind = inputs[WIND_COLUMNS].to_numpy(np.float32) / 1000.0
        self.sample_ids = inputs.sample_id.to_numpy()
        self.indexes = np.asarray(indexes, dtype=np.int64)
        self.targets = targets

    def __len__(self):
        return len(self.indexes)

    def __getitem__(self, item):
        row_index = int(self.indexes[item])
        # 반환 shape: images=(20,2,IMAGE_SIZE,IMAGE_SIZE), wind=(WIND_FEATURE_DIM,), target=(12,)
        images = np.asarray(
            self.image_array[self.image_indexes[row_index]], dtype=np.float32
        ) / 255.0
        wind_raw = self.wind[row_index]
        # 최근 태양풍 속도의 가속/감속 추세를 명시적 feature로 추가해
        # persistence(마지막 값 유지) 대비 변화량을 모델이 더 쉽게 학습하도록 합니다.
        wind_trend = np.diff(wind_raw)
        wind_features = np.concatenate([wind_raw, wind_trend], axis=0).astype(np.float32)
        # 시점별 coronal hole 특징: (20, CH_FEATURE_PER_STEP)
        coronal = np.asarray(
            self.coronal_array[self.coronal_indexes[row_index]], dtype=np.float32
        )
        # 모델이 절대값 대신 마지막 관측값(persistence) 대비 delta를 예측하도록
        # head의 출력에 이 값을 더해줍니다 (단기 horizon에서 persistence를 쉽게 못 이기는 문제 완화).
        persistence = np.asarray([wind_raw[-1]], dtype=np.float32)
        result = {
            "images": torch.from_numpy(images),
            "wind": torch.from_numpy(wind_features),
            "coronal": torch.from_numpy(coronal),
            "persistence": torch.from_numpy(persistence),
            "sample_id": self.sample_ids[row_index],
            "row_index": row_index,
        }
        if self.targets is not None:
            result["target"] = torch.from_numpy(
                np.asarray(self.targets[row_index], dtype=np.float32) / 1000.0
            )
        return result


def seed_worker(worker_id):
    worker_seed = (SEED + worker_id) % (2 ** 32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)


TRAIN_LOADER_GENERATOR = torch.Generator().manual_seed(SEED)
VAL_LOADER_GENERATOR = torch.Generator().manual_seed(SEED)
TEST_LOADER_GENERATOR = torch.Generator().manual_seed(SEED)


def make_loader(dataset, shuffle, generator=None):
    if generator is None:
        generator = torch.Generator().manual_seed(SEED)
    options = dict(
        dataset=dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
        worker_init_fn=seed_worker,
        generator=generator,
    )
    if NUM_WORKERS > 0:
        options.update(persistent_workers=True, prefetch_factor=2)
    return DataLoader(**options)


train_dataset = val_dataset = train_loader = val_loader = None
if TRAIN_MODEL:
    train_dataset = SolarWindDataset(
        train_image_array, train_image_index, train_inputs, train_index, train_targets,
        coronal_array=train_coronal_array, coronal_index=train_coronal_index,
    )
    val_dataset = SolarWindDataset(
        val_image_array, val_image_index, val_inputs, val_index, val_targets,
        coronal_array=val_coronal_array, coronal_index=val_coronal_index,
    )
    train_loader = make_loader(
        train_dataset, shuffle=True, generator=TRAIN_LOADER_GENERATOR
    )
    val_loader = make_loader(
        val_dataset, shuffle=False, generator=VAL_LOADER_GENERATOR
    )

    first_sample = train_dataset[0]
    assert first_sample["images"].shape == (20, 2, IMAGE_SIZE, IMAGE_SIZE)
    assert first_sample["wind"].shape == (WIND_FEATURE_DIM,)
    assert first_sample["coronal"].shape == (20, CH_FEATURE_PER_STEP)
    assert first_sample["target"].shape == (12,)
    print(
        first_sample["images"].shape, first_sample["wind"].shape,
        first_sample["coronal"].shape, first_sample["target"].shape,
    )


## 3. V3.5 공통 backbone + 20개 domain specialist


In [ ]:
class Inception3D(nn.Module):
    def __init__(self, in_channels, branch_channels=32):
        super().__init__()
        self.branch_1 = nn.Sequential(
            nn.Conv3d(in_channels, branch_channels, 1),
            nn.BatchNorm3d(branch_channels), nn.ReLU(inplace=True),
        )
        self.branch_3 = nn.Sequential(
            nn.Conv3d(in_channels, branch_channels, 1),
            nn.BatchNorm3d(branch_channels), nn.ReLU(inplace=True),
            nn.Conv3d(
                branch_channels, branch_channels, (1, 3, 3),
                padding=(0, 1, 1),
            ),
            nn.BatchNorm3d(branch_channels), nn.ReLU(inplace=True),
        )
        self.branch_5 = nn.Sequential(
            nn.Conv3d(in_channels, branch_channels, 1),
            nn.BatchNorm3d(branch_channels), nn.ReLU(inplace=True),
            nn.Conv3d(
                branch_channels, branch_channels, (1, 5, 5),
                padding=(0, 2, 2),
            ),
            nn.BatchNorm3d(branch_channels), nn.ReLU(inplace=True),
        )
        self.branch_pool = nn.Sequential(
            nn.MaxPool3d((1, 3, 3), stride=1, padding=(0, 1, 1)),
            nn.Conv3d(in_channels, branch_channels, 1),
            nn.BatchNorm3d(branch_channels), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return torch.cat(
            [
                self.branch_1(x), self.branch_3(x),
                self.branch_5(x), self.branch_pool(x),
            ],
            dim=1,
        )


class HorizonQueryAttention(nn.Module):
    """Static horizon query attention: sequence=(B,T,D), horizon=(H,E)."""
    def __init__(self, sequence_dim, horizon_dim, attention_dim):
        super().__init__()
        self.attention_dim = attention_dim
        self.key_projection = nn.Linear(sequence_dim, attention_dim, bias=False)
        self.query_projection = nn.Linear(horizon_dim, attention_dim, bias=False)

    def forward(self, sequence, horizon_embeddings):
        keys = self.key_projection(sequence)
        queries = self.query_projection(horizon_embeddings)
        scores = torch.einsum("bta,ha->bht", keys, queries)
        scores = scores / math.sqrt(self.attention_dim)
        weights = torch.softmax(scores, dim=-1)
        context = torch.einsum("bht,btd->bhd", weights, sequence)
        return context, weights


class BatchHorizonQueryAttention(nn.Module):
    """Sample-conditioned query: sequence=(B,T,D), query=(B,H,E)."""
    def __init__(self, sequence_dim, query_dim, attention_dim):
        super().__init__()
        self.attention_dim = attention_dim
        self.key_projection = nn.Linear(sequence_dim, attention_dim, bias=False)
        self.query_projection = nn.Linear(query_dim, attention_dim, bias=False)

    def forward(self, sequence, queries):
        keys = self.key_projection(sequence)
        projected_queries = self.query_projection(queries)
        scores = torch.einsum("bta,bha->bht", keys, projected_queries)
        scores = scores / math.sqrt(self.attention_dim)
        weights = torch.softmax(scores, dim=-1)
        context = torch.einsum("bht,btd->bhd", weights, sequence)
        return context, weights


def coronal_to_band_features(coronal):
    # cache order: channel -> threshold -> zone -> longitude band
    batch_size = coronal.shape[0]
    assert coronal.shape[1:] == (20, CH_FEATURE_PER_STEP)
    return (
        coronal.reshape(
            batch_size, 20,
            len(CHANNELS), len(CH_THRESHOLDS), CH_ZONES, CH_BANDS,
        )
        .permute(0, 1, 5, 2, 3, 4)
        .contiguous()
        .reshape(batch_size, 20, CH_BANDS, CORONAL_FEATURES_PER_BAND)
    )


def equator_only_coronal(coronal):
    view = coronal.reshape(
        coronal.shape[0], 20,
        len(CHANNELS), len(CH_THRESHOLDS), CH_ZONES, CH_BANDS,
    )
    zone_mask = torch.tensor(
        [0.0, 1.0], device=coronal.device, dtype=coronal.dtype
    ).view(1, 1, 1, 1, CH_ZONES, 1)
    return (view * zone_mask).reshape(coronal.shape[0], 20, CH_FEATURE_PER_STEP)



class SolarWindV35(nn.Module):
    VALID_VARIANTS = set(ENSEMBLE_VARIANTS)

    def __init__(
        self,
        wind_feature_dim=WIND_FEATURE_DIM,
        dropout=DROPOUT,
        conv_dropout=CONV_DROPOUT,
        variant="physics",
        horizon_embed_dim=HORIZON_EMBED_DIM,
        image_attention_dim=IMAGE_ATTENTION_DIM,
        wind_gru_hidden=WIND_GRU_HIDDEN,
        wind_attention_dim=WIND_ATTENTION_DIM,
        horizon_decoder_hidden=HORIZON_DECODER_HIDDEN,
    ):
        super().__init__()
        assert variant in self.VALID_VARIANTS, variant
        assert wind_feature_dim == 39

        self.variant = variant
        self.wind_feature_dim = wind_feature_dim
        self.uses_image = variant != "wind_short"

        self.physics_like = variant in {
            "physics",
            "rotation_track",
            "active_region_edge",
            "propagation",
            "bright_dark_boundary",
            "east_west",
            "physics_lt",
            "near_earth_continuity",
            "stream_onset",
            "hss_peak",
            "hss_duration",
            "hss_recovery",
            "slow_wind",
            "regime_change",
            "solar_long_range",
            "trajectory",
        }
        self.wide_like = variant in {
            "wide",
            "ratio_temporal",
            "ch_topology",
        }
        assert self.uses_image is False or self.physics_like or self.wide_like

        # ------------------------------------------------------------
        # Image branch
        # ------------------------------------------------------------
        if self.uses_image:
            stem_channels = 3 if variant == "ratio_temporal" else 2
            self.stem = nn.Sequential(
                nn.Conv3d(stem_channels, 32, (1, 5, 5), padding=(0, 2, 2)),
                nn.BatchNorm3d(32),
                nn.ReLU(inplace=True),
                nn.MaxPool3d((1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1)),
            )
            blocks = []
            in_channels = 32
            for _ in range(3):
                blocks.extend(
                    [
                        Inception3D(in_channels, 32),
                        nn.MaxPool3d(
                            (1, 3, 3),
                            stride=(1, 2, 2),
                            padding=(0, 1, 1),
                        ),
                        nn.Dropout3d(conv_dropout),
                    ]
                )
                in_channels = 128
            self.image_encoder = nn.Sequential(*blocks)

            if self.physics_like:
                self.spatial_pool = nn.AdaptiveAvgPool3d(
                    (None, POOL_HEIGHT, POOL_WIDTH)
                )
                spatial_units = 128 * POOL_HEIGHT * POOL_WIDTH
                self.coronal_encoder = nn.Sequential(
                    nn.Linear(CH_FEATURE_PER_STEP, 64),
                    nn.ReLU(inplace=True),
                    nn.Linear(64, 64),
                    nn.ReLU(inplace=True),
                )
                lstm_input_size = spatial_units + 64
            else:
                self.spatial_pool = nn.Identity()
                spatial_after_downsampling = IMAGE_SIZE // 16
                spatial_units = (
                    128 * spatial_after_downsampling * spatial_after_downsampling
                )
                self.coronal_encoder = None
                lstm_input_size = spatial_units

            self.image_lstm = nn.LSTM(
                input_size=lstm_input_size,
                hidden_size=128,
                batch_first=True,
                bidirectional=True,
            )
        else:
            self.stem = None
            self.image_encoder = None
            self.spatial_pool = None
            self.coronal_encoder = None
            self.image_lstm = None

        self.horizon_embedding = nn.Embedding(TARGET_HORIZONS, horizon_embed_dim)

        self.image_horizon_attention = (
            HorizonQueryAttention(256, horizon_embed_dim, image_attention_dim)
            if self.uses_image
            else None
        )

        # ------------------------------------------------------------
        # Wind branches
        # ------------------------------------------------------------
        self.wind_encoder = nn.Sequential(
            nn.Linear(wind_feature_dim, 128),
            nn.SELU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.SELU(inplace=True),
        )
        self.wind_gru = nn.GRU(
            input_size=2,
            hidden_size=wind_gru_hidden,
            batch_first=True,
            bidirectional=True,
        )
        wind_sequence_dim = wind_gru_hidden * 2
        self.wind_horizon_attention = HorizonQueryAttention(
            wind_sequence_dim, horizon_embed_dim, wind_attention_dim
        )

        # ------------------------------------------------------------
        # Coronal longitude-time token branch
        # ------------------------------------------------------------
        if variant in {"physics_lt", "propagation", "solar_long_range"}:
            self.coronal_band_encoder = nn.Sequential(
                nn.Linear(CORONAL_FEATURES_PER_BAND, CORONAL_BAND_TOKEN_DIM),
                nn.ReLU(inplace=True),
                nn.Linear(CORONAL_BAND_TOKEN_DIM, CORONAL_BAND_TOKEN_DIM),
                nn.ReLU(inplace=True),
            )
            self.coronal_time_pos = nn.Parameter(
                torch.randn(1, 20, 1, CORONAL_BAND_TOKEN_DIM) * 0.02
            )
            self.coronal_band_pos = nn.Parameter(
                torch.randn(1, 1, CH_BANDS, CORONAL_BAND_TOKEN_DIM) * 0.02
            )
        else:
            self.coronal_band_encoder = None
            self.coronal_time_pos = None
            self.coronal_band_pos = None

        if variant in {"physics_lt", "solar_long_range"}:
            self.coronal_horizon_attention = HorizonQueryAttention(
                CORONAL_BAND_TOKEN_DIM,
                horizon_embed_dim,
                CORONAL_BAND_ATTENTION_DIM,
            )
        else:
            self.coronal_horizon_attention = None

        if variant == "propagation":
            self.propagation_query = nn.Sequential(
                nn.Linear(horizon_embed_dim + 64, 64),
                nn.ReLU(inplace=True),
                nn.Linear(64, horizon_embed_dim),
            )
            self.propagation_attention = BatchHorizonQueryAttention(
                CORONAL_BAND_TOKEN_DIM,
                horizon_embed_dim,
                CORONAL_BAND_ATTENTION_DIM,
            )
        else:
            self.propagation_query = None
            self.propagation_attention = None

        # ------------------------------------------------------------
        # Rotation tracking branch
        # ------------------------------------------------------------
        if variant in {"rotation_track", "solar_long_range"}:
            self.rotation_conv = nn.Sequential(
                nn.Conv2d(
                    CORONAL_FEATURES_PER_BAND,
                    24,
                    kernel_size=(3, 3),
                    padding=(1, 1),
                ),
                nn.ReLU(inplace=True),
                nn.Conv2d(24, 24, kernel_size=(3, 3), padding=(1, 1)),
                nn.ReLU(inplace=True),
            )
            self.rotation_step_encoder = nn.Sequential(
                nn.Linear(24 * CH_BANDS, ROTATION_TRACK_DIM),
                nn.ReLU(inplace=True),
            )
            self.rotation_horizon_attention = HorizonQueryAttention(
                ROTATION_TRACK_DIM, horizon_embed_dim, 32
            )
        else:
            self.rotation_conv = None
            self.rotation_step_encoder = None
            self.rotation_horizon_attention = None

        # ------------------------------------------------------------
        # Image-derived specialist branches
        # ------------------------------------------------------------
        if variant in {
            "active_region_edge",
            "bright_dark_boundary",
            "ch_topology",
            "slow_wind",
        }:
            axis = torch.linspace(-1.0, 1.0, IMAGE_SIZE)
            yy, xx = torch.meshgrid(axis, axis, indexing="ij")
            radius2 = xx.square() + yy.square()
            disk = (radius2 <= DISK_SHRINK ** 2).float()
            equator = (
                (radius2 <= DISK_SHRINK ** 2)
                & (yy.abs() <= CH_EQUATOR_FRACTION * DISK_SHRINK)
            ).float()
            self.register_buffer(
                "_grid_x",
                xx.view(1, 1, 1, IMAGE_SIZE, IMAGE_SIZE),
                persistent=False,
            )
            self.register_buffer(
                "_grid_y",
                yy.view(1, 1, 1, IMAGE_SIZE, IMAGE_SIZE),
                persistent=False,
            )
            self.register_buffer(
                "_disk_mask",
                disk.view(1, 1, 1, IMAGE_SIZE, IMAGE_SIZE),
                persistent=False,
            )
            self.register_buffer(
                "_equator_mask",
                equator.view(1, 1, 1, IMAGE_SIZE, IMAGE_SIZE),
                persistent=False,
            )
        else:
            self._grid_x = None
            self._grid_y = None
            self._disk_mask = None
            self._equator_mask = None

        if variant in {"active_region_edge", "slow_wind"}:
            self.active_step_encoder = nn.Sequential(
                nn.Linear(12, 32),
                nn.ReLU(inplace=True),
            )
            self.active_gru = nn.GRU(
                input_size=32,
                hidden_size=32,
                batch_first=True,
                bidirectional=True,
            )
            self.active_horizon_attention = HorizonQueryAttention(
                64, horizon_embed_dim, 32
            )
        else:
            self.active_step_encoder = None
            self.active_gru = None
            self.active_horizon_attention = None

        if variant == "bright_dark_boundary":
            self.boundary_step_encoder = nn.Sequential(
                nn.Linear(12, 32),
                nn.ReLU(inplace=True),
            )
            self.boundary_gru = nn.GRU(
                input_size=32,
                hidden_size=32,
                batch_first=True,
                bidirectional=True,
            )
            self.boundary_horizon_attention = HorizonQueryAttention(
                64, horizon_embed_dim, 32
            )
        else:
            self.boundary_step_encoder = None
            self.boundary_gru = None
            self.boundary_horizon_attention = None

        if variant == "east_west":
            self.east_west_step_encoder = nn.Sequential(
                nn.Linear(8, 32),
                nn.ReLU(inplace=True),
            )
            self.east_west_gru = nn.GRU(
                input_size=32,
                hidden_size=32,
                batch_first=True,
                bidirectional=True,
            )
            self.east_west_horizon_attention = HorizonQueryAttention(
                64, horizon_embed_dim, 32
            )
        else:
            self.east_west_step_encoder = None
            self.east_west_gru = None
            self.east_west_horizon_attention = None

        if variant == "ch_topology":
            self.topology_step_encoder = nn.Sequential(
                nn.Linear(16, 32),
                nn.ReLU(inplace=True),
            )
            self.topology_gru = nn.GRU(
                input_size=32,
                hidden_size=32,
                batch_first=True,
                bidirectional=True,
            )
            self.topology_horizon_attention = HorizonQueryAttention(
                64, horizon_embed_dim, 32
            )
        else:
            self.topology_step_encoder = None
            self.topology_gru = None
            self.topology_horizon_attention = None

        # ------------------------------------------------------------
        # Regime-change wind dynamics branch
        # ------------------------------------------------------------
        if variant == "regime_change":
            self.regime_step_encoder = nn.Sequential(
                nn.Linear(4, 24),
                nn.ReLU(inplace=True),
            )
            self.regime_gru = nn.GRU(
                input_size=24,
                hidden_size=24,
                batch_first=True,
                bidirectional=True,
            )
            self.regime_horizon_attention = HorizonQueryAttention(
                REGIME_CONTEXT_DIM, horizon_embed_dim, 24
            )
        else:
            self.regime_step_encoder = None
            self.regime_gru = None
            self.regime_horizon_attention = None

        # ------------------------------------------------------------
        # Decoder dimensions
        # ------------------------------------------------------------
        fusion_dim = wind_sequence_dim + 64 + horizon_embed_dim
        if self.uses_image:
            fusion_dim += 256
        if variant in {"physics_lt", "propagation", "solar_long_range"}:
            fusion_dim += CORONAL_BAND_TOKEN_DIM
        if variant in {"rotation_track", "solar_long_range"}:
            fusion_dim += ROTATION_TRACK_DIM
        if variant in {"active_region_edge", "slow_wind"}:
            fusion_dim += AUX_CONTEXT_DIM
        if variant == "bright_dark_boundary":
            fusion_dim += AUX_CONTEXT_DIM
        if variant == "east_west":
            fusion_dim += AUX_CONTEXT_DIM
        if variant == "ch_topology":
            fusion_dim += AUX_CONTEXT_DIM
        if variant == "regime_change":
            fusion_dim += REGIME_CONTEXT_DIM

        self.horizon_decoder = nn.Sequential(
            nn.Linear(fusion_dim, horizon_decoder_hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(horizon_decoder_hidden, 1),
        )
        self.linear_ar = nn.Linear(wind_feature_dim, TARGET_HORIZONS)

        # exact persistence at initialization for every member
        nn.init.zeros_(self.linear_ar.weight)
        nn.init.zeros_(self.linear_ar.bias)
        nn.init.zeros_(self.horizon_decoder[-1].weight)
        nn.init.zeros_(self.horizon_decoder[-1].bias)

    # ------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------
    def _make_band_tokens(self, coronal):
        band_features = coronal_to_band_features(coronal)
        tokens = self.coronal_band_encoder(band_features)
        return tokens + self.coronal_time_pos + self.coronal_band_pos

    def _recent_wind_vector(self, wind):
        masked = wind.clone()
        masked[:, :20 - RECENT_STEPS] = 0.0
        # diff slots are wind[:,20:] and correspond to raw t1-t0 ... t19-t18.
        # Keep only the final RECENT_STEPS-1 differences.
        first_kept_diff = 19 - (RECENT_STEPS - 1)
        masked[:, 20:20 + first_kept_diff] = 0.0
        return masked

    def _active_region_sequence(self, images):
        # Bright-region boundary proxy, entirely from provided 193/211 frames.
        disk = self._disk_mask.to(dtype=images.dtype)
        equator = self._equator_mask.to(dtype=images.dtype)
        grid_x = self._grid_x.to(dtype=images.dtype)
        grid_y = self._grid_y.to(dtype=images.dtype)
        disk_sum = disk.sum().clamp_min(1.0)
        equator_sum = equator.sum().clamp_min(1.0)

        mean = (images * disk).sum(dim=(-1, -2), keepdim=True) / disk_sum
        var = (
            ((images - mean).square() * disk).sum(dim=(-1, -2), keepdim=True)
            / disk_sum
        )
        std = torch.sqrt(var + 1e-6)
        threshold = mean + 0.70 * std
        bright = torch.sigmoid((images - threshold) / 0.045) * disk

        dx = F.pad(
            (bright[..., :, 1:] - bright[..., :, :-1]).abs(),
            (0, 1, 0, 0),
        )
        dy = F.pad(
            (bright[..., 1:, :] - bright[..., :-1, :]).abs(),
            (0, 0, 0, 1),
        )
        edge = torch.sqrt(dx.square() + dy.square() + 1e-6) * disk

        mass = bright.sum(dim=(-1, -2), keepdim=True) + 1e-6
        bright_fraction = bright.sum(dim=(-1, -2)) / disk_sum
        edge_density = edge.sum(dim=(-1, -2)) / disk_sum
        centroid_x = (bright * grid_x).sum(dim=(-1, -2), keepdim=True) / mass
        centroid_y = (bright * grid_y).sum(dim=(-1, -2), keepdim=True) / mass
        centroid_x = centroid_x.squeeze(-1).squeeze(-1)
        centroid_y = centroid_y.squeeze(-1).squeeze(-1)
        equator_fraction = (bright * equator).sum(dim=(-1, -2)) / equator_sum

        growth = F.pad(
            bright_fraction[:, 1:] - bright_fraction[:, :-1],
            (0, 0, 1, 0),
            value=0.0,
        )
        # 6 features per channel -> 12 total
        features = torch.stack(
            [
                bright_fraction,
                edge_density,
                centroid_x,
                centroid_y,
                equator_fraction,
                growth,
            ],
            dim=-1,
        )
        return features.flatten(-2)

    def _boundary_sequence(self, images):
        # Soft contact between bright active-region proxy and dark coronal-hole proxy.
        disk = self._disk_mask.to(dtype=images.dtype)
        equator = self._equator_mask.to(dtype=images.dtype)
        disk_sum = disk.sum().clamp_min(1.0)
        equator_sum = equator.sum().clamp_min(1.0)

        mean = (images * disk).sum(dim=(-1, -2), keepdim=True) / disk_sum
        var = (
            ((images - mean).square() * disk).sum(dim=(-1, -2), keepdim=True)
            / disk_sum
        )
        std = torch.sqrt(var + 1e-6)
        bright = torch.sigmoid((images - (mean + 0.65 * std)) / 0.05) * disk
        dark = torch.sigmoid(((mean - 0.60 * std) - images) / 0.05) * disk

        b, t, c, h, w = images.shape
        bright_flat = bright.reshape(b * t * c, 1, h, w)
        dark_flat = dark.reshape(b * t * c, 1, h, w)
        local_bright = F.avg_pool2d(bright_flat, 5, stride=1, padding=2)
        local_dark = F.avg_pool2d(dark_flat, 5, stride=1, padding=2)
        contact = (
            bright_flat * local_dark + dark_flat * local_bright
        ).reshape(b, t, c, h, w) * disk

        bright_fraction = bright.sum(dim=(-1, -2)) / disk_sum
        dark_fraction = dark.sum(dim=(-1, -2)) / disk_sum
        contact_fraction = contact.sum(dim=(-1, -2)) / disk_sum
        equator_contact = (contact * equator).sum(dim=(-1, -2)) / equator_sum

        grid_x = self._grid_x.to(dtype=images.dtype)
        bright_mass = bright.sum(dim=(-1, -2), keepdim=True) + 1e-6
        dark_mass = dark.sum(dim=(-1, -2), keepdim=True) + 1e-6
        bright_x = (bright * grid_x).sum(dim=(-1, -2), keepdim=True) / bright_mass
        dark_x = (dark * grid_x).sum(dim=(-1, -2), keepdim=True) / dark_mass
        separation = (bright_x - dark_x).abs().squeeze(-1).squeeze(-1)

        contact_change = F.pad(
            contact_fraction[:, 1:] - contact_fraction[:, :-1],
            (0, 0, 1, 0),
            value=0.0,
        )
        # 6 per channel -> 12
        features = torch.stack(
            [
                bright_fraction,
                dark_fraction,
                contact_fraction,
                equator_contact,
                separation,
                contact_change,
            ],
            dim=-1,
        )
        return features.flatten(-2)

    def _east_west_sequence(self, coronal):
        band_features = coronal_to_band_features(coronal)
        band_mass = band_features.mean(dim=-1)  # B,T,6
        left = band_mass[:, :, :3].mean(dim=-1)
        right = band_mass[:, :, 3:].mean(dim=-1)
        total = left + right + 1e-6
        asymmetry = (right - left) / total
        abs_asymmetry = asymmetry.abs()
        center = band_mass[:, :, 2:4].mean(dim=-1)
        edge_difference = band_mass[:, :, 0] - band_mass[:, :, -1]
        left_change = F.pad(
            left[:, 1:] - left[:, :-1], (1, 0), value=0.0
        )
        right_change = F.pad(
            right[:, 1:] - right[:, :-1], (1, 0), value=0.0
        )
        return torch.stack(
            [
                left,
                right,
                asymmetry,
                abs_asymmetry,
                center,
                edge_difference,
                left_change,
                right_change,
            ],
            dim=-1,
        )

    def _topology_sequence(self, images):
        # Differentiable dark-region shape/connectedness proxies at 64x64.
        disk = self._disk_mask.to(dtype=images.dtype)
        equator = self._equator_mask.to(dtype=images.dtype)
        grid_x = self._grid_x.to(dtype=images.dtype)
        grid_y = self._grid_y.to(dtype=images.dtype)
        disk_sum = disk.sum().clamp_min(1.0)
        equator_sum = equator.sum().clamp_min(1.0)

        mean = (images * disk).sum(dim=(-1, -2), keepdim=True) / disk_sum
        var = (
            ((images - mean).square() * disk).sum(dim=(-1, -2), keepdim=True)
            / disk_sum
        )
        std = torch.sqrt(var + 1e-6)
        dark = torch.sigmoid(((mean - 0.55 * std) - images) / 0.05) * disk

        mass = dark.sum(dim=(-1, -2), keepdim=True) + 1e-6
        dark_fraction = dark.sum(dim=(-1, -2)) / disk_sum
        centroid_x = ((dark * grid_x).sum(dim=(-1, -2), keepdim=True) / mass)
        centroid_y = ((dark * grid_y).sum(dim=(-1, -2), keepdim=True) / mass)
        spread_x = torch.sqrt(
            (
                dark * (grid_x - centroid_x).square()
            ).sum(dim=(-1, -2), keepdim=True)
            / mass
            + 1e-6
        )
        spread_y = torch.sqrt(
            (
                dark * (grid_y - centroid_y).square()
            ).sum(dim=(-1, -2), keepdim=True)
            / mass
            + 1e-6
        )

        b, t, c, h, w = images.shape
        dark_flat = dark.reshape(b * t * c, 1, h, w)
        local_density = F.avg_pool2d(dark_flat, 9, stride=1, padding=4)
        local_density = local_density.reshape(b, t, c, h, w)
        dense_peak = local_density.amax(dim=(-1, -2))
        connectedness_proxy = (
            (dark * local_density).sum(dim=(-1, -2))
            / (dark.sum(dim=(-1, -2)) + 1e-6)
        )
        equator_fraction = (dark * equator).sum(dim=(-1, -2)) / equator_sum

        features = torch.stack(
            [
                dark_fraction,
                centroid_x.squeeze(-1).squeeze(-1),
                centroid_y.squeeze(-1).squeeze(-1),
                spread_x.squeeze(-1).squeeze(-1),
                spread_y.squeeze(-1).squeeze(-1),
                dense_peak,
                connectedness_proxy,
                equator_fraction,
            ],
            dim=-1,
        )
        return features.flatten(-2)  # 16

    def _regime_sequence(self, wind):
        raw = wind[:, :20]
        first = F.pad(raw[:, 1:] - raw[:, :-1], (1, 0), value=0.0)
        second = F.pad(first[:, 1:] - first[:, :-1], (1, 0), value=0.0)
        return torch.stack([raw, first, second, first.abs()], dim=-1)

    # ------------------------------------------------------------
    # Forward
    # ------------------------------------------------------------
    def forward(self, images, wind, persistence, coronal):
        assert images.ndim == 5 and images.shape[1:] == (
            20, 2, IMAGE_SIZE, IMAGE_SIZE
        )
        assert wind.ndim == 2 and wind.shape[1] == self.wind_feature_dim
        assert persistence.ndim == 2 and persistence.shape[1] == 1
        assert coronal.shape[1:] == (20, CH_FEATURE_PER_STEP)

        horizon_indices = torch.arange(TARGET_HORIZONS, device=wind.device)
        horizon_embeddings = self.horizon_embedding(horizon_indices)
        horizon_context = horizon_embeddings.unsqueeze(0).expand(
            wind.shape[0], -1, -1
        )

        fusion_parts = []

        # ------------------------------------------------------------
        # Image context
        # ------------------------------------------------------------
        image_source = images
        coronal_source = coronal
        if self.variant == "near_earth_continuity":
            image_source = images[:, -RECENT_STEPS:]
            coronal_source = coronal[:, -RECENT_STEPS:]

        if self.uses_image:
            if self.variant == "ratio_temporal":
                ratio = torch.log(
                    (image_source[:, :, 0:1] + 0.02)
                    / (image_source[:, :, 1:2] + 0.02)
                ).clamp(-2.5, 2.5) / 2.5
                image_input = torch.cat([image_source, ratio], dim=2)
            else:
                image_input = image_source

            encoded = image_input.permute(0, 2, 1, 3, 4).contiguous()
            encoded = self.stem(encoded)
            encoded = self.image_encoder(encoded)
            encoded = self.spatial_pool(encoded)
            encoded = encoded.permute(0, 2, 1, 3, 4).flatten(2)

            if self.coronal_encoder is not None:
                encoded = torch.cat(
                    [encoded, self.coronal_encoder(coronal_source)],
                    dim=2,
                )

            image_sequence, _ = self.image_lstm(encoded)
            image_context, _ = self.image_horizon_attention(
                image_sequence, horizon_embeddings
            )
            fusion_parts.append(image_context)

        # ------------------------------------------------------------
        # Wind context
        # ------------------------------------------------------------
        wind_for_global = wind
        wind_for_linear = wind
        if self.variant == "near_earth_continuity":
            wind_for_global = self._recent_wind_vector(wind)
            wind_for_linear = wind_for_global
            raw_wind = wind[:, :20][:, -RECENT_STEPS:]
        else:
            raw_wind = wind[:, :20]

        first_difference = F.pad(
            raw_wind[:, 1:] - raw_wind[:, :-1],
            (1, 0),
            value=0.0,
        )
        wind_sequence_input = torch.stack(
            [raw_wind, first_difference],
            dim=-1,
        )
        wind_sequence, _ = self.wind_gru(wind_sequence_input)
        wind_temporal_context, _ = self.wind_horizon_attention(
            wind_sequence, horizon_embeddings
        )
        global_wind = self.wind_encoder(wind_for_global)
        global_wind_h = global_wind.unsqueeze(1).expand(
            -1, TARGET_HORIZONS, -1
        )

        fusion_parts.extend(
            [
                wind_temporal_context,
                global_wind_h,
                horizon_context,
            ]
        )

        # ------------------------------------------------------------
        # Specialist contexts
        # ------------------------------------------------------------
        if self.variant in {"physics_lt", "solar_long_range"}:
            band_tokens = self._make_band_tokens(coronal).reshape(
                coronal.shape[0],
                20 * CH_BANDS,
                CORONAL_BAND_TOKEN_DIM,
            )
            coronal_context, _ = self.coronal_horizon_attention(
                band_tokens, horizon_embeddings
            )
            fusion_parts.append(coronal_context)

        elif self.variant == "propagation":
            band_tokens = self._make_band_tokens(coronal).reshape(
                coronal.shape[0],
                20 * CH_BANDS,
                CORONAL_BAND_TOKEN_DIM,
            )
            conditioned_query = self.propagation_query(
                torch.cat([horizon_context, global_wind_h], dim=-1)
            )
            propagation_context, _ = self.propagation_attention(
                band_tokens, conditioned_query
            )
            fusion_parts.append(propagation_context)

        if self.variant in {"rotation_track", "solar_long_range"}:
            band_features = coronal_to_band_features(coronal)
            track = band_features.permute(0, 3, 1, 2).contiguous()
            track = self.rotation_conv(track)
            track = (
                track.permute(0, 2, 3, 1)
                .contiguous()
                .flatten(2)
            )
            track = self.rotation_step_encoder(track)
            rotation_context, _ = self.rotation_horizon_attention(
                track, horizon_embeddings
            )
            fusion_parts.append(rotation_context)

        if self.variant in {"active_region_edge", "slow_wind"}:
            active = self._active_region_sequence(images)
            active = self.active_step_encoder(active)
            active_sequence, _ = self.active_gru(active)
            active_context, _ = self.active_horizon_attention(
                active_sequence, horizon_embeddings
            )
            fusion_parts.append(active_context)

        if self.variant == "bright_dark_boundary":
            boundary = self._boundary_sequence(images)
            boundary = self.boundary_step_encoder(boundary)
            boundary_sequence, _ = self.boundary_gru(boundary)
            boundary_context, _ = self.boundary_horizon_attention(
                boundary_sequence, horizon_embeddings
            )
            fusion_parts.append(boundary_context)

        if self.variant == "east_west":
            east_west = self._east_west_sequence(coronal)
            east_west = self.east_west_step_encoder(east_west)
            east_west_sequence, _ = self.east_west_gru(east_west)
            east_west_context, _ = self.east_west_horizon_attention(
                east_west_sequence, horizon_embeddings
            )
            fusion_parts.append(east_west_context)

        if self.variant == "ch_topology":
            topology = self._topology_sequence(images)
            topology = self.topology_step_encoder(topology)
            topology_sequence, _ = self.topology_gru(topology)
            topology_context, _ = self.topology_horizon_attention(
                topology_sequence, horizon_embeddings
            )
            fusion_parts.append(topology_context)

        if self.variant == "regime_change":
            regime = self._regime_sequence(wind)
            regime = self.regime_step_encoder(regime)
            regime_sequence, _ = self.regime_gru(regime)
            regime_context, _ = self.regime_horizon_attention(
                regime_sequence, horizon_embeddings
            )
            fusion_parts.append(regime_context)

        fusion = torch.cat(fusion_parts, dim=-1)
        raw_nonlinear = self.horizon_decoder(fusion).squeeze(-1)
        raw_linear = self.linear_ar(wind_for_linear)

        if self.variant == "trajectory":
            nonlinear_delta = torch.cumsum(raw_nonlinear, dim=1)
            linear_delta = torch.cumsum(raw_linear, dim=1)
        else:
            nonlinear_delta = raw_nonlinear
            linear_delta = raw_linear

        prediction = persistence + linear_delta + nonlinear_delta
        assert prediction.shape == (wind.shape[0], TARGET_HORIZONS)
        return prediction


def build_v35_model(variant, device=DEVICE):
    return SolarWindV35(
        wind_feature_dim=WIND_FEATURE_DIM,
        dropout=DROPOUT,
        conv_dropout=CONV_DROPOUT,
        variant=variant,
        horizon_embed_dim=HORIZON_EMBED_DIM,
        image_attention_dim=IMAGE_ATTENTION_DIM,
        wind_gru_hidden=WIND_GRU_HIDDEN,
        wind_attention_dim=WIND_ATTENTION_DIM,
        horizon_decoder_hidden=HORIZON_DECODER_HIDDEN,
    ).to(device)


# ------------------------------------------------------------
# Parameter / initialization safety checks
# ------------------------------------------------------------
V35_PARAMETER_COUNTS = {}
for _variant in sorted(set(ENSEMBLE_VARIANTS)):
    _model = build_v35_model(_variant)
    _count = sum(
        p.numel() for p in _model.parameters() if p.requires_grad
    )
    V35_PARAMETER_COUNTS[_variant] = _count
    print(f"variant={_variant:24s} parameters={_count:,}")
    del _model

# The two proven anchor architectures must remain exactly v8-compatible.
assert V35_PARAMETER_COUNTS["physics"] == 1496961
assert V35_PARAMETER_COUNTS["wide"] == 2471169

gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


## 4. GPU 학습 + rolling checkpoint + milestone 안전 저장


In [ ]:
# ============================================================
# V3.5 Training / resume / fault-tolerant delivery
# ============================================================

class WeightEMA:
    def __init__(self, model, decay):
        self.module = copy.deepcopy(model).eval()
        for parameter in self.module.parameters():
            parameter.requires_grad_(False)
        self.decay = decay

    @torch.no_grad()
    def update(self, model):
        for ema_value, model_value in zip(
            self.module.state_dict().values(),
            model.state_dict().values(),
        ):
            if ema_value.dtype.is_floating_point:
                ema_value.mul_(self.decay).add_(
                    model_value.detach(),
                    alpha=1.0 - self.decay,
                )
            else:
                ema_value.copy_(model_value)


def mixup_alpha_for_variant(variant):
    return float(MIXUP_ALPHA_BY_VARIANT.get(variant, MIXUP_ALPHA))


def augment_batch(images, coronal):
    batch_size, time_steps = images.shape[0], images.shape[1]
    assert FLIP_PROBABILITY == 0.0

    vertical = (
        torch.rand(batch_size, device=images.device) < VFLIP_PROBABILITY
    )
    if vertical.any():
        images[vertical] = images[vertical].flip(-2)

    if BRIGHTNESS_JITTER > 0:
        scale = (
            1.0
            + (
                torch.rand(
                    batch_size, 1, 1, 1, 1,
                    device=images.device,
                )
                * 2.0
                - 1.0
            )
            * BRIGHTNESS_JITTER
        )
        images = (images * scale).clamp_(0.0, 1.0)

    if FRAME_DROP_PROBABILITY > 0:
        keep = (
            torch.rand(
                batch_size, time_steps, 1,
                device=images.device,
            )
            >= FRAME_DROP_PROBABILITY
        ).to(images.dtype)
        images = images * keep[:, :, :, None, None]
        coronal = coronal * keep

    return images, coronal


def _weighted_mse(squared, weights):
    weights = weights.to(dtype=squared.dtype, device=squared.device)
    return (squared * weights).sum() / weights.sum().clamp_min(1e-8)


def specialist_loss(prediction, target, persistence, variant):
    squared = (prediction - target) ** 2

    if variant == "near_earth_continuity":
        horizon_weights = torch.as_tensor(
            NEAR_CONTINUITY_HORIZON_WEIGHTS,
            device=prediction.device,
            dtype=prediction.dtype,
        ).unsqueeze(0).expand_as(squared)
        return _weighted_mse(squared, horizon_weights)

    if variant == "solar_long_range":
        horizon_weights = torch.as_tensor(
            SOLAR_LONG_HORIZON_WEIGHTS,
            device=prediction.device,
            dtype=prediction.dtype,
        ).unsqueeze(0).expand_as(squared)
        return _weighted_mse(squared, horizon_weights)

    if variant == "stream_onset":
        rise = target - persistence
        rise_gate = torch.sigmoid(
            (rise - ONSET_DELTA_CENTER) / ONSET_DELTA_WIDTH
        )
        speed_gate = torch.sigmoid(
            (target - 0.450) / 0.050
        )
        weights = 1.0 + 1.35 * rise_gate * speed_gate
        return _weighted_mse(squared, weights)

    if variant == "hss_peak":
        high_gate = torch.sigmoid(
            (target - HSS_SPEED_CENTER) / HSS_SPEED_WIDTH
        )
        sample_max = target.max(dim=1, keepdim=True).values
        near_peak = torch.exp(
            -((sample_max - target) / 0.060).square()
        )
        weights = 1.0 + 0.90 * high_gate + 0.80 * high_gate * near_peak
        return _weighted_mse(squared, weights)

    if variant == "hss_duration":
        high_gate = torch.sigmoid(
            (target - HSS_SPEED_CENTER) / HSS_SPEED_WIDTH
        )
        sustained = F.avg_pool1d(
            high_gate.unsqueeze(1),
            kernel_size=3,
            stride=1,
            padding=1,
        ).squeeze(1)
        weights = 1.0 + 1.20 * sustained
        return _weighted_mse(squared, weights)

    if variant == "hss_recovery":
        current_high = torch.sigmoid(
            (persistence - HSS_SPEED_CENTER) / HSS_SPEED_WIDTH
        )
        drop_gate = torch.sigmoid(
            (
                (persistence - target)
                - RECOVERY_DELTA_CENTER
            )
            / RECOVERY_DELTA_WIDTH
        )
        weights = 1.0 + 1.35 * current_high * drop_gate
        return _weighted_mse(squared, weights)

    if variant == "slow_wind":
        slow_gate = torch.sigmoid(
            (SLOW_WIND_CENTER - target) / SLOW_WIND_WIDTH
        )
        weights = 1.0 + 1.00 * slow_gate
        return _weighted_mse(squared, weights)

    if variant == "regime_change":
        change = (target - persistence).abs()
        change_gate = torch.sigmoid(
            (change - REGIME_CHANGE_CENTER) / REGIME_CHANGE_WIDTH
        )
        weights = 1.0 + 1.10 * change_gate
        return _weighted_mse(squared, weights)

    return F.mse_loss(prediction, target)


def train_one_epoch(model, ema, optimizer, scaler, variant):
    model.train()
    squared_error_sum = 0.0
    value_count = 0
    alpha = mixup_alpha_for_variant(variant)

    for batch in train_loader:
        images = batch["images"].to(
            DEVICE, non_blocking=PIN_MEMORY
        )
        wind = batch["wind"].to(
            DEVICE, non_blocking=PIN_MEMORY
        )
        coronal = batch["coronal"].to(
            DEVICE, non_blocking=PIN_MEMORY
        )
        persistence = batch["persistence"].to(
            DEVICE, non_blocking=PIN_MEMORY
        )
        target = batch["target"].to(
            DEVICE, non_blocking=PIN_MEMORY
        )

        images, coronal = augment_batch(images, coronal)

        if alpha > 0:
            lam = float(np.random.beta(alpha, alpha))
            perm = torch.randperm(
                images.shape[0],
                device=images.device,
            )
            images = lam * images + (1.0 - lam) * images[perm]
            wind = lam * wind + (1.0 - lam) * wind[perm]
            coronal = lam * coronal + (1.0 - lam) * coronal[perm]
            persistence = (
                lam * persistence + (1.0 - lam) * persistence[perm]
            )
            target = lam * target + (1.0 - lam) * target[perm]

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):
            prediction = model(images, wind, persistence, coronal)
            loss = specialist_loss(
                prediction,
                target,
                persistence,
                variant,
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(
            model.parameters(), GRAD_CLIP_NORM
        )
        scaler.step(optimizer)
        scaler.update()
        ema.update(model)

        error_km_s = (
            prediction.detach().float() - target
        ) * 1000.0
        squared_error_sum += float(
            torch.sum(error_km_s ** 2).cpu()
        )
        value_count += error_km_s.numel()

    return math.sqrt(squared_error_sum / value_count)


@torch.no_grad()
def evaluate_rmse(model):
    model.eval()
    squared_error_sum = 0.0
    value_count = 0

    for batch in val_loader:
        images = batch["images"].to(
            DEVICE, non_blocking=PIN_MEMORY
        )
        wind = batch["wind"].to(
            DEVICE, non_blocking=PIN_MEMORY
        )
        coronal = batch["coronal"].to(
            DEVICE, non_blocking=PIN_MEMORY
        )
        persistence = batch["persistence"].to(
            DEVICE, non_blocking=PIN_MEMORY
        )
        target = batch["target"].to(
            DEVICE, non_blocking=PIN_MEMORY
        )

        with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):
            prediction = model(images, wind, persistence, coronal)

        error_km_s = (
            prediction.float() - target
        ) * 1000.0
        squared_error_sum += float(
            torch.sum(error_km_s ** 2).cpu()
        )
        value_count += error_km_s.numel()

    return math.sqrt(squared_error_sum / value_count)


@torch.no_grad()
def predict_single_model(model, loader):
    model.eval()
    predictions = []
    row_indexes = []

    for batch in loader:
        images = batch["images"].to(
            DEVICE, non_blocking=PIN_MEMORY
        )
        wind = batch["wind"].to(
            DEVICE, non_blocking=PIN_MEMORY
        )
        coronal = batch["coronal"].to(
            DEVICE, non_blocking=PIN_MEMORY
        )
        persistence = batch["persistence"].to(
            DEVICE, non_blocking=PIN_MEMORY
        )

        with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):
            output = model(images, wind, persistence, coronal)

        predictions.append(
            output.float().cpu().numpy() * 1000.0
        )
        batch_rows = batch["row_index"]
        if isinstance(batch_rows, torch.Tensor):
            batch_rows = batch_rows.cpu().numpy()
        row_indexes.extend(
            np.asarray(batch_rows, dtype=np.int64).tolist()
        )

    return (
        np.concatenate(predictions, axis=0),
        np.asarray(row_indexes, dtype=np.int64),
    )


def overall_rmse(y_true, y_pred):
    return float(
        np.sqrt(np.mean((y_pred - y_true) ** 2))
    )


def make_lr_lambda(total_epochs):
    def lr_lambda(epoch):
        if epoch < WARMUP_EPOCHS:
            return (epoch + 1) / WARMUP_EPOCHS
        progress = (
            epoch - WARMUP_EPOCHS
        ) / max(1, total_epochs - WARMUP_EPOCHS)
        return 0.5 * (
            1.0
            + math.cos(
                math.pi * min(progress, 1.0)
            )
        )
    return lr_lambda


def capture_rng_state():
    result = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch_cpu": torch.get_rng_state(),
        "train_loader_generator": (
            train_loader.generator.get_state()
            if train_loader is not None
            and train_loader.generator is not None
            else None
        ),
    }
    if torch.cuda.is_available():
        result["torch_cuda"] = torch.cuda.get_rng_state_all()
    return result


def restore_rng_state(state):
    if not state:
        return
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch_cpu"])
    if torch.cuda.is_available() and "torch_cuda" in state:
        torch.cuda.set_rng_state_all(state["torch_cuda"])
    if (
        train_loader is not None
        and train_loader.generator is not None
        and state.get("train_loader_generator") is not None
    ):
        train_loader.generator.set_state(
            state["train_loader_generator"]
        )


# ============================================================
# Ensemble search: finite singles/pairs/triples/quads
# + V8-champion-centered specialist corrections. Selected support <= 4.
# ============================================================
def _candidate_key(vector):
    return tuple(np.round(vector, 12).tolist())


def _positive_simplex_weights(parts, units=SIMPLEX_UNITS):
    """Positive weights on a finite simplex. units=10 -> 0.10 increments."""
    if parts == 1:
        yield (1.0,)
        return

    def recurse(remaining_parts, remaining_units, prefix):
        if remaining_parts == 1:
            if remaining_units >= 1:
                yield tuple(prefix + [remaining_units / units])
            return
        min_for_rest = remaining_parts - 1
        for value in range(1, remaining_units - min_for_rest + 1):
            yield from recurse(
                remaining_parts - 1,
                remaining_units - value,
                prefix + [value / units],
            )

    yield from recurse(parts, units, [])


def generate_weight_candidates(member_count, member_rmse=None):
    assert 2 <= member_count <= ENSEMBLE_SIZE
    candidates = {}

    def add(name, items):
        vector = np.zeros(member_count, dtype=np.float64)
        for member_index, weight in items:
            member_index = int(member_index)
            if not (0 <= member_index < member_count):
                return
            vector[member_index] = float(weight)
        if vector.sum() <= 0:
            return
        vector /= vector.sum()
        key = _candidate_key(vector)
        if key not in candidates:
            candidates[key] = name

    # Singles are diagnostic only; final selector requires support >=2.
    for member_index in range(member_count):
        add(f"single_m{member_index}", [(member_index, 1.0)])

    # Fine pair search: 0.05 increments.
    for first, second in itertools.combinations(range(member_count), 2):
        for first_weight in PAIR_WEIGHT_GRID:
            add(
                f"pair_m{first}_m{second}_w{first_weight:.2f}_{1-first_weight:.2f}",
                [(first, first_weight), (second, 1.0 - first_weight)],
            )

    # Positive triple simplex, 0.10 increments.
    for triple in itertools.combinations(range(member_count), 3):
        for weights in _positive_simplex_weights(3):
            add(
                "triple_"
                + "_".join(f"m{i}" for i in triple)
                + "_"
                + "_".join(f"{w:.2f}" for w in weights),
                list(zip(triple, weights)),
            )

    # Positive quad simplex, 0.10 increments.
    for quartet in itertools.combinations(range(member_count), 4):
        for weights in _positive_simplex_weights(4):
            add(
                "quad_"
                + "_".join(f"m{i}" for i in quartet)
                + "_"
                + "_".join(f"{w:.2f}" for w in weights),
                list(zip(quartet, weights)),
            )

    # When all V8 anchor members exist, explicitly explore the proven
    # champion pair plus small specialist corrections.
    if max(CHAMPION_MEMBERS) < member_count:
        champion_first, champion_second = CHAMPION_MEMBERS
        add(
            "v8_champion_equal",
            [(champion_first, 0.5), (champion_second, 0.5)],
        )

        available_specialists = [
            i for i in SPECIALIST_MEMBERS if i < member_count
        ]
        for specialist in available_specialists:
            for mass in CHAMPION_SPECIALIST_MASS_GRID:
                anchor_mass = 1.0 - float(mass)
                add(
                    f"champion_plus_m{specialist}_mass{mass:.2f}",
                    [
                        (champion_first, anchor_mass / 2.0),
                        (champion_second, anchor_mass / 2.0),
                        (specialist, mass),
                    ],
                )

        for first_spec, second_spec in itertools.combinations(
            available_specialists, 2
        ):
            for total_mass in CHAMPION_TWO_SPECIALIST_MASS_GRID:
                anchor_mass = 1.0 - float(total_mass)
                # equal specialist split
                add(
                    (
                        f"champion_plus_m{first_spec}_m{second_spec}_"
                        f"mass{total_mass:.2f}_equal"
                    ),
                    [
                        (champion_first, anchor_mass / 2.0),
                        (champion_second, anchor_mass / 2.0),
                        (first_spec, total_mass / 2.0),
                        (second_spec, total_mass / 2.0),
                    ],
                )
                # 2:1 / 1:2 specialist split
                for ratio_first in (2.0 / 3.0, 1.0 / 3.0):
                    add(
                        (
                            f"champion_plus_m{first_spec}_m{second_spec}_"
                            f"mass{total_mass:.2f}_r{ratio_first:.2f}"
                        ),
                        [
                            (champion_first, anchor_mass / 2.0),
                            (champion_second, anchor_mass / 2.0),
                            (first_spec, total_mass * ratio_first),
                            (second_spec, total_mass * (1.0 - ratio_first)),
                        ],
                    )

    if member_rmse is not None:
        order = np.argsort(np.asarray(member_rmse, dtype=np.float64))
        for k in range(2, min(MAX_SELECTED_MEMBERS, member_count) + 1):
            top = order[:k].tolist()
            add(
                f"top{k}_individual_equal",
                [(i, 1.0 / k) for i in top],
            )

    add(
        "all_members_equal_diagnostic",
        [(i, 1.0 / member_count) for i in range(member_count)],
    )

    return [
        {
            "name": name,
            "weights": np.asarray(key, dtype=np.float64),
        }
        for key, name in candidates.items()
    ]


def weighted_prediction(member_predictions, weights):
    weights = np.asarray(weights, dtype=np.float64)
    nonzero = np.flatnonzero(weights > 1e-12)
    assert nonzero.size > 0
    return np.tensordot(
        weights[nonzero],
        member_predictions[nonzero],
        axes=(0, 0),
    )


def search_weighted_ensemble(member_predictions, y_true):
    member_predictions = np.asarray(
        member_predictions, dtype=np.float64
    )
    y_true = np.asarray(y_true, dtype=np.float64)
    member_count = member_predictions.shape[0]
    assert member_count >= 2
    assert member_predictions.shape[1:] == y_true.shape

    individual_rmse = [
        overall_rmse(y_true, member_predictions[i])
        for i in range(member_count)
    ]
    candidates = generate_weight_candidates(
        member_count, individual_rmse
    )

    split = len(y_true) // 2
    y_a, y_b = y_true[:split], y_true[split:]

    rows = []
    for candidate in candidates:
        prediction = weighted_prediction(
            member_predictions,
            candidate["weights"],
        )
        score = overall_rmse(y_true, prediction)
        score_a = overall_rmse(y_a, prediction[:split])
        score_b = overall_rmse(y_b, prediction[split:])
        support = int(
            np.count_nonzero(
                candidate["weights"] > 1e-10
            )
        )
        rows.append(
            {
                "name": candidate["name"],
                "val_rmse_km_s": score,
                "half_a_rmse_km_s": score_a,
                "half_b_rmse_km_s": score_b,
                "support": support,
                "weights": candidate["weights"],
            }
        )

    rows.sort(
        key=lambda row: (
            row["val_rmse_km_s"],
            row["support"],
            row["name"],
        )
    )

    # Preferred baseline is the exact V8 champion pair once member 3 exists.
    # Before that (emergency partial run), use the best available pair.
    if max(CHAMPION_MEMBERS) < member_count:
        champion_weights = np.zeros(member_count, dtype=np.float64)
        champion_weights[list(CHAMPION_MEMBERS)] = 0.5
        baseline_name = "v8_champion_equal"
    else:
        pair_rows = [row for row in rows if row["support"] == 2]
        assert pair_rows
        champion_weights = np.asarray(
            pair_rows[0]["weights"], dtype=np.float64
        )
        baseline_name = "best_available_pair_partial"

    champion_prediction = weighted_prediction(
        member_predictions, champion_weights
    )
    champion_score = overall_rmse(y_true, champion_prediction)
    champion_a = overall_rmse(y_a, champion_prediction[:split])
    champion_b = overall_rmse(y_b, champion_prediction[split:])

    eligible = []
    for row in rows:
        if not (2 <= row["support"] <= MAX_SELECTED_MEMBERS):
            continue
        gain = champion_score - row["val_rmse_km_s"]
        half_ok = (
            row["half_a_rmse_km_s"]
            <= champion_a + HALF_SPLIT_WORSE_TOLERANCE_KM_S
            and row["half_b_rmse_km_s"]
            <= champion_b + HALF_SPLIT_WORSE_TOLERANCE_KM_S
        )
        if gain >= MIN_ENSEMBLE_GAIN_KM_S and half_ok:
            eligible.append(row)

    if eligible:
        selected = eligible[0]
        selection_note = (
            "best_sparse_candidate_beating_current_v8_champion_with_half_split_guard"
        )
    else:
        selected = {
            "name": baseline_name + "_robust_fallback",
            "val_rmse_km_s": champion_score,
            "half_a_rmse_km_s": champion_a,
            "half_b_rmse_km_s": champion_b,
            "support": int(np.count_nonzero(champion_weights > 1e-10)),
            "weights": champion_weights,
        }
        selection_note = "v8_champion_or_best_partial_fallback"

    return {
        "rows": rows,
        "selected": selected,
        "champion_score": champion_score,
        "champion_half_a": champion_a,
        "champion_half_b": champion_b,
        "candidate_count": len(rows),
        "eligible_count": len(eligible),
        "selection_note": selection_note,
        "individual_rmse": individual_rmse,
    }


# ============================================================
# Persistent rolling training state
# ============================================================
def make_train_state_payload(
    completed_state_dicts,
    completed_rmse,
    completed_predictions,
    completed_epochs_actual,
    history_rows,
):
    count = len(completed_state_dicts)
    return {
        "architecture_version": ARCHITECTURE_VERSION,
        "completed_count": count,
        "target_count": ENSEMBLE_SIZE,
        "variants": list(ENSEMBLE_VARIANTS[:count]),
        "labels": list(MEMBER_LABELS[:count]),
        "seeds": list(MEMBER_SEEDS[:count]),
        "model_state_dicts": completed_state_dicts,
        "per_member_val_rmse": list(completed_rmse),
        "member_validation_predictions": [
            np.asarray(p, dtype=np.float32)
            for p in completed_predictions
        ],
        "member_epochs_actual": list(completed_epochs_actual),
        "history_rows": list(history_rows),
        "rng_state": capture_rng_state(),
    }


def validate_train_state(payload):
    assert payload["architecture_version"] == ARCHITECTURE_VERSION
    count = int(payload["completed_count"])
    assert 0 <= count <= ENSEMBLE_SIZE
    assert int(payload["target_count"]) == ENSEMBLE_SIZE
    assert tuple(payload["variants"]) == ENSEMBLE_VARIANTS[:count]
    assert tuple(payload["labels"]) == MEMBER_LABELS[:count]
    assert tuple(int(x) for x in payload["seeds"]) == MEMBER_SEEDS[:count]
    assert len(payload["model_state_dicts"]) == count
    assert len(payload["per_member_val_rmse"]) == count
    assert len(payload["member_validation_predictions"]) == count
    assert len(payload["member_epochs_actual"]) == count
    if count:
        prediction_stack = np.asarray(
            payload["member_validation_predictions"],
            dtype=np.float32,
        )
        assert prediction_stack.shape == (
            count,
            EXPECTED_COUNTS["validation"],
            TARGET_HORIZONS,
        )
        assert np.isfinite(prediction_stack).all()
    return payload


def save_current_member_checkpoint(
    *,
    member_index,
    variant,
    seed,
    planned_epochs,
    epoch,
    model,
    ema,
    optimizer,
    scheduler,
    scaler,
    best_val_rmse,
    best_source,
    best_state_dict,
    epoch_seconds,
):
    payload = {
        "architecture_version": ARCHITECTURE_VERSION,
        "member_index": int(member_index),
        "variant": variant,
        "seed": int(seed),
        "planned_epochs": int(planned_epochs),
        "epoch": int(epoch),
        "model_state_dict": model.state_dict(),
        "ema_state_dict": ema.module.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_val_rmse": float(best_val_rmse),
        "best_source": best_source,
        "best_state_dict": best_state_dict,
        "epoch_seconds": list(epoch_seconds),
        "rng_state": capture_rng_state(),
    }
    atomic_torch_save(payload, CURRENT_MEMBER_PATH)


def load_current_member_checkpoint(
    member_index,
    variant,
    seed,
    planned_epochs,
):
    if not (RESUME_PARTIAL and CURRENT_MEMBER_PATH.is_file()):
        return None
    try:
        payload = safe_torch_load(
            CURRENT_MEMBER_PATH,
            map_location=DEVICE,
        )
        valid = (
            payload.get("architecture_version") == ARCHITECTURE_VERSION
            and int(payload.get("member_index", -1)) == member_index
            and payload.get("variant") == variant
            and int(payload.get("seed", -1)) == seed
            and int(payload.get("planned_epochs", -1)) == planned_epochs
        )
        if not valid:
            print("stale current_member.pth ignored and removed")
            CURRENT_MEMBER_PATH.unlink(missing_ok=True)
            return None
        return payload
    except Exception as error:
        print("current_member.pth load failed; starting member fresh:", repr(error))
        CURRENT_MEMBER_PATH.unlink(missing_ok=True)
        return None


# ============================================================
# Delivery model/submission helpers
# model.pth stores only selected members, keeping it small and inference-safe.
# ============================================================
def make_delivery_checkpoint(
    completed_state_dicts,
    completed_epochs_actual,
    search_result,
):
    selected = search_result["selected"]
    full_weights = np.asarray(
        selected["weights"], dtype=np.float64
    )
    selected_indices = np.flatnonzero(
        full_weights > 1e-10
    ).tolist()
    assert 2 <= len(selected_indices) <= MAX_SELECTED_MEMBERS

    selected_weights = full_weights[selected_indices]
    selected_weights = selected_weights / selected_weights.sum()
    completed_count = len(completed_state_dicts)

    checkpoint = {
        "architecture_version": ARCHITECTURE_VERSION,
        "delivery_format_version": DELIVERY_FORMAT_VERSION,
        "model_state_dicts": [
            completed_state_dicts[i]
            for i in selected_indices
        ],
        "variants": [
            ENSEMBLE_VARIANTS[i]
            for i in selected_indices
        ],
        "labels": [
            MEMBER_LABELS[i]
            for i in selected_indices
        ],
        "seeds": [
            MEMBER_SEEDS[i]
            for i in selected_indices
        ],
        "original_member_indices": selected_indices,
        "ensemble_weights": selected_weights.tolist(),
        "validation_rmse_km_s": float(
            selected["val_rmse_km_s"]
        ),
        "ensemble_strategy": selected["name"],
        "ensemble_search_count": int(
            search_result["candidate_count"]
        ),
        "ensemble_selection_note": search_result["selection_note"],
        "completed_member_count": completed_count,
        "planned_member_count": ENSEMBLE_SIZE,
        "member_epochs_actual": [
            int(completed_epochs_actual[i])
            for i in selected_indices
        ],
        "channels": list(CHANNELS),
        "wind_feature_dim": WIND_FEATURE_DIM,
        "dropout": DROPOUT,
        "conv_dropout": CONV_DROPOUT,
        "image_size": IMAGE_SIZE,
        "input_steps": 20,
        "target_horizons": TARGET_HORIZONS,
        "pool_height": POOL_HEIGHT,
        "pool_width": POOL_WIDTH,
        "decoder_config": {
            "horizon_embed_dim": HORIZON_EMBED_DIM,
            "image_attention_dim": IMAGE_ATTENTION_DIM,
            "wind_gru_hidden": WIND_GRU_HIDDEN,
            "wind_attention_dim": WIND_ATTENTION_DIM,
            "horizon_decoder_hidden": HORIZON_DECODER_HIDDEN,
        },
        "coronal_feature_config": {
            "bands": CH_BANDS,
            "thresholds": list(CH_THRESHOLDS),
            "equator_fraction": CH_EQUATOR_FRACTION,
            "disk_shrink": DISK_SHRINK,
            "zones": CH_ZONES,
            "feature_per_step": CH_FEATURE_PER_STEP,
        },
        "coronal_geometry": {
            "center_y": float(DISK_CENTER_Y),
            "center_x": float(DISK_CENTER_X),
            "radius": float(DISK_RADIUS),
            "native_shape": list(NATIVE_SHAPE),
        },
        "prediction_mode": (
            "persistence_plus_linear_ar_plus_"
            "specialist_horizon_delta_scaled_by_1000"
        ),
        "use_tta": USE_TTA,
        "initialization": "random_from_scratch_v34_v8_replay_elite",
        "external_data_used": False,
        "pretrained_weights_used": False,
        "train_validation_combined": False,
    }
    validate_delivery_checkpoint(checkpoint)
    return checkpoint


def make_stable_test_loader():
    dataset = SolarWindDataset(
        test_image_array,
        test_image_index,
        test_inputs,
        test_index,
        targets=None,
        coronal_array=test_coronal_array,
        coronal_index=test_coronal_index,
    )
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )
    return dataset, loader


@torch.no_grad()
def predict_delivery_checkpoint(checkpoint, loader):
    validate_delivery_checkpoint(checkpoint)
    weights = np.asarray(
        checkpoint["ensemble_weights"],
        dtype=np.float64,
    )
    accumulated = None
    expected_rows = None

    for local_index, (
        variant,
        state_dict,
        weight,
    ) in enumerate(
        zip(
            checkpoint["variants"],
            checkpoint["model_state_dicts"],
            weights,
        )
    ):
        model = build_v35_model(variant)
        model.load_state_dict(state_dict, strict=True)
        prediction, rows = predict_single_model(
            model, loader
        )
        if expected_rows is None:
            expected_rows = rows
            accumulated = (
                prediction.astype(np.float64) * float(weight)
            )
        else:
            assert np.array_equal(rows, expected_rows)
            accumulated += (
                prediction.astype(np.float64) * float(weight)
            )

        print(
            f"  inference selected[{local_index}] "
            f"{variant} weight={weight:.4f}"
        )
        del model
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    assert accumulated is not None
    return accumulated, expected_rows


def atomic_csv_save(frame, path):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def write_delivery_pair(checkpoint, tag):
    print()
    print("=" * 72)
    print(f"V3.5 DELIVERY: {tag}")
    print("=" * 72)
    print(
        "selected original members:",
        checkpoint["original_member_indices"],
    )
    print(
        "selected labels:",
        checkpoint["labels"],
    )
    print(
        "weights:",
        np.round(
            checkpoint["ensemble_weights"], 4
        ).tolist(),
    )
    print(
        "Validation RMSE:",
        f"{checkpoint['validation_rmse_km_s']:.6f}",
    )

    test_dataset_local, test_loader_local = make_stable_test_loader()
    prediction, rows = predict_delivery_checkpoint(
        checkpoint,
        test_loader_local,
    )
    assert np.array_equal(rows, test_index)
    assert prediction.shape == (
        EXPECTED_COUNTS["test"], TARGET_HORIZONS
    )
    assert np.isfinite(prediction).all()

    submission = pd.DataFrame(
        prediction,
        columns=TARGET_COLUMNS,
    )
    submission.insert(
        0,
        "sample_id",
        test_ids.sample_id.tolist(),
    )
    assert submission.shape == (
        EXPECTED_COUNTS["test"], 13
    )
    assert submission.columns.tolist() == EXPECTED_SUBMISSION_COLUMNS
    assert submission.sample_id.is_unique
    assert (
        submission.sample_id.tolist()
        == test_ids.sample_id.tolist()
    )
    assert np.isfinite(
        submission[TARGET_COLUMNS].to_numpy(np.float64)
    ).all()

    # Save both by replacing existing files; no version pile-up.
    atomic_torch_save(checkpoint, OUTPUT_CHECKPOINT_PATH)
    atomic_csv_save(submission, OUTPUT_SUBMISSION_PATH)

    atomic_json_save(
        {
            "tag": tag,
            "completed_member_count": checkpoint[
                "completed_member_count"
            ],
            "validation_rmse_km_s": checkpoint[
                "validation_rmse_km_s"
            ],
            "selected_member_indices": checkpoint[
                "original_member_indices"
            ],
            "selected_labels": checkpoint["labels"],
            "weights": checkpoint["ensemble_weights"],
            "ensemble_strategy": checkpoint[
                "ensemble_strategy"
            ],
            "elapsed": format_duration(elapsed_seconds()),
        },
        OUTPUT_LOG_PATH,
    )

    del test_loader_local, test_dataset_local
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    print("delivery model:", OUTPUT_CHECKPOINT_PATH)
    print("delivery csv  :", OUTPUT_SUBMISSION_PATH)
    return submission


def run_ensemble_and_delivery(
    completed_state_dicts,
    completed_predictions,
    completed_epochs_actual,
    *,
    tag,
    save_search_csv=False,
):
    assert len(completed_state_dicts) >= 2
    prediction_stack = np.stack(
        completed_predictions,
        axis=0,
    ).astype(np.float64)
    target = np.asarray(
        val_targets[val_index],
        dtype=np.float64,
    )
    search_result = search_weighted_ensemble(
        prediction_stack,
        target,
    )

    if save_search_csv:
        rows = []
        for rank, row in enumerate(
            search_result["rows"], start=1
        ):
            rows.append(
                {
                    "rank": rank,
                    "name": row["name"],
                    "val_rmse_km_s": row[
                        "val_rmse_km_s"
                    ],
                    "half_a_rmse_km_s": row[
                        "half_a_rmse_km_s"
                    ],
                    "half_b_rmse_km_s": row[
                        "half_b_rmse_km_s"
                    ],
                    "support": row["support"],
                    "members": ",".join(
                        str(i)
                        for i in np.flatnonzero(
                            row["weights"] > 1e-10
                        )
                    ),
                    "weights": json.dumps(
                        np.round(
                            row["weights"], 6
                        ).tolist()
                    ),
                }
            )
        pd.DataFrame(rows).to_csv(
            OUTPUT_DIR / "ensemble_search.csv",
            index=False,
        )

    checkpoint = make_delivery_checkpoint(
        completed_state_dicts,
        completed_epochs_actual,
        search_result,
    )
    write_delivery_pair(checkpoint, tag)
    return search_result, checkpoint


# ============================================================
# Training state initialization / resume
# ============================================================
completed_state_dicts = []
completed_rmse = []
completed_predictions = []
completed_epochs_actual = []
history_rows = []
final_search_result = None
active_checkpoint = loaded_checkpoint
last_delivery_count = 0
time_stop_requested = False


# CODE8 checkpoint-retention hook only. It never changes training/RNG/optimizer flow.
CODE8_SNAPSHOT_BANK = {}


def code8_retain_snapshot(member_index, source_name, epoch, rmse, source_model):
    member_bank = CODE8_SNAPSHOT_BANK.setdefault(
        int(member_index), {"raw": [], "ema": []}
    )
    candidates = member_bank[source_name]
    if len(candidates) >= 2 and float(rmse) >= candidates[-1]["rmse"]:
        return
    state_dict = {
        key: value.detach().clone().cpu()
        for key, value in source_model.state_dict().items()
    }
    candidates.append({
        "source": source_name,
        "epoch": int(epoch),
        "rmse": float(rmse),
        "state_dict": state_dict,
    })
    candidates.sort(key=lambda item: (item["rmse"], item["epoch"]))
    del candidates[2:]

if TRAIN_MODEL:
    if RESUME_PARTIAL and TRAIN_STATE_PATH.is_file():
        try:
            train_state = safe_torch_load(
                TRAIN_STATE_PATH,
                map_location="cpu",
            )
            if (
                train_state.get("architecture_version")
                == ARCHITECTURE_VERSION
            ):
                validate_train_state(train_state)
                completed_state_dicts = list(
                    train_state["model_state_dicts"]
                )
                completed_rmse = [
                    float(x)
                    for x in train_state[
                        "per_member_val_rmse"
                    ]
                ]
                completed_predictions = [
                    np.asarray(p, dtype=np.float32)
                    for p in train_state[
                        "member_validation_predictions"
                    ]
                ]
                completed_epochs_actual = [
                    int(x)
                    for x in train_state[
                        "member_epochs_actual"
                    ]
                ]
                history_rows = list(
                    train_state.get("history_rows") or []
                )
                restore_rng_state(
                    train_state.get("rng_state")
                )
                print(
                    "RESUME TRAIN STATE:",
                    f"{len(completed_state_dicts)}/{ENSEMBLE_SIZE}",
                )
            else:
                print("old train_state architecture; removing")
                TRAIN_STATE_PATH.unlink(missing_ok=True)
                CURRENT_MEMBER_PATH.unlink(missing_ok=True)
        except Exception as error:
            print(
                "train_state load failed; starting fresh:",
                repr(error),
            )
            TRAIN_STATE_PATH.unlink(missing_ok=True)
            CURRENT_MEMBER_PATH.unlink(missing_ok=True)

    start_member = len(completed_state_dicts)
    if start_member == 0:
        print(
            "initialized V3.5 11-member priority run "
            "(external output only)"
        )

    try:
        for member_index in range(
            start_member, ENSEMBLE_SIZE
        ):
            if training_seconds_left() <= 120:
                print(
                    "TIME GUARD: not starting another member. "
                    "Finalizing completed members."
                )
                time_stop_requested = True
                break

            variant = ENSEMBLE_VARIANTS[member_index]
            label = MEMBER_LABELS[member_index]
            member_seed = MEMBER_SEEDS[member_index]
            planned_epochs = MEMBER_EPOCHS[member_index]

            print()
            print("=" * 72)
            print(
                f"member {member_index:02d}/{ENSEMBLE_SIZE-1:02d} "
                f"| {label} | {variant} "
                f"| seed={member_seed} "
                f"| planned={planned_epochs}"
            )
            print(
                "training time left:",
                format_duration(training_seconds_left()),
            )
            print("=" * 72)

            torch.manual_seed(member_seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(member_seed)

            model = build_v35_model(variant)
            optimizer = torch.optim.AdamW(
                model.parameters(),
                lr=3e-4,
                weight_decay=WEIGHT_DECAY,
            )
            scheduler = torch.optim.lr_scheduler.LambdaLR(
                optimizer,
                make_lr_lambda(planned_epochs),
            )
            scaler = torch.amp.GradScaler(
                DEVICE.type,
                enabled=USE_AMP,
            )
            ema = WeightEMA(model, EMA_DECAY)

            best_val_rmse = float("inf")
            best_state_dict = None
            best_source = None
            start_epoch = 1
            epoch_seconds = []

            current_payload = load_current_member_checkpoint(
                member_index,
                variant,
                member_seed,
                planned_epochs,
            )
            if current_payload is not None:
                model.load_state_dict(
                    current_payload["model_state_dict"],
                    strict=True,
                )
                ema.module.load_state_dict(
                    current_payload["ema_state_dict"],
                    strict=True,
                )
                optimizer.load_state_dict(
                    current_payload[
                        "optimizer_state_dict"
                    ]
                )
                scheduler.load_state_dict(
                    current_payload[
                        "scheduler_state_dict"
                    ]
                )
                scaler.load_state_dict(
                    current_payload[
                        "scaler_state_dict"
                    ]
                )
                best_val_rmse = float(
                    current_payload["best_val_rmse"]
                )
                best_source = current_payload[
                    "best_source"
                ]
                best_state_dict = current_payload[
                    "best_state_dict"
                ]
                start_epoch = int(
                    current_payload["epoch"]
                ) + 1
                epoch_seconds = [
                    float(x)
                    for x in current_payload.get(
                        "epoch_seconds", []
                    )
                ]
                restore_rng_state(
                    current_payload.get("rng_state")
                )
                print(
                    "RESUME CURRENT MEMBER:",
                    f"epoch {start_epoch}/{planned_epochs}",
                    f"best={best_val_rmse:.6f}",
                )

            member_time_stop = False
            last_epoch_completed = start_epoch - 1

            for epoch in range(
                start_epoch, planned_epochs + 1
            ):
                typical_epoch = (
                    float(np.median(epoch_seconds[-5:]))
                    if epoch_seconds
                    else 30.0
                )
                if training_seconds_left() < max(
                    120.0,
                    1.75 * typical_epoch,
                ):
                    print(
                        "TIME GUARD triggered before epoch",
                        epoch,
                    )
                    member_time_stop = True
                    break

                started = time.perf_counter()
                train_rmse = train_one_epoch(
                    model,
                    ema,
                    optimizer,
                    scaler,
                    variant,
                )
                raw_val_rmse = evaluate_rmse(model)
                ema_val_rmse = evaluate_rmse(ema.module)
                scheduler.step()

                # CODE8 retention only: no RNG draw and no training-state mutation.
                code8_retain_snapshot(
                    member_index, "raw", epoch, raw_val_rmse, model
                )
                code8_retain_snapshot(
                    member_index, "ema", epoch, ema_val_rmse, ema.module
                )

                if ema_val_rmse <= raw_val_rmse:
                    val_rmse = ema_val_rmse
                    source_model = ema.module
                    source_name = "ema"
                else:
                    val_rmse = raw_val_rmse
                    source_model = model
                    source_name = "raw"

                elapsed = time.perf_counter() - started
                epoch_seconds.append(elapsed)
                last_epoch_completed = epoch

                if val_rmse < best_val_rmse:
                    best_val_rmse = float(val_rmse)
                    best_source = source_name
                    best_state_dict = {
                        key: value.detach().clone().cpu()
                        for key, value in source_model.state_dict().items()
                    }

                learning_rate = optimizer.param_groups[0]["lr"]
                history_rows.append(
                    {
                        "member": member_index,
                        "label": label,
                        "variant": variant,
                        "seed": member_seed,
                        "epoch": epoch,
                        "planned_epochs": planned_epochs,
                        "train_rmse_km_s": train_rmse,
                        "val_rmse_km_s": val_rmse,
                        "raw_val_rmse_km_s": raw_val_rmse,
                        "ema_val_rmse_km_s": ema_val_rmse,
                        "selected": source_name,
                        "learning_rate": learning_rate,
                        "seconds": elapsed,
                        "elapsed_total": elapsed_seconds(),
                    }
                )

                print(
                    f"m={member_index:02d} "
                    f"epoch={epoch:03d}/{planned_epochs:03d} "
                    f"train={train_rmse:.3f} "
                    f"val={val_rmse:.3f}({source_name}) "
                    f"raw={raw_val_rmse:.3f} "
                    f"ema={ema_val_rmse:.3f} "
                    f"sec={elapsed:.1f} "
                    f"left={format_duration(training_seconds_left())}"
                )

                if (
                    epoch % CURRENT_CHECKPOINT_EVERY_EPOCHS == 0
                    or epoch == planned_epochs
                ):
                    save_current_member_checkpoint(
                        member_index=member_index,
                        variant=variant,
                        seed=member_seed,
                        planned_epochs=planned_epochs,
                        epoch=epoch,
                        model=model,
                        ema=ema,
                        optimizer=optimizer,
                        scheduler=scheduler,
                        scaler=scaler,
                        best_val_rmse=best_val_rmse,
                        best_source=best_source,
                        best_state_dict=best_state_dict,
                        epoch_seconds=epoch_seconds,
                    )

                if training_seconds_left() <= 120:
                    print(
                        "TIME GUARD: current member will be "
                        "closed after this epoch."
                    )
                    member_time_stop = True
                    break

            epochs_done_this_member = last_epoch_completed
            keep_partial = (
                best_state_dict is not None
                and (
                    not member_time_stop
                    or epochs_done_this_member
                    >= MIN_PARTIAL_MEMBER_EPOCHS
                )
            )

            if keep_partial:
                model.load_state_dict(
                    best_state_dict,
                    strict=True,
                )
                member_prediction, rows = predict_single_model(
                    model, val_loader
                )
                assert np.array_equal(rows, val_index)
                member_target = np.asarray(
                    val_targets[val_index],
                    dtype=np.float64,
                )
                direct_rmse = overall_rmse(
                    member_target,
                    member_prediction,
                )

                completed_state_dicts.append(
                    best_state_dict
                )
                completed_rmse.append(
                    float(direct_rmse)
                )
                completed_predictions.append(
                    member_prediction.astype(np.float32)
                )
                completed_epochs_actual.append(
                    int(epochs_done_this_member)
                )

                print(
                    f"member={member_index:02d} DONE "
                    f"| {label} "
                    f"| val={direct_rmse:.6f} "
                    f"| best={best_source} "
                    f"| epochs={epochs_done_this_member}"
                )

                train_state = make_train_state_payload(
                    completed_state_dicts,
                    completed_rmse,
                    completed_predictions,
                    completed_epochs_actual,
                    history_rows,
                )
                atomic_torch_save(
                    train_state,
                    TRAIN_STATE_PATH,
                )
                pd.DataFrame(history_rows).to_csv(
                    OUTPUT_DIR / "history.csv",
                    index=False,
                )
                CURRENT_MEMBER_PATH.unlink(
                    missing_ok=True
                )

                completed_count = len(
                    completed_state_dicts
                )
                if completed_count in DELIVERY_MILESTONES:
                    search_result, milestone_checkpoint = (
                        run_ensemble_and_delivery(
                            completed_state_dicts,
                            completed_predictions,
                            completed_epochs_actual,
                            tag=f"milestone_{completed_count:02d}",
                            save_search_csv=False,
                        )
                    )
                    final_search_result = search_result
                    active_checkpoint = milestone_checkpoint
                    last_delivery_count = completed_count
                    if completed_count == 4:
                        replay_score = float(search_result["champion_score"])
                        replay_delta = replay_score - BASELINE_V8_VALIDATION_RMSE
                        print(
                            "V8 REPLAY CHECK | physics778 + wide780 1:1 | "
                            f"val={replay_score:.6f} | "
                            f"reference={BASELINE_V8_VALIDATION_RMSE:.6f} | "
                            f"delta={replay_delta:+.6f}"
                        )
                        if abs(replay_delta) > 0.50:
                            print(
                                "WARNING: V8 replay differs from the historical "
                                "Validation score by >0.50 km/s. "
                                "Run continues; final selector still uses the "
                                "current replayed champion as the fallback."
                            )

            else:
                print(
                    f"member={member_index:02d} discarded: "
                    f"only {epochs_done_this_member} epochs "
                    f"before time guard."
                )
                CURRENT_MEMBER_PATH.unlink(
                    missing_ok=True
                )

            del (
                model,
                optimizer,
                scheduler,
                scaler,
                ema,
            )
            gc.collect()
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

            if member_time_stop:
                time_stop_requested = True
                break

        # --------------------------------------------------------
        # Final ensemble and delivery (works even if time guard
        # stopped after a prefix of the 11-member priority list)
        # --------------------------------------------------------
        completed_count = len(completed_state_dicts)
        if completed_count < 2:
            raise RuntimeError(
                "V3.5 could not complete the two anchor members; "
                "no valid external delivery pair can be finalized."
            )

        if last_delivery_count != completed_count:
            final_search_result, active_checkpoint = (
                run_ensemble_and_delivery(
                    completed_state_dicts,
                    completed_predictions,
                    completed_epochs_actual,
                    tag=(
                        f"final_partial_{completed_count:02d}"
                        if completed_count < ENSEMBLE_SIZE
                        else "final_07"
                    ),
                    save_search_csv=True,
                )
            )
            last_delivery_count = completed_count
        else:
            # Re-run search only to save complete final comparison table.
            prediction_stack = np.stack(
                completed_predictions,
                axis=0,
            ).astype(np.float64)
            member_target = np.asarray(
                val_targets[val_index],
                dtype=np.float64,
            )
            final_search_result = search_weighted_ensemble(
                prediction_stack,
                member_target,
            )
            rows_for_csv = []
            for rank, row in enumerate(
                final_search_result["rows"],
                start=1,
            ):
                rows_for_csv.append(
                    {
                        "rank": rank,
                        "name": row["name"],
                        "val_rmse_km_s": row[
                            "val_rmse_km_s"
                        ],
                        "half_a_rmse_km_s": row[
                            "half_a_rmse_km_s"
                        ],
                        "half_b_rmse_km_s": row[
                            "half_b_rmse_km_s"
                        ],
                        "support": row["support"],
                        "members": ",".join(
                            str(i)
                            for i in np.flatnonzero(
                                row["weights"] > 1e-10
                            )
                        ),
                        "weights": json.dumps(
                            np.round(
                                row["weights"], 6
                            ).tolist()
                        ),
                    }
                )
            pd.DataFrame(
                rows_for_csv
            ).to_csv(
                OUTPUT_DIR / "ensemble_search.csv",
                index=False,
            )
            active_checkpoint = safe_torch_load(
                OUTPUT_CHECKPOINT_PATH,
                map_location="cpu",
            )
            validate_delivery_checkpoint(
                active_checkpoint
            )

        print("Final V3.5 delivery kept in external output directory only:", OUTPUT_DIR)

    except Exception as error:
        print()
        print("=" * 72)
        print("V3.5 CAUGHT AN ERROR DURING TRAINING")
        print("=" * 72)
        traceback.print_exc()

        # Best-effort emergency delivery from already completed members.
        if len(completed_state_dicts) >= 2:
            try:
                print(
                    "Attempting emergency delivery from "
                    f"{len(completed_state_dicts)} completed members..."
                )
                final_search_result, active_checkpoint = (
                    run_ensemble_and_delivery(
                        completed_state_dicts,
                        completed_predictions,
                        completed_epochs_actual,
                        tag=(
                            f"emergency_{len(completed_state_dicts):02d}"
                        ),
                        save_search_csv=True,
                    )
                )
                print(
                    "Emergency model/submission saved successfully "
                    "to the external output directory."
                )
            except Exception:
                print(
                    "Emergency delivery also failed; "
                    "last previously written milestone files remain untouched."
                )
                traceback.print_exc()
        raise

else:
    validate_delivery_checkpoint(active_checkpoint)


print()
print("=" * 72)
print("V3.5 TRAINING STAGE COMPLETE")
print("=" * 72)
if TRAIN_MODEL:
    print(
        "completed members:",
        len(completed_state_dicts),
        "/",
        ENSEMBLE_SIZE,
    )
    print(
        "actual epochs:",
        completed_epochs_actual,
    )
    print(
        "time guard used:",
        time_stop_requested,
    )
print(
    "elapsed:",
    format_duration(elapsed_seconds()),
)
print(
    "delivery checkpoint:",
    OUTPUT_CHECKPOINT_PATH,
)
print(
    "delivery submission:",
    OUTPUT_SUBMISSION_PATH,
)


## 5. Validation 전체 비교 진단


In [ ]:
# ============================================================
# V3.5 Validation diagnostics — compare every completed member
# ============================================================

if TRAIN_MODEL:
    completed_count = len(completed_state_dicts)
    assert completed_count >= 2

    member_prediction_stack = np.stack(
        completed_predictions,
        axis=0,
    ).astype(np.float64)
    validation_target = np.asarray(
        val_targets[val_index],
        dtype=np.float64,
    )
    assert member_prediction_stack.shape == (
        completed_count,
        EXPECTED_COUNTS["validation"],
        TARGET_HORIZONS,
    )

    # Delivery checkpoint stores only selected models.
    full_weight_vector = np.zeros(
        completed_count, dtype=np.float64
    )
    for original_index, weight in zip(
        active_checkpoint["original_member_indices"],
        active_checkpoint["ensemble_weights"],
    ):
        full_weight_vector[int(original_index)] = float(weight)
    assert np.isclose(full_weight_vector.sum(), 1.0)

    selected_prediction = weighted_prediction(
        member_prediction_stack,
        full_weight_vector,
    )
    selected_rmse = overall_rmse(
        validation_target,
        selected_prediction,
    )
    assert np.isclose(
        selected_rmse,
        float(active_checkpoint["validation_rmse_km_s"]),
        rtol=0.0,
        atol=1e-5,
    )

    # --------------------------------------------------------
    # Per-member overall
    # --------------------------------------------------------
    summary_rows = []
    for member_index in range(completed_count):
        summary_rows.append(
            {
                "member": member_index,
                "priority_rank": member_index + 1,
                "label": MEMBER_LABELS[member_index],
                "variant": ENSEMBLE_VARIANTS[member_index],
                "seed": MEMBER_SEEDS[member_index],
                "epochs_actual": completed_epochs_actual[member_index],
                "val_rmse_km_s": overall_rmse(
                    validation_target,
                    member_prediction_stack[member_index],
                ),
                "ensemble_weight": full_weight_vector[member_index],
                "selected": bool(
                    full_weight_vector[member_index] > 1e-10
                ),
                "parameters": V35_PARAMETER_COUNTS[
                    ENSEMBLE_VARIANTS[member_index]
                ],
            }
        )

    member_summary = pd.DataFrame(summary_rows).sort_values(
        ["val_rmse_km_s", "member"]
    )
    member_summary.to_csv(
        OUTPUT_DIR / "member_validation_summary.csv",
        index=False,
    )

    # --------------------------------------------------------
    # Horizon metrics
    # --------------------------------------------------------
    horizon_rows = []
    for member_index in range(completed_count):
        for horizon_index in range(TARGET_HORIZONS):
            horizon_rows.append(
                {
                    "member": member_index,
                    "label": MEMBER_LABELS[member_index],
                    "horizon_hours": (horizon_index + 1) * 6,
                    "rmse_km_s": overall_rmse(
                        validation_target[:, horizon_index],
                        member_prediction_stack[
                            member_index, :, horizon_index
                        ],
                    ),
                }
            )

    for horizon_index in range(TARGET_HORIZONS):
        horizon_rows.append(
            {
                "member": "ensemble",
                "label": "Selected V3.5 Ensemble",
                "horizon_hours": (horizon_index + 1) * 6,
                "rmse_km_s": overall_rmse(
                    validation_target[:, horizon_index],
                    selected_prediction[:, horizon_index],
                ),
            }
        )

    member_horizon_metrics = pd.DataFrame(horizon_rows)
    member_horizon_metrics.to_csv(
        OUTPUT_DIR / "member_horizon_metrics.csv",
        index=False,
    )

    # --------------------------------------------------------
    # Speed bins
    # --------------------------------------------------------
    speed_bins = (
        ("<350", -np.inf, 350.0),
        ("350-450", 350.0, 450.0),
        ("450-550", 450.0, 550.0),
        (">=550", 550.0, np.inf),
    )

    speed_rows = []
    for label, low, high in speed_bins:
        mask = (
            (validation_target >= low)
            & (validation_target < high)
        )
        count = int(mask.sum())
        if count == 0:
            continue

        for member_index in range(completed_count):
            rmse = float(
                np.sqrt(
                    np.mean(
                        (
                            member_prediction_stack[member_index][mask]
                            - validation_target[mask]
                        )
                        ** 2
                    )
                )
            )
            speed_rows.append(
                {
                    "speed_bin": label,
                    "member": member_index,
                    "label": MEMBER_LABELS[member_index],
                    "count": count,
                    "rmse_km_s": rmse,
                }
            )

        speed_rows.append(
            {
                "speed_bin": label,
                "member": "ensemble",
                "label": "Selected V3.5 Ensemble",
                "count": count,
                "rmse_km_s": float(
                    np.sqrt(
                        np.mean(
                            (
                                selected_prediction[mask]
                                - validation_target[mask]
                            )
                            ** 2
                        )
                    )
                ),
            }
        )

    member_speed_metrics = pd.DataFrame(speed_rows)
    member_speed_metrics.to_csv(
        OUTPUT_DIR / "member_speed_bin_metrics.csv",
        index=False,
    )

    # --------------------------------------------------------
    # Domain-regime metrics
    # --------------------------------------------------------
    persistence_km_s = (
        val_inputs.iloc[val_index][WIND_COLUMNS[-1]]
        .to_numpy(np.float64)
        .reshape(-1, 1)
    )
    persistence_grid = np.broadcast_to(
        persistence_km_s,
        validation_target.shape,
    )
    change = validation_target - persistence_grid

    regime_masks = {
        "slow_target_<450": validation_target < 450.0,
        "hss_target_>=500": validation_target >= 500.0,
        "onset_rise_>=80": change >= 80.0,
        "recovery_drop_>=80_from_high": (
            (change <= -80.0)
            & (persistence_grid >= 500.0)
        ),
        "large_change_abs_>=100": np.abs(change) >= 100.0,
        "near_continuity_abs_<40": np.abs(change) < 40.0,
        "short_horizon_6_18h": np.broadcast_to(
            np.arange(TARGET_HORIZONS).reshape(1, -1) < 3,
            validation_target.shape,
        ),
        "long_horizon_48_72h": np.broadcast_to(
            np.arange(TARGET_HORIZONS).reshape(1, -1) >= 7,
            validation_target.shape,
        ),
    }

    regime_rows = []
    for regime_name, mask in regime_masks.items():
        count = int(mask.sum())
        if count == 0:
            continue
        for member_index in range(completed_count):
            regime_rows.append(
                {
                    "regime": regime_name,
                    "member": member_index,
                    "label": MEMBER_LABELS[member_index],
                    "count": count,
                    "rmse_km_s": float(
                        np.sqrt(
                            np.mean(
                                (
                                    member_prediction_stack[
                                        member_index
                                    ][mask]
                                    - validation_target[mask]
                                )
                                ** 2
                            )
                        )
                    ),
                }
            )
        regime_rows.append(
            {
                "regime": regime_name,
                "member": "ensemble",
                "label": "Selected V3.5 Ensemble",
                "count": count,
                "rmse_km_s": float(
                    np.sqrt(
                        np.mean(
                            (
                                selected_prediction[mask]
                                - validation_target[mask]
                            )
                            ** 2
                        )
                    )
                ),
            }
        )

    member_regime_metrics = pd.DataFrame(regime_rows)
    member_regime_metrics.to_csv(
        OUTPUT_DIR / "member_regime_metrics.csv",
        index=False,
    )

    # --------------------------------------------------------
    # Residual correlation — diversity diagnostic
    # --------------------------------------------------------
    residuals = (
        member_prediction_stack
        - validation_target[None, :, :]
    ).reshape(completed_count, -1)
    residual_correlation = np.corrcoef(residuals)
    residual_correlation_frame = pd.DataFrame(
        residual_correlation,
        index=[
            f"m{i}_{MEMBER_LABELS[i]}"
            for i in range(completed_count)
        ],
        columns=[
            f"m{i}_{MEMBER_LABELS[i]}"
            for i in range(completed_count)
        ],
    )
    residual_correlation_frame.to_csv(
        OUTPUT_DIR / "member_residual_correlation.csv"
    )

    # --------------------------------------------------------
    # Final overall validation table
    # --------------------------------------------------------
    persistence_prediction = np.broadcast_to(
        persistence_km_s,
        validation_target.shape,
    )
    persistence_rmse = overall_rmse(
        validation_target,
        persistence_prediction,
    )

    validation_metrics = pd.DataFrame(
        [
            {
                "model": "Persistence",
                "rmse_km_s": persistence_rmse,
            },
            {
                "model": "V8 reference",
                "rmse_km_s": BASELINE_V8_VALIDATION_RMSE,
            },
            {
                "model": "V3.5 selected ensemble",
                "rmse_km_s": selected_rmse,
            },
        ]
    )
    validation_metrics.to_csv(
        OUTPUT_DIR / "validation_metrics.csv",
        index=False,
    )

    # --------------------------------------------------------
    # Learning curve
    # --------------------------------------------------------
    history_frame = pd.DataFrame(history_rows)
    history_frame.to_csv(
        OUTPUT_DIR / "history.csv",
        index=False,
    )
    if not history_frame.empty:
        plt.figure(figsize=(13, 7))
        for member_index in range(completed_count):
            subset = history_frame[
                history_frame.member == member_index
            ]
            if subset.empty:
                continue
            plt.plot(
                subset.epoch,
                subset.val_rmse_km_s,
                label=f"m{member_index}:{MEMBER_LABELS[member_index]}",
                alpha=0.8,
            )
        plt.xlabel("epoch")
        plt.ylabel("Validation RMSE (km/s)")
        plt.legend(fontsize=6, ncol=2)
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(
            OUTPUT_DIR / "learning_curve.png",
            dpi=140,
        )
        plt.close()

    # --------------------------------------------------------
    # Console report
    # --------------------------------------------------------
    print()
    print("=" * 90)
    print("V3.5 FINAL VALIDATION REPORT")
    print("=" * 90)
    print(
        "completed:",
        completed_count,
        "/",
        ENSEMBLE_SIZE,
        "| elapsed:",
        format_duration(elapsed_seconds()),
    )
    print(
        "strategy:",
        active_checkpoint["ensemble_strategy"],
    )
    print(
        "selected original members:",
        active_checkpoint["original_member_indices"],
    )
    print(
        "selected labels:",
        active_checkpoint["labels"],
    )
    print(
        "weights:",
        np.round(
            active_checkpoint["ensemble_weights"], 4
        ).tolist(),
    )
    print(
        "V3.5 Validation RMSE:",
        f"{selected_rmse:.6f}",
    )
    print(
        "V8 reference:",
        f"{BASELINE_V8_VALIDATION_RMSE:.6f}",
        "| delta:",
        f"{selected_rmse - BASELINE_V8_VALIDATION_RMSE:+.6f}",
    )
    print(
        "Persistence:",
        f"{persistence_rmse:.6f}",
    )

    print()
    print("PER-MEMBER VALIDATION — sorted")
    print(
        member_summary[
            [
                "member",
                "label",
                "epochs_actual",
                "val_rmse_km_s",
                "ensemble_weight",
                "selected",
            ]
        ].to_string(index=False)
    )

    print()
    print("SELECTED ENSEMBLE HORIZON RMSE")
    ensemble_horizon = member_horizon_metrics[
        member_horizon_metrics["member"] == "ensemble"
    ]
    print(
        ensemble_horizon[
            ["horizon_hours", "rmse_km_s"]
        ].to_string(index=False)
    )

    if final_search_result is not None:
        print()
        print(
            "ensemble candidates:",
            final_search_result["candidate_count"],
            "| robust eligible:",
            final_search_result["eligible_count"],
        )
        top_rows = []
        for rank, row in enumerate(
            final_search_result["rows"][:20],
            start=1,
        ):
            top_rows.append(
                {
                    "rank": rank,
                    "name": row["name"],
                    "rmse": row["val_rmse_km_s"],
                    "half_a": row["half_a_rmse_km_s"],
                    "half_b": row["half_b_rmse_km_s"],
                    "support": row["support"],
                }
            )
        print("TOP 20 ENSEMBLE CANDIDATES")
        print(pd.DataFrame(top_rows).to_string(index=False))

else:
    print(
        "TRAIN_MODEL=False: detailed all-member Validation comparison skipped "
        "(delivery checkpoint contains only selected inference models)."
    )


## 6. Test 추론 / submission 검증


In [ ]:
# ============================================================
# V3.5 Test inference / submission verification
# ============================================================

if TRAIN_MODEL:
    # Training stage already wrote the latest safe delivery pair.
    assert OUTPUT_CHECKPOINT_PATH.is_file()
    assert OUTPUT_SUBMISSION_PATH.is_file()

    active_checkpoint = safe_torch_load(
        OUTPUT_CHECKPOINT_PATH,
        map_location="cpu",
    )
    validate_delivery_checkpoint(active_checkpoint)

    submission = pd.read_csv(
        OUTPUT_SUBMISSION_PATH
    )
    assert submission.shape == (
        EXPECTED_COUNTS["test"], 13
    )
    assert (
        submission.columns.tolist()
        == EXPECTED_SUBMISSION_COLUMNS
    )
    assert submission.sample_id.is_unique
    assert (
        submission.sample_id.tolist()
        == test_ids.sample_id.tolist()
    )
    assert np.isfinite(
        submission[TARGET_COLUMNS].to_numpy(
            np.float64
        )
    ).all()

    print(
        "TRAIN_MODEL=True: latest delivery pair verified "
        "without redundant Test re-inference."
    )

else:
    active_checkpoint = safe_torch_load(
        ACTIVE_CHECKPOINT_PATH,
        map_location="cpu",
    )
    validate_delivery_checkpoint(active_checkpoint)

    reference_predictions = None
    if (
        VERIFY_EXISTING_OUTPUT
        and ACTIVE_CSV_PATH.is_file()
    ):
        reference_submission = pd.read_csv(
            ACTIVE_CSV_PATH
        )
        if (
            reference_submission.shape
            == (EXPECTED_COUNTS["test"], 13)
            and reference_submission.columns.tolist()
            == EXPECTED_SUBMISSION_COLUMNS
            and reference_submission.sample_id.tolist()
            == test_ids.sample_id.tolist()
        ):
            reference_predictions = (
                reference_submission[
                    TARGET_COLUMNS
                ].to_numpy(np.float64)
            )

    test_dataset_local, test_loader_local = (
        make_stable_test_loader()
    )
    test_prediction, predicted_rows = (
        predict_delivery_checkpoint(
            active_checkpoint,
            test_loader_local,
        )
    )
    assert np.array_equal(
        predicted_rows, test_index
    )
    assert test_prediction.shape == (
        EXPECTED_COUNTS["test"],
        TARGET_HORIZONS,
    )
    assert np.isfinite(test_prediction).all()

    submission = pd.DataFrame(
        test_prediction,
        columns=TARGET_COLUMNS,
    )
    submission.insert(
        0,
        "sample_id",
        test_ids.sample_id.tolist(),
    )

    assert submission.shape == (
        EXPECTED_COUNTS["test"], 13
    )
    assert (
        submission.columns.tolist()
        == EXPECTED_SUBMISSION_COLUMNS
    )
    assert submission.sample_id.is_unique
    assert (
        submission.sample_id.tolist()
        == test_ids.sample_id.tolist()
    )
    assert np.isfinite(
        submission[TARGET_COLUMNS].to_numpy(
            np.float64
        )
    ).all()

    if reference_predictions is not None:
        reproduced = np.allclose(
            reference_predictions,
            submission[TARGET_COLUMNS].to_numpy(
                np.float64
            ),
            rtol=REPRODUCTION_RTOL,
            atol=REPRODUCTION_ATOL_KM_S,
        )
        max_abs_diff = float(
            np.max(
                np.abs(
                    reference_predictions
                    - submission[
                        TARGET_COLUMNS
                    ].to_numpy(np.float64)
                )
            )
        )
        print(
            "REPRODUCTION CHECK:",
            "PASS" if reproduced else "FAIL",
            "| max_abs_diff=",
            f"{max_abs_diff:.9f} km/s",
        )
        assert reproduced

    atomic_csv_save(
        submission,
        OUTPUT_SUBMISSION_PATH,
    )


    del test_loader_local, test_dataset_local
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

print(
    "V3.5 submission verified:",
    OUTPUT_SUBMISSION_PATH,
)
print(
    "shape:",
    submission.shape,
)
submission.head()


## 7. 최종 규정/파일/시간 감사


In [ ]:
# ============================================================
# V3.5 final audit
# ============================================================

print()
print("=" * 80)
print("V3.5 FINAL AUDIT")
print("=" * 80)

# 1) Delivery pair exists and is internally valid.
assert OUTPUT_CHECKPOINT_PATH.is_file(), OUTPUT_CHECKPOINT_PATH
assert OUTPUT_SUBMISSION_PATH.is_file(), OUTPUT_SUBMISSION_PATH

audit_checkpoint = safe_torch_load(
    OUTPUT_CHECKPOINT_PATH,
    map_location="cpu",
)
validate_delivery_checkpoint(audit_checkpoint)

audit_submission = pd.read_csv(
    OUTPUT_SUBMISSION_PATH
)
assert audit_submission.shape == (
    EXPECTED_COUNTS["test"], 13
)
assert (
    audit_submission.columns.tolist()
    == EXPECTED_SUBMISSION_COLUMNS
)
assert audit_submission.sample_id.is_unique
assert (
    audit_submission.sample_id.tolist()
    == test_ids.sample_id.tolist()
)
assert np.isfinite(
    audit_submission[
        TARGET_COLUMNS
    ].to_numpy(np.float64)
).all()

# 2) Competition-rule metadata.
assert audit_checkpoint["external_data_used"] is False
assert audit_checkpoint["pretrained_weights_used"] is False
assert audit_checkpoint["train_validation_combined"] is False
assert audit_checkpoint["initialization"] == (
    "random_from_scratch_v34_v8_replay_elite"
)

# 3) Selected model size / weights.
selected_indices = [
    int(i)
    for i in audit_checkpoint[
        "original_member_indices"
    ]
]
selected_weights = np.asarray(
    audit_checkpoint["ensemble_weights"],
    dtype=np.float64,
)
assert 2 <= len(selected_indices) <= MAX_SELECTED_MEMBERS
assert selected_weights.shape == (
    len(selected_indices),
)
assert (selected_weights > 0).all()
assert np.isclose(
    selected_weights.sum(),
    1.0,
    rtol=0.0,
    atol=1e-8,
)

# 4) Output file sizes — useful overnight sanity check.
model_mb = (
    OUTPUT_CHECKPOINT_PATH.stat().st_size
    / (1024 ** 2)
)
csv_mb = (
    OUTPUT_SUBMISSION_PATH.stat().st_size
    / (1024 ** 2)
)

# 5) External-output safety audit.
assert OUTPUT_DIR == (Path.home() / TEAM_MEMBER / VERSION)

elapsed = elapsed_seconds()

print("architecture:", ARCHITECTURE_VERSION)
print(
    "completed member count:",
    audit_checkpoint["completed_member_count"],
    "/",
    ENSEMBLE_SIZE,
)
print(
    "selected original members:",
    selected_indices,
)
print(
    "selected labels:",
    audit_checkpoint["labels"],
)
print(
    "selected weights:",
    np.round(
        selected_weights, 4
    ).tolist(),
)
print(
    "Validation RMSE:",
    f"{audit_checkpoint['validation_rmse_km_s']:.6f}",
)
print(
    "ensemble strategy:",
    audit_checkpoint["ensemble_strategy"],
)
print(
    "model.pth size:",
    f"{model_mb:.1f} MB",
)
print(
    "submission.csv size:",
    f"{csv_mb:.2f} MB",
)
print(
    "total elapsed:",
    format_duration(elapsed),
)
print(
    "7-hour target:",
    (
        "PASS"
        if elapsed <= 7 * 3600
        else "WARNING: exceeded target"
    ),
)

print()
print("RULE AUDIT")
print("- official dataset only: PASS")
print("- external data: NOT USED")
print("- pretrained weights: NOT USED")
print("- Train + Validation merge: NOT USED")
print("- random initialization: PASS")
print("- single self-contained notebook design: PASS")
print("- output model.pth/submission.csv reproducible from selected states: PASS")
print("- horizontal flip that reverses solar rotation: DISABLED")

print("- external output only: PASS")
print("- /submission automatic write: DISABLED BY DESIGN")
print("- 아침에 결과 확인 후 필요한 3개 파일만 수동으로 제출 폴더에 복사하세요.")

print()
print("OUTPUT DIRECTORY:", OUTPUT_DIR.resolve())
print("=" * 80)


## V3.5 해석 포인트

- **V8 replay:** 첫 4명은 원래 V8 학습 순서를 그대로 실행하므로 `Physics 778 + Wide 780` champion 재현 여부를 먼저 확인한다.
- **Specialist gain:** Bright-Dark Boundary / Physics LT / CH Topology가 champion을 실제로 낮출 때만 채택한다.
- **안전장치:** specialist가 Validation에서 유의미하게 이득을 못 주면 자동으로 V8 champion pair를 유지한다.
- **비교 파일:** `member_validation_summary.csv`, `member_horizon_metrics.csv`, `member_residual_correlation.csv`, `ensemble_search.csv`.
- **외부 저장:** `outputs/v3.5`.


## CODE 7 — immutable CODE4 baseline gate and zero-cost phases

The 17 preceding cells are an exact copy of `code (4).ipynb`. No CODE7 experiment is allowed before its live baseline passes the reproduction gate.


In [ ]:
# ============================================================
# CODE 7 PHASE 0-5: immutable CODE4 baseline gate + zero-cost search
# The preceding CODE4 cells are copied verbatim and have already finished.
# ============================================================
import shutil

CODE7_STARTED_AT = time.time()
HISTORICAL_CODE4_RMSE = 62.955678
BASELINE_REPRODUCTION_TOLERANCE = 0.05
TRAIN_EXTRA_MODELS_IF_NEEDED = False  # CODE8: prior 66.x models had zero blend gain
EXTRA_MODEL_TRIGGER_RMSE = 62.70
CODE7_MAX_TOTAL_RUNTIME_HOURS = 4.0
CODE7_FINALIZATION_RESERVE_MINUTES = 20

CODE7_FINE_STEP = 0.01
CODE7_FINE_RADIUS = 0.10
CODE7_GROUP_STEP = 0.02
CODE7_GROUP_RADIUS = 0.08
CODE7_GROUP_TOP_K = 5
CODE7_FOLD_COUNT = 5
CODE7_MIN_FOLD_WINS = 3
CODE7_MAX_FOLD_REGRESSION = 0.25
CODE7_GLOBAL_ALPHA_GRID = (0.94, 0.96, 0.98, 1.00, 1.02, 1.04)
CODE7_GROUP_ALPHA_GRID = (0.94, 0.97, 1.00, 1.03, 1.06)
CODE7_GLOBAL_ALPHA_MIN_GAIN = 0.02
CODE7_GROUP_SLICES = ((0, 4), (4, 8), (8, 12))

CODE7_CONTEST_ROOT = DATA_ROOT.resolve().parent.parent
CODE7_SUBMISSION_DIR = CODE7_CONTEST_ROOT / "submission"
CODE7_SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
CODE7_MODEL_PATH = CODE7_SUBMISSION_DIR / "model.pth"
CODE7_SUBMISSION_PATH = CODE7_SUBMISSION_DIR / "submission.csv"
CODE7_REPORT_PATH = OUTPUT_DIR / "code7_search_report.csv"
CODE7_EXTRA_HISTORY_PATH = OUTPUT_DIR / "code7_extra_history.csv"

assert TRAIN_MODEL is True, "CODE 7 must execute the original CODE4 training first."
assert len(completed_state_dicts) == ENSEMBLE_SIZE
assert len(completed_predictions) == ENSEMBLE_SIZE
assert final_search_result is not None
validate_delivery_checkpoint(active_checkpoint)

# Preserve the live CODE4 winner without mutating or replacing any original object.
CODE4_ACTIVE_CHECKPOINT = active_checkpoint
CODE4_OUTPUT_CHECKPOINT_PATH = OUTPUT_CHECKPOINT_PATH
CODE4_OUTPUT_SUBMISSION_PATH = OUTPUT_SUBMISSION_PATH
CODE7_BASE_MEMBER_INDICES = tuple(
    int(index) for index in CODE4_ACTIVE_CHECKPOINT["original_member_indices"]
)
CODE7_BASE_LABELS = tuple(CODE4_ACTIVE_CHECKPOINT["labels"])
CODE7_BASE_VARIANTS = tuple(CODE4_ACTIVE_CHECKPOINT["variants"])
CODE7_BASE_SEEDS = tuple(int(seed) for seed in CODE4_ACTIVE_CHECKPOINT["seeds"])
CODE7_BASE_WEIGHTS = np.asarray(
    CODE4_ACTIVE_CHECKPOINT["ensemble_weights"], dtype=np.float64
)
CODE7_BASE_MEMBER_PREDICTIONS = np.asarray(
    member_prediction_stack[list(CODE7_BASE_MEMBER_INDICES)], dtype=np.float64
).copy()
BASELINE_CHAMPION_PREDICTION = np.asarray(
    selected_prediction, dtype=np.float64
).copy()
CODE7_VALIDATION_TARGET = np.asarray(validation_target, dtype=np.float64)
CODE7_PERSISTENCE_VALIDATION = np.asarray(
    persistence_prediction, dtype=np.float64
)

assert CODE7_BASE_MEMBER_PREDICTIONS.shape[1:] == (1199, 12)
assert CODE7_BASE_WEIGHTS.shape == (len(CODE7_BASE_MEMBER_INDICES),)
assert np.isclose(CODE7_BASE_WEIGHTS.sum(), 1.0, atol=1e-10)
assert np.all(CODE7_BASE_WEIGHTS > 0)
assert np.allclose(
    weighted_prediction(CODE7_BASE_MEMBER_PREDICTIONS, CODE7_BASE_WEIGHTS),
    BASELINE_CHAMPION_PREDICTION,
    rtol=0.0,
    atol=1e-8,
)

LIVE_CODE4_BASELINE = overall_rmse(
    CODE7_VALIDATION_TARGET, BASELINE_CHAMPION_PREDICTION
)
CODE7_BASELINE_DIFFERENCE = LIVE_CODE4_BASELINE - HISTORICAL_CODE4_RMSE
BASELINE_REPRODUCTION_OK = (
    abs(CODE7_BASELINE_DIFFERENCE) <= BASELINE_REPRODUCTION_TOLERANCE
)

print("\n" + "=" * 88)
print("CODE 7 BASELINE REPRODUCTION GATE")
print("Historical code (4):", f"{HISTORICAL_CODE4_RMSE:.6f}")
print("Live code (4):      ", f"{LIVE_CODE4_BASELINE:.6f}")
print("Difference:         ", f"{CODE7_BASELINE_DIFFERENCE:+.6f}")
print("Actual members:     ", list(CODE7_BASE_MEMBER_INDICES))
print("Actual labels:      ", list(CODE7_BASE_LABELS))
print("Actual weights:     ", np.round(CODE7_BASE_WEIGHTS, 6).tolist())
print("Gate:", "PASS" if BASELINE_REPRODUCTION_OK else "BASELINE REPRODUCTION FAILED")
print("=" * 88)

# sample_id-based deterministic folds; never used for gradient updates.
CODE7_VALIDATION_IDS = val_inputs.iloc[val_index]["sample_id"].reset_index(drop=True)
CODE7_FOLD_IDS = (
    pd.util.hash_pandas_object(CODE7_VALIDATION_IDS, index=False)
    .to_numpy(np.uint64)
    % np.uint64(CODE7_FOLD_COUNT)
).astype(np.int64)
assert set(CODE7_FOLD_IDS.tolist()) == set(range(CODE7_FOLD_COUNT))


def code7_fold_rmse(prediction):
    prediction = np.asarray(prediction, dtype=np.float64)
    return np.asarray([
        overall_rmse(
            CODE7_VALIDATION_TARGET[CODE7_FOLD_IDS == fold],
            prediction[CODE7_FOLD_IDS == fold],
        )
        for fold in range(CODE7_FOLD_COUNT)
    ], dtype=np.float64)


CODE7_BASELINE_FOLD_RMSE = code7_fold_rmse(BASELINE_CHAMPION_PREDICTION)


def code7_strategy_prediction(group_weights, group_alphas):
    group_weights = np.asarray(group_weights, dtype=np.float64)
    group_alphas = np.asarray(group_alphas, dtype=np.float64)
    assert group_weights.shape == (3, len(CODE7_BASE_MEMBER_INDICES))
    assert group_alphas.shape == (3,)
    assert np.all(group_weights >= 0)
    assert np.allclose(group_weights.sum(axis=1), 1.0, atol=1e-9)
    result = np.empty_like(CODE7_VALIDATION_TARGET, dtype=np.float64)
    for group_index, (start, stop) in enumerate(CODE7_GROUP_SLICES):
        raw = np.tensordot(
            group_weights[group_index],
            CODE7_BASE_MEMBER_PREDICTIONS[:, :, start:stop],
            axes=(0, 0),
        )
        baseline = CODE7_PERSISTENCE_VALIDATION[:, start:stop]
        result[:, start:stop] = baseline + group_alphas[group_index] * (
            raw - baseline
        )
    return result


def code7_candidate(name, prediction, group_weights, group_alphas):
    prediction = np.asarray(prediction, dtype=np.float64)
    fold_values = code7_fold_rmse(prediction)
    rmse = overall_rmse(CODE7_VALIDATION_TARGET, prediction)
    fold_delta = fold_values - CODE7_BASELINE_FOLD_RMSE
    fold_wins = int(np.sum(fold_delta < 0.0))
    robust = bool(
        rmse < LIVE_CODE4_BASELINE - 1e-10
        and fold_wins >= CODE7_MIN_FOLD_WINS
        and float(fold_delta.max()) <= CODE7_MAX_FOLD_REGRESSION
    )
    return {
        "name": name,
        "prediction": prediction,
        "rmse": float(rmse),
        "fold_rmse": fold_values,
        "fold_wins": fold_wins,
        "max_fold_regression": float(fold_delta.max()),
        "robust": robust,
        "group_weights": np.asarray(group_weights, dtype=np.float64).copy(),
        "group_alphas": np.asarray(group_alphas, dtype=np.float64).copy(),
        "extra_weights": np.zeros(0, dtype=np.float64),
        "base_component_weight": 1.0,
    }


def code7_local_simplex(center, step, radius):
    center = np.asarray(center, dtype=np.float64)
    support = len(center)
    axes = []
    for value in center[:-1]:
        low = max(0.0, value - radius)
        high = min(1.0, value + radius)
        axes.append(np.arange(low, high + step * 0.25, step))
    seen = set()
    for prefix in itertools.product(*axes):
        last = 1.0 - float(np.sum(prefix))
        vector = np.asarray((*prefix, last), dtype=np.float64)
        if last < -1e-10 or last > 1.0 + 1e-10:
            continue
        if abs(last - center[-1]) > radius + 1e-9:
            continue
        vector = np.clip(vector, 0.0, 1.0)
        if not np.isclose(vector.sum(), 1.0, atol=1e-9):
            continue
        key = tuple(np.round(vector, 10))
        if key not in seen:
            seen.add(key)
            yield vector
    center_key = tuple(np.round(center, 10))
    if center_key not in seen:
        yield center.copy()


def code7_select_best_robust(candidates, fallback):
    eligible = [candidate for candidate in candidates if candidate["robust"]]
    if not eligible:
        return fallback
    return min(eligible, key=lambda candidate: (candidate["rmse"], candidate["name"]))


CODE7_BASE_GROUP_WEIGHTS = np.repeat(CODE7_BASE_WEIGHTS[None, :], 3, axis=0)
CODE7_BASE_GROUP_ALPHAS = np.ones(3, dtype=np.float64)
CODE7_BASELINE_CANDIDATE = code7_candidate(
    "live_code4_champion",
    BASELINE_CHAMPION_PREDICTION,
    CODE7_BASE_GROUP_WEIGHTS,
    CODE7_BASE_GROUP_ALPHAS,
)
CODE7_ZERO_COST_BEST = CODE7_BASELINE_CANDIDATE
CODE7_PHASE_RESULTS = {
    "fine_global": CODE7_BASELINE_CANDIDATE,
    "group_weights": CODE7_BASELINE_CANDIDATE,
    "global_alpha": CODE7_BASELINE_CANDIDATE,
    "group_alpha": CODE7_BASELINE_CANDIDATE,
}
CODE7_SEARCH_ROWS = []


def code7_record_phase(phase, candidate):
    CODE7_SEARCH_ROWS.append({
        "phase": phase,
        "name": candidate["name"],
        "rmse_km_s": candidate["rmse"],
        "fold_wins": candidate["fold_wins"],
        "max_fold_regression": candidate["max_fold_regression"],
        "robust": candidate["robust"],
        "group_weights": json.dumps(np.round(candidate["group_weights"], 6).tolist()),
        "group_alphas": json.dumps(np.round(candidate["group_alphas"], 6).tolist()),
    })


if BASELINE_REPRODUCTION_OK:
    # PHASE 2: 0.01 local fine search around the actual CODE4 champion support.
    fine_candidates = []
    for weights in code7_local_simplex(
        CODE7_BASE_WEIGHTS, CODE7_FINE_STEP, CODE7_FINE_RADIUS
    ):
        group_weights = np.repeat(weights[None, :], 3, axis=0)
        prediction = code7_strategy_prediction(group_weights, np.ones(3))
        fine_candidates.append(code7_candidate(
            "fine_global_" + "_".join(f"{value:.2f}" for value in weights),
            prediction,
            group_weights,
            np.ones(3),
        ))
    fine_best = code7_select_best_robust(
        fine_candidates, CODE7_BASELINE_CANDIDATE
    )
    CODE7_PHASE_RESULTS["fine_global"] = fine_best
    CODE7_ZERO_COST_BEST = min(
        (CODE7_ZERO_COST_BEST, fine_best), key=lambda candidate: candidate["rmse"]
    )
    code7_record_phase("fine_global", fine_best)

    # PHASE 3: group candidates stay close to the original global champion.
    group_options = []
    for group_index, (start, stop) in enumerate(CODE7_GROUP_SLICES):
        ranked = []
        for weights in code7_local_simplex(
            CODE7_BASE_WEIGHTS, CODE7_GROUP_STEP, CODE7_GROUP_RADIUS
        ):
            prediction = np.tensordot(
                weights,
                CODE7_BASE_MEMBER_PREDICTIONS[:, :, start:stop],
                axes=(0, 0),
            )
            score = overall_rmse(
                CODE7_VALIDATION_TARGET[:, start:stop], prediction
            )
            ranked.append((score, tuple(np.round(weights, 10)), weights.copy()))
        ranked.sort(key=lambda item: (item[0], item[1]))
        group_options.append([item[2] for item in ranked[:CODE7_GROUP_TOP_K]])

    group_candidates = []
    for short_weights, middle_weights, long_weights in itertools.product(*group_options):
        group_weights = np.stack(
            [short_weights, middle_weights, long_weights], axis=0
        )
        prediction = code7_strategy_prediction(group_weights, np.ones(3))
        group_candidates.append(code7_candidate(
            "three_group_weights",
            prediction,
            group_weights,
            np.ones(3),
        ))
    group_best = code7_select_best_robust(
        group_candidates, CODE7_ZERO_COST_BEST
    )
    CODE7_PHASE_RESULTS["group_weights"] = group_best
    CODE7_ZERO_COST_BEST = min(
        (CODE7_ZERO_COST_BEST, group_best), key=lambda candidate: candidate["rmse"]
    )
    code7_record_phase("group_weights", group_best)

    # PHASE 4a: one predefined global residual alpha.
    alpha_candidates = []
    alpha_base = CODE7_ZERO_COST_BEST
    for alpha in CODE7_GLOBAL_ALPHA_GRID:
        alphas = np.repeat(float(alpha), 3)
        prediction = code7_strategy_prediction(alpha_base["group_weights"], alphas)
        alpha_candidates.append(code7_candidate(
            f"global_alpha_{alpha:.2f}",
            prediction,
            alpha_base["group_weights"],
            alphas,
        ))
    global_alpha_best = code7_select_best_robust(
        alpha_candidates, CODE7_ZERO_COST_BEST
    )
    CODE7_PHASE_RESULTS["global_alpha"] = global_alpha_best
    global_alpha_gain = CODE7_ZERO_COST_BEST["rmse"] - global_alpha_best["rmse"]
    CODE7_ZERO_COST_BEST = min(
        (CODE7_ZERO_COST_BEST, global_alpha_best),
        key=lambda candidate: candidate["rmse"],
    )
    code7_record_phase("global_alpha", global_alpha_best)

    # PHASE 4b: only if global shrinkage had a meaningful robust gain.
    if global_alpha_gain >= CODE7_GLOBAL_ALPHA_MIN_GAIN:
        group_alpha_candidates = []
        group_alpha_base = CODE7_ZERO_COST_BEST
        for alphas in itertools.product(CODE7_GROUP_ALPHA_GRID, repeat=3):
            alphas = np.asarray(alphas, dtype=np.float64)
            prediction = code7_strategy_prediction(
                group_alpha_base["group_weights"], alphas
            )
            group_alpha_candidates.append(code7_candidate(
                "three_group_alpha_" + "_".join(f"{x:.2f}" for x in alphas),
                prediction,
                group_alpha_base["group_weights"],
                alphas,
            ))
        group_alpha_best = code7_select_best_robust(
            group_alpha_candidates, CODE7_ZERO_COST_BEST
        )
        CODE7_PHASE_RESULTS["group_alpha"] = group_alpha_best
        CODE7_ZERO_COST_BEST = min(
            (CODE7_ZERO_COST_BEST, group_alpha_best),
            key=lambda candidate: candidate["rmse"],
        )
        code7_record_phase("group_alpha", group_alpha_best)
    else:
        code7_record_phase("group_alpha_not_triggered", CODE7_ZERO_COST_BEST)

    # PHASE 5: CODE4 keeps only the epoch-wise winning raw/EMA state.
    CODE7_RAW_EMA_STATUS = (
        "UNAVAILABLE: immutable CODE4 stores only the winning raw/EMA state; "
        "the original training loop was not modified."
    )
    pd.DataFrame(CODE7_SEARCH_ROWS).to_csv(CODE7_REPORT_PATH, index=False)
    print("\nZERO-COST SEARCH")
    for phase_name, result in CODE7_PHASE_RESULTS.items():
        print(
            phase_name,
            f"RMSE={result['rmse']:.6f}",
            f"fold_wins={result['fold_wins']}/5",
            f"robust={result['robust']}",
        )
    print("Raw/EMA diversity:", CODE7_RAW_EMA_STATUS)
else:
    CODE7_RAW_EMA_STATUS = "SKIPPED: BASELINE REPRODUCTION FAILED"
    print("BASELINE REPRODUCTION FAILED — all additional searches/training are disabled.")


## CODE 7 — optional complementary models

These models run only after CODE4 and only when zero-cost search does not reach the trigger.


In [ ]:
# ============================================================
# CODE 7 PHASE 6-7: at most two complementary models, after CODE4
# ============================================================
CODE7_EXTRA_MODEL_SEED = 1781
CODE7_TRAJECTORY_LAMBDA = 0.05
CODE7_EXTRA_MODEL_WEIGHT_GRID = (0.05, 0.10, 0.15, 0.20, 0.25)
CODE7_EXTRA_TOTAL_WEIGHT_LIMIT = 0.30
CODE7_EXTRA_FINE_STEP = 0.025
CODE7_EXTRA_FINE_RADIUS = 0.05
CODE7_EXTRA_HISTORY_ROWS = []
CODE7_EXTRA_RESULTS = []
CODE7_EXTRA_STATUS = {
    "temporal_difference": "NOT RUN",
    "trajectory_auxiliary": "NOT RUN",
}


class TemporalDifferenceInputStem(nn.Module):
    """Option B: raw 193/211 + their temporal differences at the first stem only."""
    def __init__(self, original_stem):
        super().__init__()
        original_conv = original_stem[0]
        assert isinstance(original_conv, nn.Conv3d)
        assert original_conv.in_channels == 2 and original_conv.out_channels == 32
        self.conv = nn.Conv3d(
            4,
            original_conv.out_channels,
            kernel_size=original_conv.kernel_size,
            stride=original_conv.stride,
            padding=original_conv.padding,
            dilation=original_conv.dilation,
            groups=original_conv.groups,
            bias=original_conv.bias is not None,
        )
        self.norm = original_stem[1]
        self.activation = original_stem[2]
        self.pool = original_stem[3]

    def forward(self, image_channels_first):
        assert image_channels_first.ndim == 5
        assert image_channels_first.shape[1] == 2
        first_delta = torch.zeros_like(image_channels_first[:, :, :1])
        later_delta = (
            image_channels_first[:, :, 1:]
            - image_channels_first[:, :, :-1]
        )
        temporal_delta = torch.cat([first_delta, later_delta], dim=2)
        four_channel_input = torch.cat(
            [image_channels_first, temporal_delta], dim=1
        )
        assert four_channel_input.shape[1] == 4
        return self.pool(self.activation(self.norm(self.conv(four_channel_input))))


class TemporalDifferenceSolarWind(SolarWindV35):
    def __init__(self):
        super().__init__(
            wind_feature_dim=WIND_FEATURE_DIM,
            dropout=DROPOUT,
            conv_dropout=CONV_DROPOUT,
            variant="physics",
            horizon_embed_dim=HORIZON_EMBED_DIM,
            image_attention_dim=IMAGE_ATTENTION_DIM,
            wind_gru_hidden=WIND_GRU_HIDDEN,
            wind_attention_dim=WIND_ATTENTION_DIM,
            horizon_decoder_hidden=HORIZON_DECODER_HIDDEN,
        )
        self.stem = TemporalDifferenceInputStem(self.stem)


def code7_build_model(model_kind, device=DEVICE):
    if model_kind == "temporal_difference":
        return TemporalDifferenceSolarWind().to(device)
    if model_kind == "trajectory_auxiliary":
        return build_v35_model("physics", device=device)
    raise ValueError(f"Unknown CODE7 model kind: {model_kind}")


def code7_reset_extra_randomness(seed):
    # This is called only after immutable CODE4 has completely finished.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    TRAIN_LOADER_GENERATOR.manual_seed(seed)


def code7_train_trajectory_epoch(model, ema, optimizer, scaler):
    model.train()
    squared_error_sum = 0.0
    value_count = 0
    alpha = mixup_alpha_for_variant("physics")
    for batch in train_loader:
        images = batch["images"].to(DEVICE, non_blocking=PIN_MEMORY)
        wind = batch["wind"].to(DEVICE, non_blocking=PIN_MEMORY)
        coronal = batch["coronal"].to(DEVICE, non_blocking=PIN_MEMORY)
        persistence = batch["persistence"].to(DEVICE, non_blocking=PIN_MEMORY)
        target = batch["target"].to(DEVICE, non_blocking=PIN_MEMORY)
        images, coronal = augment_batch(images, coronal)
        if alpha > 0:
            lam = float(np.random.beta(alpha, alpha))
            permutation = torch.randperm(images.shape[0], device=images.device)
            images = lam * images + (1.0 - lam) * images[permutation]
            wind = lam * wind + (1.0 - lam) * wind[permutation]
            coronal = lam * coronal + (1.0 - lam) * coronal[permutation]
            persistence = lam * persistence + (1.0 - lam) * persistence[permutation]
            target = lam * target + (1.0 - lam) * target[permutation]
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):
            prediction = model(images, wind, persistence, coronal)
            loss_main = F.mse_loss(prediction, target)
            loss_delta = F.mse_loss(
                prediction[:, 1:] - prediction[:, :-1],
                target[:, 1:] - target[:, :-1],
            )
            loss = loss_main + CODE7_TRAJECTORY_LAMBDA * loss_delta
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()
        ema.update(model)
        error_km_s = (prediction.detach().float() - target) * 1000.0
        squared_error_sum += float(torch.sum(error_km_s ** 2).cpu())
        value_count += error_km_s.numel()
    return math.sqrt(squared_error_sum / value_count)


def code7_estimated_extra_seconds():
    base_seconds = [
        float(row["seconds"])
        for row in history_rows
        if row.get("variant") in {"physics", "wide"}
        and float(row.get("seconds", 0.0)) > 0.0
    ]
    per_epoch = float(np.median(base_seconds[-40:])) if base_seconds else 30.0
    return 40.0 * per_epoch * 1.20


def code7_extra_runtime_available():
    finish_deadline = RUN_STARTED_AT + CODE7_MAX_TOTAL_RUNTIME_HOURS * 3600.0
    reserve = CODE7_FINALIZATION_RESERVE_MINUTES * 60.0
    return time.time() + code7_estimated_extra_seconds() + reserve <= finish_deadline


def code7_train_extra_model(model_kind):
    planned_epochs = 40
    seed = CODE7_EXTRA_MODEL_SEED
    code7_reset_extra_randomness(seed)
    model = code7_build_model(model_kind)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=3e-4, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, make_lr_lambda(planned_epochs)
    )
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)
    ema = WeightEMA(model, EMA_DECAY)
    best_rmse = float("inf")
    best_state = None
    best_source = None
    best_epoch = 0
    started = time.perf_counter()
    for epoch in range(1, planned_epochs + 1):
        epoch_started = time.perf_counter()
        if model_kind == "trajectory_auxiliary":
            train_rmse = code7_train_trajectory_epoch(
                model, ema, optimizer, scaler
            )
        else:
            train_rmse = train_one_epoch(
                model, ema, optimizer, scaler, "physics"
            )
        raw_rmse = evaluate_rmse(model)
        ema_rmse = evaluate_rmse(ema.module)
        scheduler.step()
        if ema_rmse <= raw_rmse:
            val_rmse, source_model, source_name = ema_rmse, ema.module, "ema"
        else:
            val_rmse, source_model, source_name = raw_rmse, model, "raw"
        if val_rmse < best_rmse:
            best_rmse = float(val_rmse)
            best_source = source_name
            best_epoch = epoch
            best_state = {
                key: value.detach().clone().cpu()
                for key, value in source_model.state_dict().items()
            }
        elapsed = time.perf_counter() - epoch_started
        CODE7_EXTRA_HISTORY_ROWS.append({
            "model_kind": model_kind,
            "seed": seed,
            "epoch": epoch,
            "train_rmse_km_s": train_rmse,
            "raw_val_rmse_km_s": raw_rmse,
            "ema_val_rmse_km_s": ema_rmse,
            "selected_source": source_name,
            "learning_rate": optimizer.param_groups[0]["lr"],
            "seconds": elapsed,
        })
        print(
            f"CODE7 {model_kind} epoch={epoch:02d}/40 "
            f"train={train_rmse:.3f} raw={raw_rmse:.3f} "
            f"ema={ema_rmse:.3f} best={best_rmse:.3f}({best_source})"
        )
    assert best_state is not None and 1 <= best_epoch <= 40
    model.load_state_dict(best_state, strict=True)
    prediction, rows = predict_single_model(model, val_loader)
    assert np.array_equal(rows, val_index)
    direct_rmse = overall_rmse(CODE7_VALIDATION_TARGET, prediction)
    assert np.isclose(direct_rmse, best_rmse, rtol=1e-5, atol=1e-2)
    result = {
        "kind": model_kind,
        "variant": "physics",
        "seed": seed,
        "state_dict": best_state,
        "best_source": best_source,
        "best_epoch": best_epoch,
        "prediction": np.asarray(prediction, dtype=np.float64),
        "standalone_rmse": float(direct_rmse),
        "parameters": int(sum(parameter.numel() for parameter in model.parameters())),
        "training_seconds": float(time.perf_counter() - started),
    }
    del model, ema, optimizer, scheduler, scaler
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return result


CODE7_EXTRA_TRIGGERED = bool(
    BASELINE_REPRODUCTION_OK
    and TRAIN_EXTRA_MODELS_IF_NEEDED
    and CODE7_ZERO_COST_BEST["rmse"] > EXTRA_MODEL_TRIGGER_RMSE
)

if CODE7_EXTRA_TRIGGERED:
    for model_kind in ("temporal_difference", "trajectory_auxiliary"):
        if not code7_extra_runtime_available():
            CODE7_EXTRA_STATUS[model_kind] = "SKIPPED: runtime reserve protected"
            print(CODE7_EXTRA_STATUS[model_kind])
            continue
        print("\nTRAIN EXTRA MODEL:", model_kind)
        extra_result = code7_train_extra_model(model_kind)
        champion_residual = (
            BASELINE_CHAMPION_PREDICTION - CODE7_VALIDATION_TARGET
        ).reshape(-1)
        extra_residual = (
            extra_result["prediction"] - CODE7_VALIDATION_TARGET
        ).reshape(-1)
        extra_result["residual_correlation"] = float(
            np.corrcoef(champion_residual, extra_residual)[0, 1]
        )
        CODE7_EXTRA_RESULTS.append(extra_result)
        CODE7_EXTRA_STATUS[model_kind] = (
            f"DONE RMSE={extra_result['standalone_rmse']:.6f} "
            f"corr={extra_result['residual_correlation']:.6f}"
        )
        print(CODE7_EXTRA_STATUS[model_kind])
else:
    reason = (
        "disabled in CODE8 after 66.x standalone / zero blend gain"
        if not TRAIN_EXTRA_MODELS_IF_NEEDED
        else (
            "baseline gate failed"
            if not BASELINE_REPRODUCTION_OK
            else f"zero-cost RMSE <= {EXTRA_MODEL_TRIGGER_RMSE:.2f}"
        )
    )
    for model_kind in CODE7_EXTRA_STATUS:
        CODE7_EXTRA_STATUS[model_kind] = "SKIPPED: " + reason
    print("Extra models skipped:", reason)

if CODE7_EXTRA_HISTORY_ROWS:
    pd.DataFrame(CODE7_EXTRA_HISTORY_ROWS).to_csv(
        CODE7_EXTRA_HISTORY_PATH, index=False
    )

# PHASE 8: small constrained mass for complementary models only.
CODE7_FINAL_CANDIDATE = CODE7_ZERO_COST_BEST
if BASELINE_REPRODUCTION_OK and CODE7_EXTRA_RESULTS:
    weight_axes = [
        (0.0,) + CODE7_EXTRA_MODEL_WEIGHT_GRID
        for _ in CODE7_EXTRA_RESULTS
    ]
    extra_candidates = []
    for extra_weights in itertools.product(*weight_axes):
        if not any(weight > 0 for weight in extra_weights):
            continue
        total_extra = float(sum(extra_weights))
        if total_extra > CODE7_EXTRA_TOTAL_WEIGHT_LIMIT + 1e-10:
            continue
        prediction = (1.0 - total_extra) * CODE7_ZERO_COST_BEST["prediction"]
        for weight, result in zip(extra_weights, CODE7_EXTRA_RESULTS):
            prediction = prediction + float(weight) * result["prediction"]
        candidate = code7_candidate(
            "extra_blend_" + "_".join(f"{weight:.2f}" for weight in extra_weights),
            prediction,
            CODE7_ZERO_COST_BEST["group_weights"],
            CODE7_ZERO_COST_BEST["group_alphas"],
        )
        candidate["extra_weights"] = np.asarray(extra_weights, dtype=np.float64)
        candidate["base_component_weight"] = 1.0 - total_extra
        extra_candidates.append(candidate)
    coarse_extra_best = code7_select_best_robust(
        extra_candidates, CODE7_ZERO_COST_BEST
    )
    CODE7_FINAL_CANDIDATE = coarse_extra_best

    # Only a genuinely improved coarse blend unlocks a very small local grid.
    if (
        coarse_extra_best["robust"]
        and coarse_extra_best["rmse"] < CODE7_ZERO_COST_BEST["rmse"]
    ):
        fine_axes = []
        for center_weight in coarse_extra_best["extra_weights"]:
            low = max(0.0, float(center_weight) - CODE7_EXTRA_FINE_RADIUS)
            high = min(0.25, float(center_weight) + CODE7_EXTRA_FINE_RADIUS)
            fine_axes.append(np.arange(
                low,
                high + CODE7_EXTRA_FINE_STEP * 0.25,
                CODE7_EXTRA_FINE_STEP,
            ))
        fine_extra_candidates = []
        for extra_weights in itertools.product(*fine_axes):
            total_extra = float(sum(extra_weights))
            if total_extra <= 0.0 or total_extra > CODE7_EXTRA_TOTAL_WEIGHT_LIMIT + 1e-10:
                continue
            prediction = (1.0 - total_extra) * CODE7_ZERO_COST_BEST["prediction"]
            for weight, result in zip(extra_weights, CODE7_EXTRA_RESULTS):
                prediction = prediction + float(weight) * result["prediction"]
            candidate = code7_candidate(
                "extra_local_fine_" + "_".join(
                    f"{weight:.3f}" for weight in extra_weights
                ),
                prediction,
                CODE7_ZERO_COST_BEST["group_weights"],
                CODE7_ZERO_COST_BEST["group_alphas"],
            )
            candidate["extra_weights"] = np.asarray(extra_weights, dtype=np.float64)
            candidate["base_component_weight"] = 1.0 - total_extra
            fine_extra_candidates.append(candidate)
        CODE7_FINAL_CANDIDATE = code7_select_best_robust(
            fine_extra_candidates, coarse_extra_best
        )
        extra_candidates.extend(fine_extra_candidates)

    for result_index, result in enumerate(CODE7_EXTRA_RESULTS):
        one_model = [
            candidate for candidate in extra_candidates
            if candidate["extra_weights"][result_index] > 0
            and np.count_nonzero(candidate["extra_weights"]) == 1
        ]
        result["best_blend_rmse"] = min(
            [candidate["rmse"] for candidate in one_model]
            + [CODE7_ZERO_COST_BEST["rmse"]]
        )

# Never promote a candidate unless it beats both historical and live CODE4.
CODE7_CANDIDATE_BEATS_CODE4 = bool(
    BASELINE_REPRODUCTION_OK
    and CODE7_FINAL_CANDIDATE["robust"]
    and CODE7_FINAL_CANDIDATE["rmse"]
    < min(HISTORICAL_CODE4_RMSE, LIVE_CODE4_BASELINE) - 1e-10
)
if not CODE7_CANDIDATE_BEATS_CODE4:
    CODE7_FINAL_CANDIDATE = CODE7_BASELINE_CANDIDATE


## CODE 7 — frozen Validation decision, Test inference and output

Test is accessed only after every candidate and weight has been frozen on Validation.


In [ ]:
# ============================================================
# CODE 7 PHASE 9-10: frozen Validation decision -> Test -> outputs
# ============================================================
def code7_atomic_copy(source, destination):
    source, destination = Path(source), Path(destination)
    assert source.is_file()
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".tmp")
    shutil.copy2(source, temporary)
    os.replace(temporary, destination)


def code7_validate_checkpoint(checkpoint):
    required = {
        "architecture_version", "base_model_state_dicts", "base_variants",
        "base_labels", "base_seeds", "base_original_member_indices",
        "base_group_weights", "base_group_alphas", "base_component_weight",
        "extra_model_state_dicts", "extra_model_kinds", "extra_variants",
        "extra_seeds", "extra_weights", "validation_rmse_km_s",
        "historical_code4_rmse", "live_code4_rmse", "fold_rmse_km_s",
        "fold_wins", "image_size", "channels", "wind_feature_dim",
        "target_horizons", "coronal_feature_config", "preprocessing",
        "external_data_used", "pretrained_weights_used",
        "train_validation_combined", "test_used_for_selection",
    }
    missing = sorted(required - set(checkpoint))
    assert not missing, missing
    base_count = len(checkpoint["base_model_state_dicts"])
    assert base_count == len(checkpoint["base_variants"])
    assert base_count == len(checkpoint["base_labels"])
    assert base_count == len(checkpoint["base_seeds"])
    group_weights = np.asarray(checkpoint["base_group_weights"], dtype=np.float64)
    assert group_weights.shape == (3, base_count)
    assert np.all(group_weights >= 0)
    assert np.allclose(group_weights.sum(axis=1), 1.0, atol=1e-9)
    group_alphas = np.asarray(checkpoint["base_group_alphas"], dtype=np.float64)
    assert group_alphas.shape == (3,)
    extra_count = len(checkpoint["extra_model_state_dicts"])
    assert extra_count == len(checkpoint["extra_model_kinds"])
    assert extra_count == len(checkpoint["extra_variants"])
    assert extra_count == len(checkpoint["extra_seeds"])
    assert extra_count == len(checkpoint["extra_weights"])
    assert all(
        isinstance(state, dict) and state
        for state in checkpoint["base_model_state_dicts"]
    )
    assert all(
        isinstance(state, dict) and state
        for state in checkpoint["extra_model_state_dicts"]
    )
    assert set(checkpoint["extra_model_kinds"]) <= {
        "temporal_difference", "trajectory_auxiliary"
    }
    extra_weights = np.asarray(checkpoint["extra_weights"], dtype=np.float64)
    assert np.all(extra_weights > 0)
    assert np.isclose(
        float(checkpoint["base_component_weight"]) + extra_weights.sum(),
        1.0,
        atol=1e-9,
    )
    assert checkpoint["image_size"] == IMAGE_SIZE
    assert tuple(checkpoint["channels"]) == CHANNELS
    assert checkpoint["wind_feature_dim"] == WIND_FEATURE_DIM
    assert checkpoint["target_horizons"] == TARGET_HORIZONS
    assert checkpoint["external_data_used"] is False
    assert checkpoint["pretrained_weights_used"] is False
    assert checkpoint["train_validation_combined"] is False
    assert checkpoint["test_used_for_selection"] is False
    return checkpoint


def code7_predict_state(model_kind, variant, state_dict, loader):
    if model_kind == "code4_base":
        model = build_v35_model(variant)
    else:
        model = code7_build_model(model_kind)
    model.load_state_dict(state_dict, strict=True)
    prediction, rows = predict_single_model(model, loader)
    del model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return prediction, rows


if CODE7_CANDIDATE_BEATS_CODE4:
    selected_extra_indices = np.flatnonzero(
        np.asarray(CODE7_FINAL_CANDIDATE["extra_weights"], dtype=np.float64) > 0
    ).tolist()
    code7_checkpoint = {
        "architecture_version": "code7_immutable_code4_plus_constrained_search_v1",
        "delivery_format_version": 1,
        "base_model_state_dicts": list(CODE4_ACTIVE_CHECKPOINT["model_state_dicts"]),
        "base_variants": list(CODE7_BASE_VARIANTS),
        "base_labels": list(CODE7_BASE_LABELS),
        "base_seeds": list(CODE7_BASE_SEEDS),
        "base_original_member_indices": list(CODE7_BASE_MEMBER_INDICES),
        "base_group_weights": CODE7_FINAL_CANDIDATE["group_weights"].tolist(),
        "base_group_alphas": CODE7_FINAL_CANDIDATE["group_alphas"].tolist(),
        "base_component_weight": float(CODE7_FINAL_CANDIDATE["base_component_weight"]),
        "extra_model_state_dicts": [
            CODE7_EXTRA_RESULTS[index]["state_dict"]
            for index in selected_extra_indices
        ],
        "extra_model_kinds": [
            CODE7_EXTRA_RESULTS[index]["kind"]
            for index in selected_extra_indices
        ],
        "extra_variants": [
            CODE7_EXTRA_RESULTS[index]["variant"]
            for index in selected_extra_indices
        ],
        "extra_seeds": [
            CODE7_EXTRA_RESULTS[index]["seed"]
            for index in selected_extra_indices
        ],
        "extra_weights": [
            float(CODE7_FINAL_CANDIDATE["extra_weights"][index])
            for index in selected_extra_indices
        ],
        "validation_rmse_km_s": float(CODE7_FINAL_CANDIDATE["rmse"]),
        "historical_code4_rmse": HISTORICAL_CODE4_RMSE,
        "live_code4_rmse": LIVE_CODE4_BASELINE,
        "fold_rmse_km_s": CODE7_FINAL_CANDIDATE["fold_rmse"].tolist(),
        "fold_wins": int(CODE7_FINAL_CANDIDATE["fold_wins"]),
        "selection_name": CODE7_FINAL_CANDIDATE["name"],
        "image_size": IMAGE_SIZE,
        "channels": list(CHANNELS),
        "wind_feature_dim": WIND_FEATURE_DIM,
        "target_horizons": TARGET_HORIZONS,
        "coronal_feature_config": {
            "bands": CH_BANDS,
            "thresholds": list(CH_THRESHOLDS),
            "equator_fraction": CH_EQUATOR_FRACTION,
            "disk_shrink": DISK_SHRINK,
            "feature_per_step": CH_FEATURE_PER_STEP,
            "geometry": {
                "center_y": float(DISK_CENTER_Y),
                "center_x": float(DISK_CENTER_X),
                "radius": float(DISK_RADIUS),
                "native_shape": list(NATIVE_SHAPE),
            },
        },
        "preprocessing": {
            "image_steps": 20,
            "image_scale": "uint8_to_float32_div255",
            "wind_features": "raw20_plus_first_difference19",
            "wind_scale": 1000.0,
            "target_scale": 1000.0,
            "test_order": "test_ids_csv_exact_order",
        },
        "training_recipe": {
            "epochs": 40,
            "optimizer": "AdamW",
            "learning_rate": 3e-4,
            "weight_decay": WEIGHT_DECAY,
            "warmup_epochs": WARMUP_EPOCHS,
            "ema_decay": EMA_DECAY,
            "mixup_alpha": MIXUP_ALPHA,
        },
        "external_data_used": False,
        "pretrained_weights_used": False,
        "train_validation_combined": False,
        "test_used_for_selection": False,
        "initialization": "random_from_scratch_competition_train_only",
    }
    code7_validate_checkpoint(code7_checkpoint)

    test_dataset_code7, test_loader_code7 = make_stable_test_loader()
    code7_base_test_predictions = []
    predicted_rows_reference = None
    for variant, state_dict in zip(
        code7_checkpoint["base_variants"],
        code7_checkpoint["base_model_state_dicts"],
    ):
        prediction, rows = code7_predict_state(
            "code4_base", variant, state_dict, test_loader_code7
        )
        if predicted_rows_reference is None:
            predicted_rows_reference = rows
        else:
            assert np.array_equal(rows, predicted_rows_reference)
        code7_base_test_predictions.append(prediction)

    code7_extra_test_predictions = []
    for model_kind, variant, state_dict in zip(
        code7_checkpoint["extra_model_kinds"],
        code7_checkpoint["extra_variants"],
        code7_checkpoint["extra_model_state_dicts"],
    ):
        prediction, rows = code7_predict_state(
            model_kind, variant, state_dict, test_loader_code7
        )
        assert np.array_equal(rows, predicted_rows_reference)
        code7_extra_test_predictions.append(prediction)

    assert np.array_equal(predicted_rows_reference, test_index)
    base_stack = np.asarray(code7_base_test_predictions, dtype=np.float64)
    test_persistence = np.repeat(
        test_inputs.iloc[test_index][[WIND_COLUMNS[-1]]].to_numpy(np.float64),
        TARGET_HORIZONS,
        axis=1,
    )
    code7_test_prediction = np.empty(
        (EXPECTED_COUNTS["test"], TARGET_HORIZONS), dtype=np.float64
    )
    group_weights = np.asarray(code7_checkpoint["base_group_weights"], dtype=np.float64)
    group_alphas = np.asarray(code7_checkpoint["base_group_alphas"], dtype=np.float64)
    for group_index, (start, stop) in enumerate(CODE7_GROUP_SLICES):
        raw = np.tensordot(
            group_weights[group_index], base_stack[:, :, start:stop], axes=(0, 0)
        )
        baseline = test_persistence[:, start:stop]
        code7_test_prediction[:, start:stop] = baseline + group_alphas[group_index] * (
            raw - baseline
        )
    code7_test_prediction *= float(code7_checkpoint["base_component_weight"])
    for weight, prediction in zip(
        code7_checkpoint["extra_weights"], code7_extra_test_predictions
    ):
        code7_test_prediction += float(weight) * prediction
    assert code7_test_prediction.shape == (3868, 12)
    assert np.isfinite(code7_test_prediction).all()

    code7_submission = pd.DataFrame(code7_test_prediction, columns=TARGET_COLUMNS)
    code7_submission.insert(0, "sample_id", test_ids["sample_id"].tolist())
    assert code7_submission.shape == (3868, 13)
    assert code7_submission.columns.tolist() == EXPECTED_SUBMISSION_COLUMNS
    assert code7_submission["sample_id"].is_unique
    assert code7_submission["sample_id"].tolist() == test_ids["sample_id"].tolist()
    assert np.isfinite(code7_submission[TARGET_COLUMNS].to_numpy(np.float64)).all()
    atomic_torch_save(code7_checkpoint, CODE7_MODEL_PATH)
    atomic_csv_save(code7_submission, CODE7_SUBMISSION_PATH)
    CODE7_OUTPUT_MODE = "CODE7 IMPROVED CANDIDATE"
    del test_loader_code7, test_dataset_code7
else:
    # Exact original CODE4 champion artifacts are preserved and published.
    code7_atomic_copy(CODE4_OUTPUT_CHECKPOINT_PATH, CODE7_MODEL_PATH)
    code7_atomic_copy(CODE4_OUTPUT_SUBMISSION_PATH, CODE7_SUBMISSION_PATH)
    CODE7_OUTPUT_MODE = "EXACT CODE4 CHAMPION FALLBACK"

published_submission_code7 = pd.read_csv(CODE7_SUBMISSION_PATH)
published_model_code7 = safe_torch_load(CODE7_MODEL_PATH, map_location="cpu")
if CODE7_CANDIDATE_BEATS_CODE4:
    code7_validate_checkpoint(published_model_code7)
else:
    validate_delivery_checkpoint(published_model_code7)
assert published_submission_code7.shape == (3868, 13)
assert published_submission_code7.columns.tolist() == EXPECTED_SUBMISSION_COLUMNS
assert published_submission_code7["sample_id"].is_unique
assert published_submission_code7["sample_id"].tolist() == test_ids["sample_id"].tolist()
assert np.isfinite(
    published_submission_code7[TARGET_COLUMNS].to_numpy(np.float64)
).all()
assert CODE7_MODEL_PATH.is_file() and CODE7_SUBMISSION_PATH.is_file()
print("CODE7 output mode:", CODE7_OUTPUT_MODE)
print("model:", CODE7_MODEL_PATH)
print("submission:", CODE7_SUBMISSION_PATH)


## CODE 7 — final report

If no robust candidate beats CODE4, the exact CODE4 champion artifacts are published.


In [ ]:
# ============================================================
# CODE 7 final report and submission directory audit
# ============================================================
CODE7_FINAL_VALIDATION_PREDICTION = (
    CODE7_FINAL_CANDIDATE["prediction"]
    if CODE7_CANDIDATE_BEATS_CODE4
    else BASELINE_CHAMPION_PREDICTION
)
CODE7_FINAL_VALIDATION_RMSE = overall_rmse(
    CODE7_VALIDATION_TARGET, CODE7_FINAL_VALIDATION_PREDICTION
)
CODE7_FINAL_FOLD_RMSE = code7_fold_rmse(CODE7_FINAL_VALIDATION_PREDICTION)
CODE7_HORIZON_RMSE = np.sqrt(np.mean(
    (CODE7_FINAL_VALIDATION_PREDICTION - CODE7_VALIDATION_TARGET) ** 2,
    axis=0,
))

if not BASELINE_REPRODUCTION_OK:
    CODE7_FINAL_DECISION = "CODE (4) REMAINS BEST"
elif CODE7_FINAL_VALIDATION_RMSE < 62.30:
    CODE7_FINAL_DECISION = "VERY STRONG IMPROVEMENT"
elif CODE7_FINAL_VALIDATION_RMSE < 62.60:
    CODE7_FINAL_DECISION = "STRONG IMPROVEMENT"
elif CODE7_FINAL_VALIDATION_RMSE < 62.80:
    CODE7_FINAL_DECISION = "GOOD IMPROVEMENT"
elif CODE7_FINAL_VALIDATION_RMSE < HISTORICAL_CODE4_RMSE:
    CODE7_FINAL_DECISION = "MARGINAL IMPROVEMENT"
else:
    CODE7_FINAL_DECISION = "CODE (4) REMAINS BEST"

extra_by_kind = {result["kind"]: result for result in CODE7_EXTRA_RESULTS}
print("\n" + "=" * 92)
print("CODE 7 FINAL REPORT")
print("=" * 92)
print("Historical CODE4:", f"{HISTORICAL_CODE4_RMSE:.6f}")
print("Live reproduced CODE4:", f"{LIVE_CODE4_BASELINE:.6f}")
print("Fine global weight:", f"{CODE7_PHASE_RESULTS['fine_global']['rmse']:.6f}")
print(
    "Fine global weights:",
    np.round(CODE7_PHASE_RESULTS["fine_global"]["group_weights"][0], 6).tolist(),
)
print("3-group weight:", f"{CODE7_PHASE_RESULTS['group_weights']['rmse']:.6f}")
print(
    "3-group weights:",
    np.round(CODE7_PHASE_RESULTS["group_weights"]["group_weights"], 6).tolist(),
)
print("Residual alpha global:", f"{CODE7_PHASE_RESULTS['global_alpha']['rmse']:.6f}")
print("Residual alpha group:", f"{CODE7_PHASE_RESULTS['group_alpha']['rmse']:.6f}")
print("Raw/EMA diversity:", CODE7_RAW_EMA_STATUS)

for label, kind in (
    ("Temporal Difference model", "temporal_difference"),
    ("Trajectory Auxiliary model", "trajectory_auxiliary"),
):
    print("\n" + label + ":")
    if kind in extra_by_kind:
        result = extra_by_kind[kind]
        print("standalone:", f"{result['standalone_rmse']:.6f}")
        print("correlation:", f"{result['residual_correlation']:.6f}")
        print("best blend:", f"{result.get('best_blend_rmse', np.nan):.6f}")
        print("best source/epoch:", result["best_source"], result["best_epoch"])
    else:
        print(CODE7_EXTRA_STATUS[kind])

print("\nFINAL:")
print("RMSE:", f"{CODE7_FINAL_VALIDATION_RMSE:.6f}")
print(
    "Improvement vs code4:",
    f"{HISTORICAL_CODE4_RMSE - CODE7_FINAL_VALIDATION_RMSE:+.6f}",
)
print("output mode:", CODE7_OUTPUT_MODE)

print("\nFOLD RESULTS")
for fold, (baseline_value, final_value) in enumerate(
    zip(CODE7_BASELINE_FOLD_RMSE, CODE7_FINAL_FOLD_RMSE)
):
    print(
        f"fold{fold}: baseline={baseline_value:.6f} "
        f"final={final_value:.6f} change={final_value - baseline_value:+.6f}"
    )

print("\nHORIZON")
for horizon_index, value in enumerate(CODE7_HORIZON_RMSE):
    print(f"{(horizon_index + 1) * 6:2d}h: {value:.6f}")

print("\nFINAL MEMBERS:", list(CODE7_BASE_LABELS))
selected_extra_names = []
if CODE7_FINAL_CANDIDATE["extra_weights"].size:
    selected_extra_names = [
        result["kind"]
        for result, weight in zip(
            CODE7_EXTRA_RESULTS, CODE7_FINAL_CANDIDATE["extra_weights"]
        )
        if float(weight) > 0.0
    ]
print("FINAL EXTRA MEMBERS:", selected_extra_names)
print("FINAL WEIGHTS BY GROUP:")
print(np.round(CODE7_FINAL_CANDIDATE["group_weights"], 6))
print("GROUP ALPHA:", np.round(CODE7_FINAL_CANDIDATE["group_alphas"], 6).tolist())
print("BASE COMPONENT WEIGHT:", CODE7_FINAL_CANDIDATE["base_component_weight"])
if CODE7_FINAL_CANDIDATE["extra_weights"].size:
    print("EXTRA WEIGHTS:", np.round(CODE7_FINAL_CANDIDATE["extra_weights"], 6).tolist())
print("FINAL DECISION:", CODE7_FINAL_DECISION)
print("total elapsed:", format_duration(elapsed_seconds()))
print("=" * 92)

expected_submission_files = {"code.ipynb", "model.pth", "submission.csv"}
actual_submission_files = {
    path.name for path in CODE7_SUBMISSION_DIR.iterdir() if path.is_file()
}
print("submission directory:", CODE7_SUBMISSION_DIR)
print("files:", sorted(actual_submission_files))
if "code.ipynb" not in actual_submission_files:
    print(
        "MANUAL ACTION REQUIRED: upload한 code (7).ipynb를 "
        "submission/code.ipynb로 복사하세요."
    )
extra_submission_files = sorted(
    actual_submission_files - expected_submission_files
)
if extra_submission_files:
    warnings.warn(
        f"submission 폴더의 추가 파일을 수동 확인하세요: {extra_submission_files}"
    )


## CODE 8 — dual baseline gate, retained snapshots, TTA and specialists

The original training arithmetic/order is unchanged; only CPU checkpoint retention was added.


In [ ]:
# ============================================================
# CODE 8 GATE + snapshots/TTA/specialist diversity
# ============================================================
CODE8_STARTED_AT = time.time()
CODE8_HISTORICAL_CODE4 = 62.955678
CODE8_HISTORICAL_CODE7 = 62.906333
CODE8_CODE4_TOLERANCE = 0.05
CODE8_CODE7_TOLERANCE = 0.02
CODE8_MIN_FOLD_WINS = 3
CODE8_MAX_FOLD_REGRESSION = 0.10
CODE8_BLEND_WEIGHT_GRID = (0.025, 0.05, 0.075, 0.10, 0.125, 0.15, 0.20)
CODE8_GROUP_CHEAP_WEIGHT_GRID = (0.0, 0.05, 0.10, 0.15)
CODE8_VERTICAL_TTA_LAMBDAS = (0.25, 0.50)
CODE8_BRIGHTNESS_SCALES = (0.95, 1.00, 1.05)
CODE8_RUN_BRIGHTNESS_TTA = True
CODE8_SHORTLIST_LIMIT = 3
CODE12_FREEZE_CODE8_WINNER = True
CODE12_DEFER_TEST_INFERENCE = True

CODE8_LIVE_CODE4 = float(LIVE_CODE4_BASELINE)
CODE8_LIVE_CODE7 = float(CODE7_FINAL_VALIDATION_RMSE)
CODE8_REPRODUCTION_OK = bool(
    abs(CODE8_LIVE_CODE4 - CODE8_HISTORICAL_CODE4) <= CODE8_CODE4_TOLERANCE
    and abs(CODE8_LIVE_CODE7 - CODE8_HISTORICAL_CODE7) <= CODE8_CODE7_TOLERANCE
)
print("\n" + "=" * 90)
print("CODE 8 DUAL BASELINE GATE")
print("Historical CODE4:", f"{CODE8_HISTORICAL_CODE4:.6f}")
print("Live CODE4:      ", f"{CODE8_LIVE_CODE4:.6f}")
print("Historical CODE7:", f"{CODE8_HISTORICAL_CODE7:.6f}")
print("Live CODE7:      ", f"{CODE8_LIVE_CODE7:.6f}")
print("Gate:", "PASS" if CODE8_REPRODUCTION_OK else "BASELINE REPRODUCTION FAILED")
print("=" * 90)

CODE8_TARGET = np.asarray(CODE7_VALIDATION_TARGET, dtype=np.float64)
CODE8_PERSISTENCE = np.asarray(CODE7_PERSISTENCE_VALIDATION, dtype=np.float64)
CODE8_BASE_PREDICTION = np.asarray(
    CODE7_FINAL_VALIDATION_PREDICTION, dtype=np.float64
).copy()
CODE8_BASE_MEMBER_STACK = np.asarray(
    CODE7_BASE_MEMBER_PREDICTIONS, dtype=np.float64
)
CODE8_BASE_MEMBER_COUNT = len(CODE7_BASE_MEMBER_INDICES)
CODE8_BASE_FOLD_RMSE = code7_fold_rmse(CODE8_BASE_PREDICTION)
CODE8_CHAMPION_RESIDUAL = (CODE8_BASE_PREDICTION - CODE8_TARGET).reshape(-1)


def code8_expand_group_values(group_values):
    group_values = np.asarray(group_values, dtype=np.float64)
    assert group_values.shape[0] == 3
    return np.concatenate([
        np.repeat(
            group_values[group_index][None, :], stop - start, axis=0
        )
        for group_index, (start, stop) in enumerate(CODE7_GROUP_SLICES)
    ], axis=0)


CODE8_CODE7_HORIZON_WEIGHTS = code8_expand_group_values(
    np.asarray(CODE7_FINAL_CANDIDATE["group_weights"], dtype=np.float64)
)
CODE8_CODE7_ALPHA = code8_expand_group_values(
    np.asarray(CODE7_FINAL_CANDIDATE["group_alphas"], dtype=np.float64)[:, None]
).reshape(TARGET_HORIZONS)
assert CODE8_CODE7_HORIZON_WEIGHTS.shape == (
    TARGET_HORIZONS, CODE8_BASE_MEMBER_COUNT
)
assert np.allclose(CODE8_CODE7_HORIZON_WEIGHTS.sum(axis=1), 1.0)


def code8_member_ensemble(member_stack, horizon_weights, persistence, alpha):
    member_stack = np.asarray(member_stack, dtype=np.float64)
    horizon_weights = np.asarray(horizon_weights, dtype=np.float64)
    alpha = np.asarray(alpha, dtype=np.float64).reshape(1, TARGET_HORIZONS)
    assert member_stack.shape[0] == horizon_weights.shape[1]
    assert horizon_weights.shape[0] == TARGET_HORIZONS
    raw = np.empty((member_stack.shape[1], TARGET_HORIZONS), dtype=np.float64)
    for horizon in range(TARGET_HORIZONS):
        raw[:, horizon] = np.tensordot(
            horizon_weights[horizon], member_stack[:, :, horizon], axes=(0, 0)
        )
    return persistence + alpha * (raw - persistence)


assert np.allclose(
    code8_member_ensemble(
        CODE8_BASE_MEMBER_STACK,
        CODE8_CODE7_HORIZON_WEIGHTS,
        CODE8_PERSISTENCE,
        CODE8_CODE7_ALPHA,
    ),
    CODE8_BASE_PREDICTION,
    atol=1e-6,
    rtol=0.0,
)


def code8_fold_rmse(prediction):
    return np.asarray([
        overall_rmse(
            CODE8_TARGET[CODE7_FOLD_IDS == fold],
            prediction[CODE7_FOLD_IDS == fold],
        )
        for fold in range(CODE7_FOLD_COUNT)
    ], dtype=np.float64)


def code8_copy_recipe(recipe):
    return {
        "base_horizon_weights": np.asarray(
            recipe["base_horizon_weights"], dtype=np.float64
        ).copy(),
        "base_alpha": np.asarray(recipe["base_alpha"], dtype=np.float64).copy(),
        # State dicts / tree objects are immutable candidate artifacts and shared.
        "component": recipe.get("component"),
        "component_weight": np.asarray(
            recipe["component_weight"], dtype=np.float64
        ).copy(),
        "final_alpha": np.asarray(recipe["final_alpha"], dtype=np.float64).copy(),
    }


def code8_evaluate(name, prediction, recipe):
    prediction = np.asarray(prediction, dtype=np.float64)
    fold_rmse = code8_fold_rmse(prediction)
    fold_delta = fold_rmse - CODE8_BASE_FOLD_RMSE
    rmse = overall_rmse(CODE8_TARGET, prediction)
    return {
        "name": name,
        "prediction": prediction,
        "rmse": float(rmse),
        "fold_rmse": fold_rmse,
        "fold_wins": int(np.sum(fold_delta < 0.0)),
        "worst_fold_degradation": float(fold_delta.max()),
        "robust": bool(
            rmse < CODE8_LIVE_CODE7 - 1e-10
            and int(np.sum(fold_delta < 0.0)) >= CODE8_MIN_FOLD_WINS
            and float(fold_delta.max()) < CODE8_MAX_FOLD_REGRESSION
        ),
        "recipe": code8_copy_recipe(recipe),
    }


CODE8_BASE_RECIPE = {
    "base_horizon_weights": CODE8_CODE7_HORIZON_WEIGHTS.copy(),
    "base_alpha": CODE8_CODE7_ALPHA.copy(),
    "component": None,
    "component_weight": np.zeros(TARGET_HORIZONS, dtype=np.float64),
    "final_alpha": np.ones(TARGET_HORIZONS, dtype=np.float64),
}
CODE8_BASE_CANDIDATE = code8_evaluate(
    "code7_live_best", CODE8_BASE_PREDICTION, CODE8_BASE_RECIPE
)
CODE8_PARENT_CANDIDATES = [CODE8_BASE_CANDIDATE]
CODE8_DIVERSITY_DIAGNOSTICS = []


def code8_residual_correlation(prediction):
    residual = (np.asarray(prediction) - CODE8_TARGET).reshape(-1)
    return float(np.corrcoef(CODE8_CHAMPION_RESIDUAL, residual)[0, 1])


def code8_component_candidate(name, component_prediction, component_spec, weight_by_horizon):
    weight_by_horizon = np.asarray(weight_by_horizon, dtype=np.float64)
    assert weight_by_horizon.shape == (TARGET_HORIZONS,)
    prediction = (
        (1.0 - weight_by_horizon[None, :]) * CODE8_BASE_PREDICTION
        + weight_by_horizon[None, :] * np.asarray(component_prediction, dtype=np.float64)
    )
    recipe = code8_copy_recipe(CODE8_BASE_RECIPE)
    recipe["component"] = component_spec
    recipe["component_weight"] = weight_by_horizon.copy()
    return code8_evaluate(name, prediction, recipe)


@torch.no_grad()
def code8_predict_transform(model, loader, transform_kind, scale=1.0):
    model.eval()
    predictions, row_indexes = [], []
    for batch in loader:
        images = batch["images"].to(DEVICE, non_blocking=PIN_MEMORY)
        wind = batch["wind"].to(DEVICE, non_blocking=PIN_MEMORY)
        coronal = batch["coronal"].to(DEVICE, non_blocking=PIN_MEMORY)
        persistence = batch["persistence"].to(DEVICE, non_blocking=PIN_MEMORY)
        if transform_kind == "vertical_flip":
            images = images.flip(-2)
        elif transform_kind == "brightness":
            images = (images * float(scale)).clamp(0.0, 1.0)
        elif transform_kind != "identity":
            raise ValueError(transform_kind)
        with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):
            output = model(images, wind, persistence, coronal)
        predictions.append(output.float().cpu().numpy() * 1000.0)
        rows = batch["row_index"]
        if isinstance(rows, torch.Tensor):
            rows = rows.cpu().numpy()
        row_indexes.extend(np.asarray(rows, dtype=np.int64).tolist())
    return np.concatenate(predictions), np.asarray(row_indexes, dtype=np.int64)


def code8_predict_base_members_transform(loader, transform_kind, scale=1.0):
    predictions = []
    expected_rows = None
    for variant, state_dict in zip(
        CODE4_ACTIVE_CHECKPOINT["variants"],
        CODE4_ACTIVE_CHECKPOINT["model_state_dicts"],
    ):
        model = build_v35_model(variant)
        model.load_state_dict(state_dict, strict=True)
        prediction, rows = code8_predict_transform(
            model, loader, transform_kind, scale=scale
        )
        if expected_rows is None:
            expected_rows = rows
        else:
            assert np.array_equal(rows, expected_rows)
        predictions.append(prediction)
        del model
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    return np.asarray(predictions, dtype=np.float64), expected_rows


CODE8_SNAPSHOT_DIAGNOSTICS = []
CODE8_USEFUL_SNAPSHOTS = []
CODE8_TTA_DIAGNOSTICS = []
CODE8_SPECIALIST_DIAGNOSTICS = []
CODE8_BRIGHTNESS_COMPONENT = None

if CODE8_REPRODUCTION_OK and not CODE12_FREEZE_CODE8_WINNER:
    # A) retained raw/EMA snapshots.  At most two useful alternatives/member.
    for base_position, member_index in enumerate(CODE7_BASE_MEMBER_INDICES):
        selected_member_prediction = CODE8_BASE_MEMBER_STACK[base_position]
        selected_member_rmse = overall_rmse(CODE8_TARGET, selected_member_prediction)
        selected_member_residual = (
            selected_member_prediction - CODE8_TARGET
        ).reshape(-1)
        member_options = []
        source_bank = CODE8_SNAPSHOT_BANK.get(int(member_index), {})
        for source_name in ("raw", "ema"):
            for snapshot in source_bank.get(source_name, []):
                model = build_v35_model(ENSEMBLE_VARIANTS[member_index])
                model.load_state_dict(snapshot["state_dict"], strict=True)
                prediction, rows = predict_single_model(model, val_loader)
                assert np.array_equal(rows, val_index)
                del model
                rmse = overall_rmse(CODE8_TARGET, prediction)
                residual = (prediction - CODE8_TARGET).reshape(-1)
                selected_corr = float(np.corrcoef(selected_member_residual, residual)[0, 1])
                champion_corr = code8_residual_correlation(prediction)
                diagnostic = {
                    "member_index": int(member_index),
                    "base_position": int(base_position),
                    "source": source_name,
                    "epoch": int(snapshot["epoch"]),
                    "rmse": float(rmse),
                    "selected_member_correlation": selected_corr,
                    "champion_correlation": champion_corr,
                }
                CODE8_SNAPSHOT_DIAGNOSTICS.append(diagnostic)
                if (
                    selected_corr < 0.995
                    and rmse <= selected_member_rmse + 3.0
                ):
                    member_options.append((rmse, selected_corr, snapshot, prediction, diagnostic))
        member_options.sort(key=lambda item: (item[0], item[1], item[4]["epoch"]))
        for rmse, selected_corr, snapshot, prediction, diagnostic in member_options[:2]:
            component_spec = {
                "kind": "snapshot",
                "variant": ENSEMBLE_VARIANTS[member_index],
                "seed": int(MEMBER_SEEDS[member_index]),
                "member_index": int(member_index),
                "source": diagnostic["source"],
                "epoch": diagnostic["epoch"],
                "state_dict": snapshot["state_dict"],
            }
            CODE8_USEFUL_SNAPSHOTS.append({
                "name": (
                    f"snapshot_m{member_index}_{diagnostic['source']}_"
                    f"e{diagnostic['epoch']}"
                ),
                "prediction": np.asarray(prediction, dtype=np.float64),
                "component": component_spec,
                "standalone_rmse": float(rmse),
                "correlation": code8_residual_correlation(prediction),
            })

    # B) deterministic vertical TTA. Coronal features remain unchanged, matching training.
    vflip_member_stack, vflip_rows = code8_predict_base_members_transform(
        val_loader, "vertical_flip"
    )
    assert np.array_equal(vflip_rows, val_index)
    vflip_prediction = code8_member_ensemble(
        vflip_member_stack,
        CODE8_CODE7_HORIZON_WEIGHTS,
        CODE8_PERSISTENCE,
        CODE8_CODE7_ALPHA,
    )
    CODE8_TTA_DIAGNOSTICS.append({
        "kind": "vertical_flip",
        "rmse": overall_rmse(CODE8_TARGET, vflip_prediction),
        "correlation": code8_residual_correlation(vflip_prediction),
    })
    for tta_lambda in CODE8_VERTICAL_TTA_LAMBDAS:
        spec = {
            "kind": "vertical_flip_ensemble",
            "ensemble_horizon_weights": CODE8_CODE7_HORIZON_WEIGHTS.copy(),
            "ensemble_alpha": CODE8_CODE7_ALPHA.copy(),
        }
        CODE8_PARENT_CANDIDATES.append(code8_component_candidate(
            f"vertical_tta_lambda_{tta_lambda:.2f}",
            vflip_prediction,
            spec,
            np.repeat(float(tta_lambda), TARGET_HORIZONS),
        ))

    # Optional deterministic brightness average, evaluated but never auto-tuned on Test.
    if CODE8_RUN_BRIGHTNESS_TTA:
        brightness_predictions = []
        for brightness_scale in CODE8_BRIGHTNESS_SCALES:
            if np.isclose(brightness_scale, 1.0):
                brightness_predictions.append(CODE8_BASE_PREDICTION)
                continue
            scaled_stack, scaled_rows = code8_predict_base_members_transform(
                val_loader, "brightness", scale=brightness_scale
            )
            assert np.array_equal(scaled_rows, val_index)
            brightness_predictions.append(code8_member_ensemble(
                scaled_stack,
                CODE8_CODE7_HORIZON_WEIGHTS,
                CODE8_PERSISTENCE,
                CODE8_CODE7_ALPHA,
            ))
        brightness_average = np.mean(brightness_predictions, axis=0)
        CODE8_BRIGHTNESS_COMPONENT = {
            "name": "brightness_tta_average",
            "prediction": brightness_average,
            "component": {
                "kind": "brightness_ensemble",
                "scales": list(CODE8_BRIGHTNESS_SCALES),
                "ensemble_horizon_weights": CODE8_CODE7_HORIZON_WEIGHTS.copy(),
                "ensemble_alpha": CODE8_CODE7_ALPHA.copy(),
            },
            "standalone_rmse": overall_rmse(CODE8_TARGET, brightness_average),
            "correlation": code8_residual_correlation(brightness_average),
        }
        CODE8_TTA_DIAGNOSTICS.append({
            "kind": "brightness_average",
            "rmse": CODE8_BRIGHTNESS_COMPONENT["standalone_rmse"],
            "correlation": CODE8_BRIGHTNESS_COMPONENT["correlation"],
        })

    # Existing CODE4 specialists: diagnostic by group, then 5-15% only in one group.
    base_indices_set = set(map(int, CODE7_BASE_MEMBER_INDICES))
    for member_index in range(len(completed_predictions)):
        if member_index in base_indices_set:
            continue
        specialist_prediction = np.asarray(
            member_prediction_stack[member_index], dtype=np.float64
        )
        group_rmse = []
        for start, stop in CODE7_GROUP_SLICES:
            group_rmse.append(overall_rmse(
                CODE8_TARGET[:, start:stop], specialist_prediction[:, start:stop]
            ))
        diagnostic = {
            "member_index": int(member_index),
            "label": MEMBER_LABELS[member_index],
            "variant": ENSEMBLE_VARIANTS[member_index],
            "short_rmse": group_rmse[0],
            "middle_rmse": group_rmse[1],
            "long_rmse": group_rmse[2],
            "correlation": code8_residual_correlation(specialist_prediction),
        }
        CODE8_SPECIALIST_DIAGNOSTICS.append(diagnostic)
        specialist_candidates = []
        for group_index, (start, stop) in enumerate(CODE7_GROUP_SLICES):
            for weight in (0.05, 0.10, 0.15):
                horizon_weight = np.zeros(TARGET_HORIZONS, dtype=np.float64)
                horizon_weight[start:stop] = weight
                component = {
                    "kind": "specialist",
                    "variant": ENSEMBLE_VARIANTS[member_index],
                    "seed": int(MEMBER_SEEDS[member_index]),
                    "member_index": int(member_index),
                    "label": MEMBER_LABELS[member_index],
                    "state_dict": completed_state_dicts[member_index],
                }
                specialist_candidates.append(code8_component_candidate(
                    f"specialist_m{member_index}_g{group_index}_w{weight:.2f}",
                    specialist_prediction,
                    component,
                    horizon_weight,
                ))
        best_specialist = min(
            specialist_candidates, key=lambda candidate: candidate["rmse"]
        )
        if best_specialist["robust"]:
            CODE8_PARENT_CANDIDATES.append(best_specialist)

    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
else:
    if CODE12_FREEZE_CODE8_WINNER and CODE8_REPRODUCTION_OK:
        print("CODE8 legacy diversity phases skipped: frozen winner reconstruction mode.")
    else:
        print("CODE8 diversity phases skipped: dual baseline reproduction failed.")


## CODE 8 — Train-only tabular and analog residual candidates

All scalers and regressors are fitted exclusively on Competition Train.


In [ ]:
# ============================================================
# CODE 12 memory-safe freeze of failed CODE8 legacy candidates
# ============================================================
# The downloaded CODE8 run proved that ExtraTrees, Ridge, kNN and TTA/specialists
# do not enter the winner.  They are intentionally not imported, fitted or inferred.
CODE8_SKLEARN_AVAILABLE = False
CODE8_CHEAP_CANDIDATES = []
CODE8_CHEAP_DIAGNOSTICS = []
CODE8_BLEND_DIAGNOSTICS = []
CODE8_SHORTLIST = []
CODE8_TRAIN_FEATURES = None
CODE8_VAL_FEATURES = None
CODE8_ANALOG_TRAIN_FEATURES = None
CODE8_ANALOG_VAL_FEATURES = None
print("CODE8 legacy diversity search: DISABLED (winner component=None; memory-safe freeze)")


## CODE 8 — smooth two-parameter calibration

Only a three-candidate shortlist advances to smooth horizon calibration.


In [ ]:
# ============================================================
# CODE 8 confirmed winner freeze (downloaded source-of-truth run)
# ============================================================
CODE8_FROZEN_HORIZON_WEIGHTS = np.column_stack([
    np.linspace(0.30, 0.35, TARGET_HORIZONS),
    np.linspace(0.375, 0.40, TARGET_HORIZONS),
    np.linspace(0.325, 0.25, TARGET_HORIZONS),
]).astype(np.float64)
CODE8_FROZEN_BASE_ALPHA = np.ones(TARGET_HORIZONS, dtype=np.float64)
CODE8_FROZEN_FINAL_ALPHA = np.linspace(
    0.99, 0.95, TARGET_HORIZONS, dtype=np.float64
)
CODE8_EXPECTED_LABELS = ("V8 Physics 2", "V8 Wide 1", "V8 Wide 2")

assert tuple(CODE7_BASE_LABELS) == CODE8_EXPECTED_LABELS, (
    "CODE8 champion labels changed: " + repr(tuple(CODE7_BASE_LABELS))
)
assert CODE8_BASE_MEMBER_STACK.shape == (3, 1199, TARGET_HORIZONS)
assert np.allclose(CODE8_FROZEN_HORIZON_WEIGHTS.sum(axis=1), 1.0, atol=1e-12)

code8_frozen_parent = code8_member_ensemble(
    CODE8_BASE_MEMBER_STACK,
    CODE8_FROZEN_HORIZON_WEIGHTS,
    CODE8_PERSISTENCE,
    CODE8_FROZEN_BASE_ALPHA,
)
code8_frozen_prediction = CODE8_PERSISTENCE + CODE8_FROZEN_FINAL_ALPHA[None, :] * (
    code8_frozen_parent - CODE8_PERSISTENCE
)
code8_frozen_recipe = {
    "base_horizon_weights": CODE8_FROZEN_HORIZON_WEIGHTS.copy(),
    "base_alpha": CODE8_FROZEN_BASE_ALPHA.copy(),
    "component": None,
    "component_weight": np.zeros(TARGET_HORIZONS, dtype=np.float64),
    "final_alpha": CODE8_FROZEN_FINAL_ALPHA.copy(),
}
code8_frozen_candidate = code8_evaluate(
    "frozen_code8_winner_linear_weight_alpha",
    code8_frozen_prediction,
    code8_frozen_recipe,
)
CODE8_CALIBRATION_CANDIDATES = [code8_frozen_candidate]
CODE8_QUADRATIC_TRIGGERED = False
CODE8_FINAL_CANDIDATE = code8_frozen_candidate
CODE8_BEATS_CODE7 = bool(
    CODE8_REPRODUCTION_OK
    and code8_frozen_candidate["rmse"] < CODE8_LIVE_CODE7
    and abs(code8_frozen_candidate["rmse"] - 62.903266) <= 0.01
)
if CODE8_REPRODUCTION_OK and not CODE8_BEATS_CODE7:
    raise RuntimeError(
        "Frozen CODE8 winner did not reproduce 62.903266; CODE12 is stopped."
    )
print("Frozen CODE8 winner RMSE:", f"{code8_frozen_candidate['rmse']:.6f}")
print("Frozen CODE8 component: None")


## CODE 8 — frozen Validation decision, Test inference and artifacts


In [ ]:
# ============================================================
# CODE12 defers Test inference and final artifact publication
# ============================================================
assert CODE12_DEFER_TEST_INFERENCE is True
assert CODE8_BEATS_CODE7, "Frozen CODE8 baseline must reproduce before CODE12"
CODE8_OUTPUT_MODE = "CODE12 DEFERRED"
print("CODE8 Test inference deferred until CODE12 Validation gate/fallback.")


## CODE 8 — final diversity and calibration report


In [ ]:
# ============================================================
# CODE 8 final diagnostics/report
# ============================================================
CODE8_FINAL_PREDICTION = (
    CODE8_FINAL_CANDIDATE["prediction"]
    if CODE8_BEATS_CODE7
    else CODE8_BASE_PREDICTION
)
CODE8_FINAL_RMSE = overall_rmse(CODE8_TARGET, CODE8_FINAL_PREDICTION)
CODE8_FINAL_FOLDS = code8_fold_rmse(CODE8_FINAL_PREDICTION)
CODE8_FINAL_HORIZONS = np.sqrt(np.mean(
    (CODE8_FINAL_PREDICTION - CODE8_TARGET) ** 2, axis=0
))

if CODE8_FINAL_RMSE < 62.40:
    CODE8_DECISION = "VERY STRONG IMPROVEMENT"
elif CODE8_FINAL_RMSE < 62.60:
    CODE8_DECISION = "STRONG IMPROVEMENT"
elif CODE8_FINAL_RMSE < 62.80:
    CODE8_DECISION = "GOOD IMPROVEMENT"
elif CODE8_FINAL_RMSE < CODE8_HISTORICAL_CODE7:
    CODE8_DECISION = "MARGINAL IMPROVEMENT"
else:
    CODE8_DECISION = "CODE 7 REMAINS BEST"

print("\n" + "=" * 96)
print("CODE 8 DIVERSITY & CALIBRATION REPORT")
print("=" * 96)
print("Historical CODE4:", f"{CODE8_HISTORICAL_CODE4:.6f}")
print("Live CODE4:", f"{CODE8_LIVE_CODE4:.6f}")
print("Historical CODE7:", f"{CODE8_HISTORICAL_CODE7:.6f}")
print("Live CODE7:", f"{CODE8_LIVE_CODE7:.6f}")
print("\nSNAPSHOTS")
if CODE8_SNAPSHOT_DIAGNOSTICS:
    print(pd.DataFrame(CODE8_SNAPSHOT_DIAGNOSTICS).to_string(index=False))
else:
    print("No useful retained snapshot available")
print("\nTTA")
print(pd.DataFrame(CODE8_TTA_DIAGNOSTICS).to_string(index=False))
print("\nSPECIALISTS")
print(pd.DataFrame(CODE8_SPECIALIST_DIAGNOSTICS).to_string(index=False))
print("\nTABULAR / ANALOG")
print(pd.DataFrame(CODE8_CHEAP_DIAGNOSTICS).to_string(index=False))
print("\nBEST BLENDS")
if CODE8_BLEND_DIAGNOSTICS:
    print(pd.DataFrame(CODE8_BLEND_DIAGNOSTICS).sort_values(
        "best_blend_rmse"
    ).head(20).to_string(index=False))
else:
    print("No blend candidate evaluated")
print("\nSHORTLIST")
for candidate in CODE8_SHORTLIST:
    print(
        candidate["name"],
        f"RMSE={candidate['rmse']:.6f}",
        f"folds={candidate['fold_wins']}/5",
        f"worst={candidate['worst_fold_degradation']:+.6f}",
    )
print("\nFINAL")
print("strategy:", CODE8_FINAL_CANDIDATE["name"])
print("RMSE:", f"{CODE8_FINAL_RMSE:.6f}")
print("improvement vs CODE7:", f"{CODE8_LIVE_CODE7 - CODE8_FINAL_RMSE:+.6f}")
print("fold wins:", CODE8_FINAL_CANDIDATE["fold_wins"], "/5")
print(
    "worst fold degradation:",
    f"{CODE8_FINAL_CANDIDATE['worst_fold_degradation']:+.6f}",
)
final_recipe = CODE8_FINAL_CANDIDATE["recipe"]
print("base horizon weights:")
print(np.round(final_recipe["base_horizon_weights"], 6))
print("base alpha:", np.round(final_recipe["base_alpha"], 6).tolist())
print(
    "component:",
    None if final_recipe["component"] is None else final_recipe["component"]["kind"],
)
print(
    "component weight:",
    np.round(final_recipe["component_weight"], 6).tolist(),
)
print("final alpha:", np.round(final_recipe["final_alpha"], 6).tolist())
print("FOLDS")
for fold, (base_value, final_value) in enumerate(
    zip(CODE8_BASE_FOLD_RMSE, CODE8_FINAL_FOLDS)
):
    print(
        f"fold{fold}: CODE7={base_value:.6f} CODE8={final_value:.6f} "
        f"change={final_value - base_value:+.6f}"
    )
print("HORIZONS")
for horizon, value in enumerate(CODE8_FINAL_HORIZONS, start=1):
    print(f"{horizon * 6:2d}h: {value:.6f}")
print("output mode:", CODE8_OUTPUT_MODE)
print("FINAL DECISION:", CODE8_DECISION)
print("total elapsed:", format_duration(elapsed_seconds()))
print("submission model:", CODE7_MODEL_PATH)
print("submission csv:", CODE7_SUBMISSION_PATH)
print("=" * 96)

submission_files_code8 = {
    path.name for path in CODE7_SUBMISSION_DIR.iterdir() if path.is_file()
}
print("submission files:", sorted(submission_files_code8))
if "code.ipynb" not in submission_files_code8:
    print(
        "MANUAL ACTION REQUIRED: code (8).ipynb를 "
        "submission/code.ipynb로 복사하세요."
    )


# CODE 12 — Sparse Oracle-Guided Expert Routing

Direct file-level descendant of CODE8. The original 11-member training recipe above is
preserved. CODE10/CODE11 predictions are not used. Existing CODE8 specialist predictions
are first audited for oracle headroom; only then may a tiny Train-only event classifier
provide a target-free hard routing signal. CODE8 remains the default prediction everywhere.


In [ ]:
# ============================================================
# Exact CODE8 gate + all existing experts + oracle upper bounds
# ============================================================
from torch.utils.data import TensorDataset

CODE12_HISTORICAL_CODE8 = 62.903266
CODE12_REPRODUCTION_TOLERANCE = 0.01
CODE12_ORACLE_SAMPLE_GATE = 0.10
CODE12_ORACLE_ELEMENT_GATE = 0.20
CODE12_REALISTIC_HEADROOM_GATE = 0.10
CODE12_MAX_ROUND1 = 2000
CODE12_MAX_ROUND2 = 500
CODE12_MAX_ROUND3 = 300
CODE12_MIN_FOLD_WINS = 3
CODE12_RECOMMENDED_FOLD_WINS = 4
CODE12_MAX_FOLD_DEGRADATION = 0.06
CODE12_LAST_SHOT_FULL_GAIN = 0.10
CODE12_LAST_SHOT_OOF_GAIN = 0.08
CODE12_GATE_SEEDS = (1501, 1502, 1503)
CODE12_GATE_MAX_EPOCHS = 25
CODE12_GATE_MIN_EPOCHS = 20
CODE12_GATE_PATIENCE = 5
CODE12_GATE_BATCH_SIZE = 256
CODE12_GATE_LR = 1e-3
CODE12_GATE_WEIGHT_DECAY = 1e-4

CODE12_TARGET = np.asarray(CODE8_TARGET, dtype=np.float64)
CODE12_PERSISTENCE = np.asarray(CODE8_PERSISTENCE, dtype=np.float64)
CODE12_P8 = np.asarray(CODE8_FINAL_CANDIDATE["prediction"], dtype=np.float64).copy()
CODE12_BASE_MEMBER_STACK = np.asarray(CODE8_BASE_MEMBER_STACK, dtype=np.float64).copy()
CODE12_ALL_MEMBER_STACK = np.asarray(member_prediction_stack, dtype=np.float64).copy()
CODE12_LIVE_CODE8 = overall_rmse(CODE12_TARGET, CODE12_P8)
CODE12_REPRODUCTION_OK = bool(
    abs(CODE12_LIVE_CODE8 - CODE12_HISTORICAL_CODE8)
    <= CODE12_REPRODUCTION_TOLERANCE
)
assert CODE12_TARGET.shape == (1199, 12)
assert CODE12_P8.shape == (1199, 12)
assert CODE12_BASE_MEMBER_STACK.shape == (3, 1199, 12)
assert CODE12_ALL_MEMBER_STACK.shape == (ENSEMBLE_SIZE, 1199, 12)
assert len(completed_state_dicts) == ENSEMBLE_SIZE == 11
print("\n" + "=" * 92)
print("CODE12 BASELINE REPRODUCTION GATE")
print("Historical CODE8:", f"{CODE12_HISTORICAL_CODE8:.6f}")
print("Live CODE8:      ", f"{CODE12_LIVE_CODE8:.6f}")
print("Gate:", "PASS" if CODE12_REPRODUCTION_OK else "CODE8 REPRODUCTION FAILED")
print("=" * 92)
if not CODE12_REPRODUCTION_OK:
    raise RuntimeError("CODE8 REPRODUCTION FAILED — CODE12 stopped")

CODE12_ALL_STATES = [
    {key: value.detach().cpu().clone() for key, value in state.items()}
    for state in completed_state_dicts
]
CODE12_BASE_STATES = [
    {key: value.detach().cpu().clone() for key, value in state.items()}
    for state in CODE4_ACTIVE_CHECKPOINT["model_state_dicts"]
]
CODE12_BASE_VARIANTS = tuple(CODE4_ACTIVE_CHECKPOINT["variants"])
CODE12_BASE_LABELS = tuple(CODE4_ACTIVE_CHECKPOINT["labels"])
CODE12_BASE_SEEDS = tuple(int(value) for value in CODE4_ACTIVE_CHECKPOINT["seeds"])
assert CODE12_BASE_LABELS == ("V8 Physics 2", "V8 Wide 1", "V8 Wide 2")


def code12_rmse(target, prediction):
    return float(np.sqrt(np.mean(np.square(
        np.asarray(prediction, dtype=np.float64)
        - np.asarray(target, dtype=np.float64)
    ))))


CODE12_BASE_SQUARED = np.square(CODE12_TARGET - CODE12_P8)
CODE12_BASE_SAMPLE_SSE = CODE12_BASE_SQUARED.sum(axis=1)
CODE12_ORACLE_ROWS = []
CODE12_ORACLE_PREDICTIONS = {}
CODE12_REALISTIC_ORACLES = []
for expert_index in range(ENSEMBLE_SIZE):
    expert = CODE12_ALL_MEMBER_STACK[expert_index]
    expert_squared = np.square(CODE12_TARGET - expert)
    gain = CODE12_BASE_SQUARED - expert_squared
    expert_sample_sse = expert_squared.sum(axis=1)
    element_mask = gain > 0.0
    sample_mask = expert_sample_sse < CODE12_BASE_SAMPLE_SSE
    element_oracle = np.where(element_mask, expert, CODE12_P8)
    sample_oracle = np.where(sample_mask[:, None], expert, CODE12_P8)
    expert_horizon_sse = expert_squared.sum(axis=0)
    base_horizon_sse = CODE12_BASE_SQUARED.sum(axis=0)
    horizon_mask = expert_horizon_sse < base_horizon_sse
    horizon_oracle = np.where(horizon_mask[None, :], expert, CODE12_P8)
    element_oracle_rmse = code12_rmse(CODE12_TARGET, element_oracle)
    sample_oracle_rmse = code12_rmse(CODE12_TARGET, sample_oracle)
    horizon_oracle_rmse = code12_rmse(CODE12_TARGET, horizon_oracle)
    correlation = float(np.corrcoef(
        (expert - CODE12_TARGET).reshape(-1),
        (CODE12_P8 - CODE12_TARGET).reshape(-1),
    )[0, 1])
    row = {
        "expert_index": expert_index,
        "expert": MEMBER_LABELS[expert_index],
        "standalone_rmse": code12_rmse(CODE12_TARGET, expert),
        "residual_correlation": correlation,
        "beats_element_pct": float(100.0 * element_mask.mean()),
        "beats_sample_pct": float(100.0 * sample_mask.mean()),
        "positive_gain_sse": float(gain[gain > 0].sum()),
        "negative_gain_sse": float(-gain[gain < 0].sum()),
        "element_oracle_rmse": element_oracle_rmse,
        "sample_oracle_rmse": sample_oracle_rmse,
        "horizon_oracle_rmse": horizon_oracle_rmse,
        "element_oracle_improvement": CODE12_LIVE_CODE8 - element_oracle_rmse,
        "sample_oracle_improvement": CODE12_LIVE_CODE8 - sample_oracle_rmse,
    }
    row["passes_oracle_potential"] = bool(
        row["sample_oracle_improvement"] >= CODE12_ORACLE_SAMPLE_GATE
        or row["element_oracle_improvement"] >= CODE12_ORACLE_ELEMENT_GATE
    )
    CODE12_ORACLE_ROWS.append(row)
    CODE12_ORACLE_PREDICTIONS[expert_index] = {
        "element": element_oracle, "sample": sample_oracle, "horizon": horizon_oracle,
    }
    # Diagnostic only: target-derived top positive-gain masks never enter routing.
    flat_gain = gain.reshape(-1)
    positive_order = np.argsort(-flat_gain, kind="mergesort")
    for requested_coverage in (0.05, 0.10, 0.15, 0.20, 0.25):
        count = int(round(flat_gain.size * requested_coverage))
        selected = positive_order[:count]
        selected = selected[flat_gain[selected] > 0.0]
        oracle_mask = np.zeros(flat_gain.size, dtype=bool)
        oracle_mask[selected] = True
        oracle_mask = oracle_mask.reshape(CODE12_P8.shape)
        for weight in (0.05, 0.10, 0.15, 0.20, 0.25, 0.30):
            prediction = CODE12_P8 + weight * oracle_mask * (expert - CODE12_P8)
            CODE12_REALISTIC_ORACLES.append({
                "expert_index": expert_index, "expert": MEMBER_LABELS[expert_index],
                "coverage": float(oracle_mask.mean()), "weight": weight,
                "rmse": code12_rmse(CODE12_TARGET, prediction),
            })

CODE12_ORACLE_TABLE = pd.DataFrame(CODE12_ORACLE_ROWS)
# All seven existing specialists enter the cheap candidate bank. Oracle filtering
# itself is repeated inside each 4-fold meta-training partition below.
CODE12_ROUTING_INDICES = list(SPECIALIST_MEMBERS)
code12_eligible_realistic = [
    row for row in CODE12_REALISTIC_ORACLES
    if row["expert_index"] in SPECIALIST_MEMBERS
]
CODE12_BEST_REALISTIC_ORACLE = (
    min(code12_eligible_realistic, key=lambda row: (
        row["rmse"], row["coverage"], row["weight"]
    )) if code12_eligible_realistic else None
)
CODE12_REALISTIC_ORACLE_IMPROVEMENT = (
    CODE12_LIVE_CODE8 - CODE12_BEST_REALISTIC_ORACLE["rmse"]
    if CODE12_BEST_REALISTIC_ORACLE is not None else 0.0
)
CODE12_ORACLE_HEADROOM_OK = bool(
    CODE12_REALISTIC_ORACLE_IMPROVEMENT >= CODE12_REALISTIC_HEADROOM_GATE
)
print("\nORACLE TABLE")
print(CODE12_ORACLE_TABLE.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print("Best realistic sparse oracle:", CODE12_BEST_REALISTIC_ORACLE)
print("Specialist candidate bank:", [MEMBER_LABELS[i] for i in CODE12_ROUTING_INDICES])
print("Full-Validation oracle-pass specialists:", [
    MEMBER_LABELS[i] for i in SPECIALIST_MEMBERS
    if CODE12_ORACLE_ROWS[i]["passes_oracle_potential"]
])
if not CODE12_ORACLE_HEADROOM_OK:
    print("ROUTING HAS INSUFFICIENT HEADROOM — gate training/search will be skipped")


In [ ]:
# ============================================================
# Target-free routing signals + conditional Train-only event gate
# ============================================================
def code12_resolve_coronal_indices(inputs, row_index, coronal_index):
    row_index = np.asarray(row_index, dtype=np.int64)
    selected = inputs.iloc[row_index]
    if isinstance(coronal_index, dict):
        resolved = np.asarray([
            [coronal_index[name] for name in names]
            for names in selected[IMAGE_COLUMNS].itertuples(index=False, name=None)
        ], dtype=np.int64)
    else:
        numeric = np.asarray(coronal_index, dtype=np.int64)
        if numeric.ndim != 2 or numeric.shape[1] != len(IMAGE_COLUMNS):
            raise ValueError(f"invalid coronal_index shape: {numeric.shape}")
        resolved = numeric[row_index]
    assert resolved.shape == (len(row_index), 20)
    return selected, resolved


def code12_build_gate_features(inputs, row_index, coronal_array, coronal_index):
    selected, resolved = code12_resolve_coronal_indices(
        inputs, row_index, coronal_index
    )
    wind_raw = selected[WIND_COLUMNS].to_numpy(np.float32) / 1000.0
    wind_diff = np.diff(wind_raw, axis=1).astype(np.float32)
    coronal = np.asarray(coronal_array[resolved], dtype=np.float32)
    assert coronal.shape == (len(row_index), 20, 72)
    recent4 = coronal[:, -4:, :]
    centered = np.asarray([-1.5, -0.5, 0.5, 1.5], dtype=np.float32)
    recent_slope = np.sum(
        recent4 * centered[None, :, None], axis=1
    ) / np.sum(np.square(centered))
    summaries = np.concatenate([
        coronal[:, -1, :], coronal.mean(axis=1), coronal.std(axis=1),
        coronal[:, -1, :] - coronal[:, 0, :], recent4.mean(axis=1),
        recent_slope,
    ], axis=1).astype(np.float32)
    features = np.concatenate([wind_raw, wind_diff, summaries], axis=1).astype(np.float32)
    assert features.shape == (len(row_index), 471)
    assert np.isfinite(features).all()
    return features


def code12_internal_split(inputs, row_index, fraction=0.20, seed=1500):
    row_index = np.asarray(row_index, dtype=np.int64)
    n_rows = len(row_index)
    parent = np.arange(n_rows, dtype=np.int64)
    def find(index):
        while parent[index] != index:
            parent[index] = parent[parent[index]]
            index = parent[index]
        return index
    def union(left, right):
        left_root, right_root = find(left), find(right)
        if left_root != right_root:
            parent[right_root] = left_root
    owners = {}
    selected = inputs.iloc[row_index]
    for local_row, names in enumerate(
        selected[IMAGE_COLUMNS].itertuples(index=False, name=None)
    ):
        for name in names:
            union(local_row, owners.setdefault(name, local_row))
    groups = {}
    for local_row in range(n_rows):
        groups.setdefault(find(local_row), []).append(local_row)
    largest_fraction = max(map(len, groups.values())) / n_rows
    rng = np.random.default_rng(seed)
    if len(groups) >= 5 and largest_fraction <= 0.35:
        keys = np.asarray(sorted(groups), dtype=np.int64)
        rng.shuffle(keys)
        holdout, target_size = [], int(round(n_rows * fraction))
        for key in keys:
            if len(holdout) >= target_size:
                break
            holdout.extend(groups[int(key)])
        holdout = np.asarray(sorted(holdout), dtype=np.int64)
        kind = "shared-image group split"
    else:
        holdout = np.sort(rng.permutation(n_rows)[:int(round(n_rows * fraction))])
        kind = "deterministic shuffled split (image graph too connected)"
    internal_train = np.setdiff1d(np.arange(n_rows), holdout, assume_unique=True)
    assert len(np.intersect1d(internal_train, holdout)) == 0
    return internal_train, holdout, kind


class Code12EventGate(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(471, 192), nn.LayerNorm(192), nn.GELU(),
            nn.Dropout(0.15), nn.Linear(192, 96), nn.GELU(), nn.Linear(96, 24),
        )
    def forward(self, features):
        return self.network(features)


def code12_binary_roc_auc(labels, probabilities):
    labels = np.asarray(labels, dtype=np.int64)
    probabilities = np.asarray(probabilities, dtype=np.float64)
    positives, negatives = int(labels.sum()), int(len(labels) - labels.sum())
    if positives == 0 or negatives == 0:
        return float("nan")
    order = np.argsort(probabilities, kind="mergesort")
    sorted_probabilities = probabilities[order]
    ranks = np.arange(1, len(labels) + 1, dtype=np.float64)
    start = 0
    while start < len(labels):
        stop = start + 1
        while stop < len(labels) and sorted_probabilities[stop] == sorted_probabilities[start]:
            stop += 1
        ranks[start:stop] = 0.5 * ((start + 1) + stop)
        start = stop
    positive_rank_sum = ranks[labels[order] == 1].sum()
    return float(
        (positive_rank_sum - positives * (positives + 1) / 2)
        / (positives * negatives)
    )


def code12_binary_pr_auc(labels, probabilities):
    labels = np.asarray(labels, dtype=np.int64)
    positives = int(labels.sum())
    if positives == 0:
        return float("nan")
    order = np.argsort(-np.asarray(probabilities), kind="mergesort")
    sorted_labels = labels[order]
    true_positive = np.cumsum(sorted_labels)
    precision = true_positive / np.arange(1, len(labels) + 1)
    return float(precision[sorted_labels == 1].sum() / positives)


def code12_predict_gate(model, features, batch_size=1024):
    model.eval()
    outputs = []
    with torch.no_grad():
        for start in range(0, len(features), batch_size):
            batch = torch.from_numpy(features[start:start + batch_size]).to(
                DEVICE, non_blocking=PIN_MEMORY
            )
            outputs.append(torch.sigmoid(model(batch)).cpu().numpy())
    return np.concatenate(outputs, axis=0).astype(np.float32)


CODE12_GATE_STATES = []
CODE12_GATE_DIAGNOSTICS = []
CODE12_GATE_MEAN = CODE12_GATE_STD = None
CODE12_EVENT_Q75 = CODE12_EVENT_Q90 = None
CODE12_VAL_GATE_PROBABILITIES = None
CODE12_GATE_INFORMATIVE = False
CODE12_GATE_FEATURES_TRAIN = CODE12_GATE_FEATURES_VAL = None
CODE12_INTERNAL_TRAIN = CODE12_INTERNAL_HOLDOUT = None
CODE12_SPLIT_KIND = "not executed"

if CODE12_ORACLE_HEADROOM_OK:
    CODE12_GATE_FEATURES_TRAIN = code12_build_gate_features(
        train_inputs, train_index, train_coronal_array, train_coronal_index
    )
    CODE12_GATE_FEATURES_VAL = code12_build_gate_features(
        val_inputs, val_index, val_coronal_array, val_coronal_index
    )
    CODE12_INTERNAL_TRAIN, CODE12_INTERNAL_HOLDOUT, CODE12_SPLIT_KIND = (
        code12_internal_split(train_inputs, train_index)
    )
    train_persistence = np.repeat(
        train_inputs.iloc[train_index][[WIND_COLUMNS[-1]]].to_numpy(np.float64),
        TARGET_HORIZONS, axis=1,
    )
    train_delta = np.abs(np.asarray(train_targets[train_index]) - train_persistence)
    # Event definition uses Competition Train only, matching the proven CODE10 gate.
    CODE12_EVENT_Q75 = np.quantile(train_delta, 0.75, axis=0)
    CODE12_EVENT_Q90 = np.quantile(train_delta, 0.90, axis=0)
    event_labels = np.concatenate([
        train_delta >= CODE12_EVENT_Q75[None, :],
        train_delta >= CODE12_EVENT_Q90[None, :],
    ], axis=1).astype(np.float32)
    CODE12_GATE_MEAN = CODE12_GATE_FEATURES_TRAIN[CODE12_INTERNAL_TRAIN].mean(
        axis=0, dtype=np.float64
    ).astype(np.float32)
    CODE12_GATE_STD = CODE12_GATE_FEATURES_TRAIN[CODE12_INTERNAL_TRAIN].std(
        axis=0, dtype=np.float64
    ).astype(np.float32)
    CODE12_GATE_STD = np.where(
        CODE12_GATE_STD < 1e-6, 1.0, CODE12_GATE_STD
    ).astype(np.float32)
    train_scaled = (
        (CODE12_GATE_FEATURES_TRAIN - CODE12_GATE_MEAN) / CODE12_GATE_STD
    ).astype(np.float32)
    val_scaled = (
        (CODE12_GATE_FEATURES_VAL - CODE12_GATE_MEAN) / CODE12_GATE_STD
    ).astype(np.float32)

    def code12_train_gate(seed):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        dataset = TensorDataset(
            torch.from_numpy(train_scaled[CODE12_INTERNAL_TRAIN]),
            torch.from_numpy(event_labels[CODE12_INTERNAL_TRAIN]),
        )
        generator = torch.Generator().manual_seed(seed)
        loader = DataLoader(
            dataset, batch_size=CODE12_GATE_BATCH_SIZE, shuffle=True,
            generator=generator, num_workers=0, pin_memory=PIN_MEMORY,
        )
        positive = event_labels[CODE12_INTERNAL_TRAIN].sum(axis=0)
        negative = len(CODE12_INTERNAL_TRAIN) - positive
        pos_weight = np.clip(negative / np.maximum(positive, 1.0), 1.0, 20.0)
        criterion = nn.BCEWithLogitsLoss(
            pos_weight=torch.from_numpy(pos_weight.astype(np.float32)).to(DEVICE)
        )
        model = Code12EventGate().to(DEVICE)
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=CODE12_GATE_LR,
            weight_decay=CODE12_GATE_WEIGHT_DECAY,
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=CODE12_GATE_MAX_EPOCHS
        )
        holdout_x = torch.from_numpy(train_scaled[CODE12_INTERNAL_HOLDOUT]).to(DEVICE)
        holdout_y = torch.from_numpy(event_labels[CODE12_INTERNAL_HOLDOUT]).to(DEVICE)
        best_loss, best_epoch, best_state, patience = float("inf"), 0, None, 0
        for epoch in range(1, CODE12_GATE_MAX_EPOCHS + 1):
            model.train()
            for batch_x, batch_y in loader:
                batch_x = batch_x.to(DEVICE, non_blocking=PIN_MEMORY)
                batch_y = batch_y.to(DEVICE, non_blocking=PIN_MEMORY)
                optimizer.zero_grad(set_to_none=True)
                loss = criterion(model(batch_x), batch_y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()
            scheduler.step()
            model.eval()
            with torch.no_grad():
                holdout_loss = float(criterion(model(holdout_x), holdout_y).item())
            if holdout_loss < best_loss - 1e-6:
                best_loss, best_epoch, patience = holdout_loss, epoch, 0
                best_state = {
                    key: value.detach().cpu().clone()
                    for key, value in model.state_dict().items()
                }
            else:
                patience += 1
            if epoch >= CODE12_GATE_MIN_EPOCHS and patience >= CODE12_GATE_PATIENCE:
                break
        assert best_state is not None and best_epoch >= 1 and np.isfinite(best_loss)
        del model, loader, dataset
        return best_state, int(best_epoch), float(best_loss)

    gate_predictions = []
    for gate_seed in CODE12_GATE_SEEDS:
        gate_state, best_epoch, best_loss = code12_train_gate(gate_seed)
        gate_model = Code12EventGate().to(DEVICE)
        gate_model.load_state_dict(gate_state, strict=True)
        gate_predictions.append(code12_predict_gate(gate_model, val_scaled))
        CODE12_GATE_STATES.append(gate_state)
        CODE12_GATE_DIAGNOSTICS.append({
            "seed": gate_seed, "best_epoch": best_epoch,
            "internal_holdout_bce": best_loss,
        })
        del gate_model
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    CODE12_VAL_GATE_PROBABILITIES = np.mean(gate_predictions, axis=0).astype(np.float32)
    val_delta = np.abs(CODE12_TARGET - CODE12_PERSISTENCE)
    val_labels = np.concatenate([
        val_delta >= CODE12_EVENT_Q75[None, :],
        val_delta >= CODE12_EVENT_Q90[None, :],
    ], axis=1).astype(np.int64)
    CODE12_EVENT_ROC = np.asarray([
        code12_binary_roc_auc(val_labels[:, output], CODE12_VAL_GATE_PROBABILITIES[:, output])
        for output in range(24)
    ])
    CODE12_EVENT_PR = np.asarray([
        code12_binary_pr_auc(val_labels[:, output], CODE12_VAL_GATE_PROBABILITIES[:, output])
        for output in range(24)
    ])
    CODE12_LARGE_ROC = float(np.nanmean(CODE12_EVENT_ROC[:12]))
    CODE12_EXTREME_ROC = float(np.nanmean(CODE12_EVENT_ROC[12:]))
    CODE12_LARGE_PR = float(np.nanmean(CODE12_EVENT_PR[:12]))
    CODE12_EXTREME_PR = float(np.nanmean(CODE12_EVENT_PR[12:]))
    CODE12_GATE_INFORMATIVE = not (
        CODE12_LARGE_ROC < 0.58 and CODE12_EXTREME_ROC < 0.58
    )
    CODE12_EVENT_RISK = (
        0.65 * CODE12_VAL_GATE_PROBABILITIES[:, :12]
        + 0.35 * CODE12_VAL_GATE_PROBABILITIES[:, 12:]
    ).astype(np.float64)
else:
    CODE12_EVENT_ROC = CODE12_EVENT_PR = np.full(24, np.nan)
    CODE12_LARGE_ROC = CODE12_EXTREME_ROC = float("nan")
    CODE12_LARGE_PR = CODE12_EXTREME_PR = float("nan")
    CODE12_EVENT_RISK = np.zeros_like(CODE12_P8)

# All scales use Validation predictions/input only, never Validation targets.
code12_member_std = CODE12_BASE_MEMBER_STACK.std(axis=0)
code12_delta_abs = np.abs(CODE12_P8 - CODE12_PERSISTENCE)
code12_physics_wide_gap = np.abs(
    CODE12_BASE_MEMBER_STACK[0] - CODE12_BASE_MEMBER_STACK[1:].mean(axis=0)
)
CODE12_U_SCALE = np.maximum(np.quantile(code12_member_std, 0.75, axis=0), 1e-6)
CODE12_D_SCALE = np.maximum(np.quantile(code12_delta_abs, 0.75, axis=0), 1e-6)
CODE12_PW_SCALE = np.maximum(np.quantile(code12_physics_wide_gap, 0.75, axis=0), 1e-6)
CODE12_U = np.clip(code12_member_std / CODE12_U_SCALE[None, :], 0.0, 1.0)
CODE12_D = np.clip(code12_delta_abs / CODE12_D_SCALE[None, :], 0.0, 1.0)
CODE12_PW = np.clip(code12_physics_wide_gap / CODE12_PW_SCALE[None, :], 0.0, 1.0)
CODE12_SCORE_ARRAYS = {
    "uncertainty_delta": CODE12_U * CODE12_D,
    "physics_wide_delta": CODE12_PW * CODE12_D,
}
if CODE12_GATE_INFORMATIVE:
    CODE12_SCORE_ARRAYS.update({
        "event_risk": CODE12_EVENT_RISK,
        "event_uncertainty": CODE12_EVENT_RISK * CODE12_U,
    })
print("event gate executed:", bool(CODE12_GATE_STATES))
print("event gate informative:", CODE12_GATE_INFORMATIVE)
print("large ROC/PR:", CODE12_LARGE_ROC, CODE12_LARGE_PR)
print("extreme ROC/PR:", CODE12_EXTREME_ROC, CODE12_EXTREME_PR)
print("routing score families:", sorted(CODE12_SCORE_ARRAYS))


In [ ]:
# ============================================================
# Regime oracle diagnostics (targets used only for this report)
# ============================================================
code12_train_persistence = np.repeat(
    train_inputs.iloc[train_index][[WIND_COLUMNS[-1]]].to_numpy(np.float64),
    TARGET_HORIZONS, axis=1,
)
code12_train_target = np.asarray(train_targets[train_index], dtype=np.float64)
code12_train_change = np.abs(code12_train_target - code12_train_persistence)
code12_change_edges = np.quantile(code12_train_change.reshape(-1), (0.50, 0.75, 0.90))
code12_speed_edges = np.quantile(code12_train_target.reshape(-1), (0.75, 0.90))
code12_val_change = np.abs(CODE12_TARGET - CODE12_PERSISTENCE)
CODE12_REGIME_MASKS = {
    "change_small": code12_val_change < code12_change_edges[0],
    "change_medium": (
        (code12_val_change >= code12_change_edges[0])
        & (code12_val_change < code12_change_edges[1])
    ),
    "change_large": (
        (code12_val_change >= code12_change_edges[1])
        & (code12_val_change < code12_change_edges[2])
    ),
    "change_extreme": code12_val_change >= code12_change_edges[2],
    "speed_normal": CODE12_TARGET < code12_speed_edges[0],
    "speed_high": (
        (CODE12_TARGET >= code12_speed_edges[0])
        & (CODE12_TARGET < code12_speed_edges[1])
    ),
    "speed_extreme": CODE12_TARGET >= code12_speed_edges[1],
}
for group_name, (start, stop) in {
    "horizon_short": (0, 4), "horizon_middle": (4, 8), "horizon_long": (8, 12),
}.items():
    mask = np.zeros_like(CODE12_TARGET, dtype=bool)
    mask[:, start:stop] = True
    CODE12_REGIME_MASKS[group_name] = mask
for prefix, values in (
    ("disagreement", CODE12_U), ("predicted_delta", CODE12_D),
):
    edges = np.quantile(values.reshape(-1), (0.25, 0.50, 0.75))
    lower = -np.inf
    for quartile, upper in enumerate((*edges, np.inf), start=1):
        CODE12_REGIME_MASKS[f"{prefix}_q{quartile}"] = (
            (values > lower) & (values <= upper)
        )
        lower = upper
if CODE12_GATE_INFORMATIVE:
    edges = np.quantile(CODE12_EVENT_RISK.reshape(-1), (0.25, 0.50, 0.75))
    lower = -np.inf
    for quartile, upper in enumerate((*edges, np.inf), start=1):
        CODE12_REGIME_MASKS[f"event_probability_q{quartile}"] = (
            (CODE12_EVENT_RISK > lower) & (CODE12_EVENT_RISK <= upper)
        )
        lower = upper

CODE12_REGIME_ORACLE_ROWS = []
for expert_index in range(ENSEMBLE_SIZE):
    expert = CODE12_ALL_MEMBER_STACK[expert_index]
    for regime, mask in CODE12_REGIME_MASKS.items():
        count = int(mask.sum())
        if count == 0:
            continue
        base_sse = float(np.square(CODE12_P8 - CODE12_TARGET)[mask].sum())
        expert_sse = float(np.square(expert - CODE12_TARGET)[mask].sum())
        CODE12_REGIME_ORACLE_ROWS.append({
            "expert_index": expert_index, "expert": MEMBER_LABELS[expert_index],
            "regime": regime, "count": count,
            "code8_rmse": float(np.sqrt(base_sse / count)),
            "expert_rmse": float(np.sqrt(expert_sse / count)),
            "delta_rmse": float(np.sqrt(expert_sse / count) - np.sqrt(base_sse / count)),
            "sse_gain": base_sse - expert_sse,
        })
CODE12_REGIME_ORACLE_TABLE = pd.DataFrame(CODE12_REGIME_ORACLE_ROWS)
print("\nREGIME ORACLE TABLE")
print(CODE12_REGIME_ORACLE_TABLE.to_string(
    index=False, float_format=lambda value: f"{value:.6f}"
))


In [ ]:
# ============================================================
# Three-round sparse routing with nested deterministic 5-fold meta selection
# ============================================================
CODE12_FOLD_IDS = np.asarray(CODE7_FOLD_IDS, dtype=np.int64)
assert set(CODE12_FOLD_IDS.tolist()) == set(range(5))
CODE12_BASE_FOLD_RMSE = np.asarray([
    code12_rmse(
        CODE12_TARGET[CODE12_FOLD_IDS == fold],
        CODE12_P8[CODE12_FOLD_IDS == fold],
    )
    for fold in range(5)
])
CODE12_HORIZON_GROUPS = {
    "all": (0, 12), "short": (0, 4), "middle": (4, 8), "long": (8, 12),
}
CODE12_META_ORACLE_ELIGIBILITY = {}
for heldout_fold in range(5):
    fit_mask = CODE12_FOLD_IDS != heldout_fold
    fit_target = CODE12_TARGET[fit_mask]
    fit_base = CODE12_P8[fit_mask]
    fit_base_rmse = code12_rmse(fit_target, fit_base)
    for expert_index in CODE12_ROUTING_INDICES:
        fit_expert = CODE12_ALL_MEMBER_STACK[expert_index, fit_mask]
        base_squared = np.square(fit_target - fit_base)
        expert_squared = np.square(fit_target - fit_expert)
        element_prediction = np.where(
            expert_squared < base_squared, fit_expert, fit_base
        )
        sample_mask = expert_squared.sum(axis=1) < base_squared.sum(axis=1)
        sample_prediction = np.where(sample_mask[:, None], fit_expert, fit_base)
        element_gain = fit_base_rmse - code12_rmse(fit_target, element_prediction)
        sample_gain = fit_base_rmse - code12_rmse(fit_target, sample_prediction)
        CODE12_META_ORACLE_ELIGIBILITY[(heldout_fold, expert_index)] = bool(
            sample_gain >= CODE12_ORACLE_SAMPLE_GATE
            or element_gain >= CODE12_ORACLE_ELEMENT_GATE
        )


def code12_meta_oracle_allows(config, heldout_fold):
    return all(
        CODE12_META_ORACLE_ELIGIBILITY[
            (heldout_fold, int(route["expert_index"]))
        ]
        for route in config["routes"]
    )


def code12_config_key(config):
    return json.dumps(config, sort_keys=True, separators=(",", ":"))


def code12_horizon_mask(group, rows):
    start, stop = CODE12_HORIZON_GROUPS[group]
    mask = np.zeros((rows, TARGET_HORIZONS), dtype=bool)
    mask[:, start:stop] = True
    return mask


def code12_freeze_route(expert_index, score_kind, horizon_group, coverage, weight):
    score = np.asarray(CODE12_SCORE_ARRAYS[score_kind], dtype=np.float64)
    allowed = code12_horizon_mask(horizon_group, len(score))
    allowed_values = score[allowed]
    desired_count = min(
        int(round(score.size * float(coverage))), len(allowed_values)
    )
    desired_count = max(desired_count, 1)
    threshold = float(np.partition(allowed_values, len(allowed_values) - desired_count)[
        len(allowed_values) - desired_count
    ])
    return {
        "expert_index": int(expert_index), "expert": MEMBER_LABELS[expert_index],
        "score_kind": score_kind, "horizon_group": horizon_group,
        "coverage_requested": float(coverage), "threshold": threshold,
        "weight": float(weight),
    }


def code12_route_mask(route, score_arrays):
    score = np.asarray(score_arrays[route["score_kind"]], dtype=np.float64)
    allowed = code12_horizon_mask(route["horizon_group"], len(score))
    # Strict threshold prevents tied low-confidence scores from expanding coverage.
    return allowed & (score > float(route["threshold"]))


def code12_apply_routes(config, base_prediction, expert_predictions, score_arrays):
    prediction = np.asarray(base_prediction, dtype=np.float64).copy()
    occupied = np.zeros_like(prediction, dtype=bool)
    for route in config["routes"]:
        route_mask = code12_route_mask(route, score_arrays) & ~occupied
        expert = np.asarray(expert_predictions[int(route["expert_index"])], dtype=np.float64)
        prediction += (
            float(route["weight"]) * route_mask * (expert - base_prediction)
        )
        occupied |= route_mask
    return prediction, occupied


def code12_evaluate_config(config):
    prediction, mask = code12_apply_routes(
        config, CODE12_P8, {
            index: CODE12_ALL_MEMBER_STACK[index]
            for index in CODE12_ROUTING_INDICES
        }, CODE12_SCORE_ARRAYS,
    )
    sample_sse = np.square(prediction - CODE12_TARGET).sum(axis=1)
    fold_rmse = np.asarray([
        code12_rmse(
            CODE12_TARGET[CODE12_FOLD_IDS == fold],
            prediction[CODE12_FOLD_IDS == fold],
        )
        for fold in range(5)
    ])
    fold_delta = fold_rmse - CODE12_BASE_FOLD_RMSE
    return {
        "config": config, "key": code12_config_key(config),
        "full_rmse": code12_rmse(CODE12_TARGET, prediction),
        "fold_rmse": fold_rmse, "fold_wins": int(np.sum(fold_delta < 0.0)),
        "worst_fold_degradation": float(fold_delta.max()),
        "coverage": float(mask.mean()), "sample_sse": sample_sse.astype(np.float64),
    }


def code12_meta_select(round_name, records):
    if not records:
        return None
    records = sorted(records, key=lambda row: row["key"])
    oof_prediction = np.empty_like(CODE12_TARGET)
    chosen = []
    for heldout_fold in range(5):
        fit_mask = CODE12_FOLD_IDS != heldout_fold
        fit_records = [
            row for row in records
            if code12_meta_oracle_allows(row["config"], heldout_fold)
        ]
        if not fit_records:
            heldout_mask = CODE12_FOLD_IDS == heldout_fold
            oof_prediction[heldout_mask] = CODE12_P8[heldout_mask]
            chosen.append(None)
            continue
        best = min(fit_records, key=lambda row: (
            float(np.sqrt(row["sample_sse"][fit_mask].sum() / (fit_mask.sum() * 12))),
            -row["fold_wins"], row["worst_fold_degradation"],
            row["coverage"], row["key"],
        ))
        heldout_mask = CODE12_FOLD_IDS == heldout_fold
        prediction, _ = code12_apply_routes(
            best["config"], CODE12_P8,
            {index: CODE12_ALL_MEMBER_STACK[index] for index in CODE12_ROUTING_INDICES},
            CODE12_SCORE_ARRAYS,
        )
        oof_prediction[heldout_mask] = prediction[heldout_mask]
        chosen.append(best)
    counts = {}
    for row in chosen:
        if row is None:
            continue
        counts[row["key"]] = counts.get(row["key"], 0) + 1
    if counts:
        mode_count = max(counts.values())
        frozen = min(
            (row for row in records if counts.get(row["key"], 0) == mode_count),
            key=lambda row: (
                row["full_rmse"], -row["fold_wins"],
                row["worst_fold_degradation"], row["coverage"], row["key"],
            ),
        )
    else:
        mode_count = 0
        frozen = min(records, key=lambda row: (
            row["full_rmse"], -row["fold_wins"],
            row["worst_fold_degradation"], row["coverage"], row["key"],
        ))
    frozen_prediction, _ = code12_apply_routes(
        frozen["config"], CODE12_P8,
        {index: CODE12_ALL_MEMBER_STACK[index] for index in CODE12_ROUTING_INDICES},
        CODE12_SCORE_ARRAYS,
    )
    return {
        "round": round_name, "config_count": len(records), "chosen": chosen,
        "frozen_record": frozen, "prediction": frozen_prediction,
        "oof_prediction": oof_prediction,
        "oof_meta_rmse": code12_rmse(CODE12_TARGET, oof_prediction),
        "parameter_mode_fraction": mode_count / 5.0,
    }


CODE12_ROUND1_RECORDS = []
CODE12_ROUND2_RECORDS = []
CODE12_ROUND3_RECORDS = []
CODE12_TOP20_ROUND1 = []
CODE12_META_RESULTS = []
if CODE12_ORACLE_HEADROOM_OK:
    round1_configs = {}
    for expert_index in CODE12_ROUTING_INDICES:
        for score_kind in sorted(CODE12_SCORE_ARRAYS):
            for horizon_group in CODE12_HORIZON_GROUPS:
                for coverage in (0.05, 0.10, 0.20):
                    for weight in (0.025, 0.05, 0.10, 0.15, 0.25):
                        route = code12_freeze_route(
                            expert_index, score_kind, horizon_group, coverage, weight
                        )
                        config = {"kind": "single", "routes": [route]}
                        round1_configs[code12_config_key(config)] = config
    assert len(round1_configs) <= CODE12_MAX_ROUND1
    CODE12_ROUND1_RECORDS = [
        code12_evaluate_config(config) for config in round1_configs.values()
    ]
    CODE12_TOP20_ROUND1 = sorted(CODE12_ROUND1_RECORDS, key=lambda row: (
        row["full_rmse"], -row["fold_wins"], row["worst_fold_degradation"],
        row["coverage"], row["key"],
    ))[:20]
    print("\nROUND 1 TOP 20")
    print(pd.DataFrame([{
        "rank": rank,
        "expert": row["config"]["routes"][0]["expert"],
        "score": row["config"]["routes"][0]["score_kind"],
        "horizon": row["config"]["routes"][0]["horizon_group"],
        "weight": row["config"]["routes"][0]["weight"],
        "coverage": row["coverage"], "full_rmse": row["full_rmse"],
        "fold_wins": row["fold_wins"],
        "worst_fold": row["worst_fold_degradation"],
    } for rank, row in enumerate(CODE12_TOP20_ROUND1, start=1)]).to_string(
        index=False, float_format=lambda value: f"{value:.6f}"
    ))
    CODE12_META_RESULTS.append(
        code12_meta_select("round1", CODE12_ROUND1_RECORDS)
    )

    round2_configs = {}
    for parent in CODE12_TOP20_ROUND1[:12]:
        route0 = parent["config"]["routes"][0]
        base_coverage = float(route0["coverage_requested"])
        coverage_grid = sorted(set(
            min(0.25, max(0.05, base_coverage + offset))
            for offset in (-0.05, -0.025, 0.0, 0.025, 0.05)
        ))
        for coverage in coverage_grid:
            for weight in (0.025, 0.05, 0.075, 0.10, 0.15, 0.20, 0.25, 0.30):
                route = code12_freeze_route(
                    route0["expert_index"], route0["score_kind"],
                    route0["horizon_group"], coverage, weight,
                )
                config = {"kind": "single_refined", "routes": [route]}
                round2_configs[code12_config_key(config)] = config
    if len(round2_configs) > CODE12_MAX_ROUND2:
        round2_configs = dict(list(sorted(round2_configs.items()))[:CODE12_MAX_ROUND2])
    CODE12_ROUND2_RECORDS = [
        code12_evaluate_config(config) for config in round2_configs.values()
    ]
    CODE12_META_RESULTS.append(
        code12_meta_select("round2", CODE12_ROUND2_RECORDS)
    )

    pair_pool = sorted(CODE12_ROUND2_RECORDS, key=lambda row: (
        row["full_rmse"], -row["fold_wins"], row["worst_fold_degradation"],
        row["coverage"], row["key"],
    ))[:12]
    round3_configs = {}
    for left_index in range(len(pair_pool)):
        for right_index in range(left_index + 1, len(pair_pool)):
            left_route = pair_pool[left_index]["config"]["routes"][0]
            right_route = pair_pool[right_index]["config"]["routes"][0]
            if left_route["expert_index"] == right_route["expert_index"]:
                continue
            # Better individual route gets first claim; overlap is never double-blended.
            config = {
                "kind": "two_expert_nonoverlap",
                "routes": [copy.deepcopy(left_route), copy.deepcopy(right_route)],
            }
            round3_configs[code12_config_key(config)] = config
    assert len(round3_configs) <= CODE12_MAX_ROUND3
    CODE12_ROUND3_RECORDS = [
        record for record in (
            code12_evaluate_config(config) for config in round3_configs.values()
        )
        if record["coverage"] <= 0.250001
    ]
    if CODE12_ROUND3_RECORDS:
        CODE12_META_RESULTS.append(
            code12_meta_select("round3", CODE12_ROUND3_RECORDS)
        )

CODE12_META_RESULTS = [result for result in CODE12_META_RESULTS if result is not None]

def code12_last_shot_passes(result):
    if result is None:
        return False
    frozen = result["frozen_record"]
    full_oracle_pass = all(
        CODE12_ORACLE_ROWS[int(route["expert_index"])]["passes_oracle_potential"]
        for route in frozen["config"]["routes"]
    )
    return bool(
        full_oracle_pass
        and CODE12_LIVE_CODE8 - frozen["full_rmse"] >= CODE12_LAST_SHOT_FULL_GAIN
        and CODE12_LIVE_CODE8 - result["oof_meta_rmse"] >= CODE12_LAST_SHOT_OOF_GAIN
        and frozen["fold_wins"] >= CODE12_RECOMMENDED_FOLD_WINS
        and frozen["worst_fold_degradation"] < CODE12_MAX_FOLD_DEGRADATION
    )

eligible_results = [
    result for result in CODE12_META_RESULTS if code12_last_shot_passes(result)
]
if eligible_results:
    CODE12_BEST_META = min(eligible_results, key=lambda result: (
        result["oof_meta_rmse"],
        -result["frozen_record"]["fold_wins"],
        result["frozen_record"]["worst_fold_degradation"],
        result["frozen_record"]["full_rmse"],
        result["frozen_record"]["coverage"],
        -result["parameter_mode_fraction"], result["round"],
    ))
    CODE12_LAST_SHOT = True
elif CODE12_META_RESULTS:
    CODE12_BEST_META = min(CODE12_META_RESULTS, key=lambda result: (
        result["oof_meta_rmse"],
        -result["frozen_record"]["fold_wins"],
        result["frozen_record"]["worst_fold_degradation"],
        result["frozen_record"]["full_rmse"],
        result["frozen_record"]["coverage"],
        -result["parameter_mode_fraction"], result["round"],
    ))
    CODE12_LAST_SHOT = False
else:
    CODE12_BEST_META = None
    CODE12_LAST_SHOT = False

CODE12_BEST_EVALUATED_PREDICTION = (
    CODE12_BEST_META["prediction"] if CODE12_BEST_META is not None else CODE12_P8
)
CODE12_FINAL_VAL_PREDICTION = (
    CODE12_BEST_EVALUATED_PREDICTION if CODE12_LAST_SHOT else CODE12_P8
)
CODE12_FINAL_CONFIG = (
    CODE12_BEST_META["frozen_record"]["config"] if CODE12_LAST_SHOT else None
)
print("\nROUND COUNTS:", {
    "round1": len(CODE12_ROUND1_RECORDS),
    "round2": len(CODE12_ROUND2_RECORDS),
    "round3": len(CODE12_ROUND3_RECORDS),
})
for result in CODE12_META_RESULTS:
    frozen = result["frozen_record"]
    print(
        result["round"], "full=", f"{frozen['full_rmse']:.6f}",
        "OOF=", f"{result['oof_meta_rmse']:.6f}",
        "folds=", f"{frozen['fold_wins']}/5",
        "worst=", f"{frozen['worst_fold_degradation']:+.6f}",
        "coverage=", f"{frozen['coverage']:.4f}",
    )
print("LAST_SHOT:", "YES" if CODE12_LAST_SHOT else "NO — CODE8 fallback")


In [ ]:
# CODE13 defers CODE12 artifact publication until local refinement is complete.
# All CODE12 predictions, states, search records, and diagnostics remain unchanged.
CODE13_DEFERRED_CODE12_ARTIFACT_PUBLICATION = True
assert CODE12_REPRODUCTION_OK
print("CODE12 artifact publication deferred: CODE13 will publish the best robust local route.")


## CODE12 constraints

- CODE8 is unchanged for every element unless the strict last-shot gate passes.
- Oracle target masks are diagnostic only and are never serialized or applied to Test.
- Event-gate fitting and checkpoint selection use Competition Train only.
- Official Validation is used only for oracle diagnosis and discrete routing/model selection.
- Test is touched only after every route, threshold, weight, and expert has been frozen.
- No external data, pretrained weights, CODE10/11 predictions, pseudo-labels, or leaderboard fitting.


# CODE 13 — Final local refinement of the only robust winner

This section runs **after** the untouched CODE8/CODE12 training and prediction path.
It performs no backbone training and creates no new model family.  It first reproduces
the actual CODE12 CH Topology sparse route, then refines only its target-free hard gate.

Default prediction is always frozen CODE8.  Only gated elements are moved toward an
already-trained CODE8 CH Topology expert; optional HSS routing can act only on elements
not claimed by CH Topology.  Official Validation is used only for model/meta-selection.


In [ ]:
# ============================================================
# CODE13 reproduction gate and Stage A: CH Topology local search
# ============================================================
CODE13_HISTORICAL_CODE8 = 62.903266
CODE13_HISTORICAL_CODE12_ROUTE = 62.820879
CODE13_REPRODUCTION_TOLERANCE = 0.002
CODE13_CH_INDEX = 4
CODE13_HSS_INDEX = 6
assert MEMBER_LABELS[CODE13_CH_INDEX] == "CH Topology"
assert MEMBER_LABELS[CODE13_HSS_INDEX] == "HSS Peak"
assert abs(CODE12_LIVE_CODE8 - CODE13_HISTORICAL_CODE8) <= CODE13_REPRODUCTION_TOLERANCE
assert CODE12_GATE_INFORMATIVE, "CODE12 event gate is required for the proven route."
assert CODE12_EVENT_RISK.shape == CODE12_TARGET.shape == (1199, 12)
assert CODE12_U.shape == CODE12_TARGET.shape


def code13_reference_record():
    matches = []
    for row in CODE12_ROUND2_RECORDS:
        routes = row["config"].get("routes", [])
        if len(routes) != 1:
            continue
        route = routes[0]
        if (
            int(route["expert_index"]) == CODE13_CH_INDEX
            and route["score_kind"] == "event_uncertainty"
            and route["horizon_group"] == "all"
            and abs(float(route["coverage_requested"]) - 0.05) < 1e-12
            and abs(float(route["weight"]) - 0.30) < 1e-12
        ):
            matches.append(row)
    assert matches, "CODE12 Round2 CH/event_uncertainty/5%/0.30 record missing."
    return min(matches, key=lambda row: row["full_rmse"])


CODE13_REFERENCE_RECORD = code13_reference_record()
CODE13_REFERENCE_CONFIG = copy.deepcopy(CODE13_REFERENCE_RECORD["config"])
CODE13_REFERENCE_PREDICTION, CODE13_REFERENCE_MASK = code12_apply_routes(
    CODE13_REFERENCE_CONFIG,
    CODE12_P8,
    {CODE13_CH_INDEX: CODE12_ALL_MEMBER_STACK[CODE13_CH_INDEX]},
    CODE12_SCORE_ARRAYS,
)
CODE13_LIVE_CODE12_ROUTE = code12_rmse(CODE12_TARGET, CODE13_REFERENCE_PREDICTION)
assert abs(
    CODE13_LIVE_CODE12_ROUTE - CODE13_HISTORICAL_CODE12_ROUTE
) <= CODE13_REPRODUCTION_TOLERANCE, (
    "CODE12 ROUTING REPRODUCTION FAILED",
    CODE13_LIVE_CODE12_ROUTE,
    CODE13_HISTORICAL_CODE12_ROUTE,
)
assert np.isclose(
    CODE13_LIVE_CODE12_ROUTE, CODE13_REFERENCE_RECORD["full_rmse"], atol=1e-10
)

CODE13_BASE_FOLD_RMSE = np.asarray([
    code12_rmse(
        CODE12_TARGET[CODE12_FOLD_IDS == fold],
        CODE13_REFERENCE_PREDICTION[CODE12_FOLD_IDS == fold],
    )
    for fold in range(5)
])
CODE13_REFERENCE_FOLD_RMSE = np.asarray(CODE13_REFERENCE_RECORD["fold_rmse"])
CODE13_REFERENCE_FOLD_DELTA_VS_CODE8 = (
    CODE13_REFERENCE_FOLD_RMSE - CODE12_BASE_FOLD_RMSE
)
assert int(np.sum(CODE13_REFERENCE_FOLD_DELTA_VS_CODE8 < 0.0)) == 4
assert float(CODE13_REFERENCE_FOLD_DELTA_VS_CODE8.max()) <= 0.06

# Requested threshold values and coverage values induce the same rank ordering.
# Exact top-k coverage is deterministic and avoids duplicate threshold candidates.
CODE13_COVERAGE_GRID = (
    0.020, 0.025, 0.030, 0.035, 0.040, 0.045, 0.050,
    0.055, 0.060, 0.065, 0.070, 0.080, 0.090, 0.100,
)
CODE13_WEIGHT_GRID = (
    0.20, 0.225, 0.25, 0.275, 0.30, 0.325, 0.35,
    0.375, 0.40, 0.45, 0.50, 0.55, 0.60,
)
CODE13_HORIZON_GROUPS = {
    "all": (0, 12),
    "mid_long": (4, 12),       # 30h--72h
    "long_48": (7, 12),        # 48h--72h
    "short_mid": (0, 8),       # 6h--48h
}
CODE13_SCORE_ARRAYS = {
    "event_uncertainty": np.asarray(
        CODE12_EVENT_RISK * CODE12_U, dtype=np.float64
    ),
    "event_extreme": np.asarray(
        CODE12_VAL_GATE_PROBABILITIES[:, 12:], dtype=np.float64
    ),
}
assert np.allclose(
    CODE13_SCORE_ARRAYS["event_uncertainty"],
    CODE12_SCORE_ARRAYS["event_uncertainty"],
)


def code13_config_key(config):
    return json.dumps(config, sort_keys=True, separators=(",", ":"))


def code13_horizon_mask(group, rows):
    start, stop = CODE13_HORIZON_GROUPS[group]
    mask = np.zeros((rows, TARGET_HORIZONS), dtype=bool)
    mask[:, start:stop] = True
    return mask


def code13_freeze_route(expert_index, score_kind, horizon_group, coverage, weight):
    score = np.asarray(CODE13_SCORE_ARRAYS[score_kind], dtype=np.float64)
    allowed = code13_horizon_mask(horizon_group, len(score))
    allowed_values = score[allowed]
    desired_count = min(
        max(int(round(score.size * float(coverage))), 1), len(allowed_values)
    )
    kth = len(allowed_values) - desired_count
    threshold = float(np.partition(allowed_values, kth)[kth])
    return {
        "expert_index": int(expert_index),
        "expert": MEMBER_LABELS[int(expert_index)],
        "score_kind": str(score_kind),
        "horizon_group": str(horizon_group),
        "coverage_requested": float(coverage),
        "threshold": threshold,
        "weight": float(weight),
    }


def code13_route_mask(route, score_arrays):
    score = np.asarray(score_arrays[route["score_kind"]], dtype=np.float64)
    allowed = code13_horizon_mask(route["horizon_group"], len(score))
    return allowed & (score > float(route["threshold"]))


def code13_apply_routes(config, base_prediction, expert_predictions, score_arrays):
    prediction = np.asarray(base_prediction, dtype=np.float64).copy()
    occupied = np.zeros_like(prediction, dtype=bool)
    for route in config["routes"]:
        route_mask = code13_route_mask(route, score_arrays) & ~occupied
        expert_prediction = np.asarray(
            expert_predictions[int(route["expert_index"])], dtype=np.float64
        )
        prediction += float(route["weight"]) * route_mask * (
            expert_prediction - base_prediction
        )
        occupied |= route_mask
    return prediction, occupied


CODE13_EXPERT_PREDICTIONS = {
    CODE13_CH_INDEX: CODE12_ALL_MEMBER_STACK[CODE13_CH_INDEX],
    CODE13_HSS_INDEX: CODE12_ALL_MEMBER_STACK[CODE13_HSS_INDEX],
}


def code13_evaluate_config(config):
    prediction, changed = code13_apply_routes(
        config, CODE12_P8, CODE13_EXPERT_PREDICTIONS, CODE13_SCORE_ARRAYS
    )
    sample_sse = np.square(prediction - CODE12_TARGET).sum(axis=1)
    fold_rmse = np.asarray([
        code12_rmse(
            CODE12_TARGET[CODE12_FOLD_IDS == fold],
            prediction[CODE12_FOLD_IDS == fold],
        )
        for fold in range(5)
    ])
    fold_delta_code8 = fold_rmse - CODE12_BASE_FOLD_RMSE
    fold_delta_reference = fold_rmse - CODE13_BASE_FOLD_RMSE
    return {
        "config": config,
        "key": code13_config_key(config),
        "full_rmse": code12_rmse(CODE12_TARGET, prediction),
        "fold_rmse": fold_rmse,
        "fold_wins": int(np.sum(fold_delta_code8 < 0.0)),
        "fold_wins_vs_reference": int(np.sum(fold_delta_reference < 0.0)),
        "worst_fold_degradation": float(fold_delta_code8.max()),
        "worst_fold_vs_reference": float(fold_delta_reference.max()),
        "coverage": float(changed.mean()),
        "sample_sse": sample_sse.astype(np.float64),
    }


def code13_meta_select(stage_name, records):
    """Nested deterministic 5-fold selection; held-out targets never select params."""
    if not records:
        return None
    records = sorted(records, key=lambda row: row["key"])
    oof_prediction = np.empty_like(CODE12_TARGET)
    chosen = []
    fit_score_by_key = {row["key"]: [] for row in records}
    for heldout_fold in range(5):
        fit_mask = CODE12_FOLD_IDS != heldout_fold
        fit_fold_ids = [fold for fold in range(5) if fold != heldout_fold]
        fit_records = [
            row for row in records
            if code12_meta_oracle_allows(row["config"], heldout_fold)
        ]
        assert fit_records, (stage_name, heldout_fold)
        def fit_key(row):
            fit_rmse = float(np.sqrt(
                row["sample_sse"][fit_mask].sum() / (fit_mask.sum() * TARGET_HORIZONS)
            ))
            fit_deltas = row["fold_rmse"][fit_fold_ids] - CODE12_BASE_FOLD_RMSE[fit_fold_ids]
            fit_wins = int(np.sum(fit_deltas < 0.0))
            fit_worst = float(fit_deltas.max())
            return (fit_rmse, -fit_wins, fit_worst, row["coverage"], row["key"])
        best = min(fit_records, key=fit_key)
        fit_score_by_key[best["key"]].append(fit_key(best)[0])
        heldout_mask = CODE12_FOLD_IDS == heldout_fold
        prediction, _ = code13_apply_routes(
            best["config"], CODE12_P8, CODE13_EXPERT_PREDICTIONS,
            CODE13_SCORE_ARRAYS,
        )
        oof_prediction[heldout_mask] = prediction[heldout_mask]
        chosen.append(best)
    counts = {}
    for row in chosen:
        counts[row["key"]] = counts.get(row["key"], 0) + 1
    mode_count = max(counts.values())
    frozen_candidates = [
        row for row in records if counts.get(row["key"], 0) == mode_count
    ]
    frozen = min(frozen_candidates, key=lambda row: (
        np.mean(fit_score_by_key[row["key"]]),
        row["full_rmse"], -row["fold_wins"],
        row["worst_fold_degradation"], row["coverage"], row["key"],
    ))
    frozen_prediction, _ = code13_apply_routes(
        frozen["config"], CODE12_P8, CODE13_EXPERT_PREDICTIONS,
        CODE13_SCORE_ARRAYS,
    )
    return {
        "stage": stage_name,
        "config_count": len(records),
        "chosen": chosen,
        "frozen_record": frozen,
        "prediction": frozen_prediction,
        "oof_prediction": oof_prediction,
        "oof_meta_rmse": code12_rmse(CODE12_TARGET, oof_prediction),
        "parameter_mode_fraction": mode_count / 5.0,
    }


def code13_fixed_reference_meta():
    record = code13_evaluate_config(CODE13_REFERENCE_CONFIG)
    return {
        "stage": "code12_reference",
        "config_count": 1,
        "chosen": [record] * 5,
        "frozen_record": record,
        "prediction": CODE13_REFERENCE_PREDICTION,
        "oof_prediction": CODE13_REFERENCE_PREDICTION.copy(),
        "oof_meta_rmse": CODE13_LIVE_CODE12_ROUTE,
        "parameter_mode_fraction": 1.0,
    }


CODE13_REFERENCE_META = code13_fixed_reference_meta()
stage_a_configs = {}
for horizon_group in CODE13_HORIZON_GROUPS:
    for coverage in CODE13_COVERAGE_GRID:
        for weight in CODE13_WEIGHT_GRID:
            route = code13_freeze_route(
                CODE13_CH_INDEX, "event_uncertainty", horizon_group,
                coverage, weight,
            )
            config = {"kind": "code13_ch_local", "routes": [route]}
            stage_a_configs[code13_config_key(config)] = config
assert len(stage_a_configs) <= 1000
CODE13_STAGE_A_RECORDS = [
    code13_evaluate_config(config) for config in stage_a_configs.values()
]
CODE13_STAGE_A_META = code13_meta_select("stage_a", CODE13_STAGE_A_RECORDS)
CODE13_STAGE_A_TOP10 = sorted(CODE13_STAGE_A_RECORDS, key=lambda row: (
    row["full_rmse"], -row["fold_wins"], row["worst_fold_degradation"],
    row["coverage"], row["key"],
))[:10]

print("\nCODE13 REPRODUCTION")
print("CODE8:", f"{CODE12_LIVE_CODE8:.6f}")
print("CODE12 routing:", f"{CODE13_LIVE_CODE12_ROUTE:.6f}")
print("CODE12 coverage:", f"{CODE13_REFERENCE_MASK.mean():.6f}")
print("CODE12 threshold:", CODE13_REFERENCE_CONFIG["routes"][0]["threshold"])
print("CODE12 weight:", CODE13_REFERENCE_CONFIG["routes"][0]["weight"])
print("STAGE A candidates:", len(CODE13_STAGE_A_RECORDS))
print("STAGE A meta OOF:", f"{CODE13_STAGE_A_META['oof_meta_rmse']:.6f}")
print("STAGE A frozen:", CODE13_STAGE_A_META["frozen_record"]["config"])


In [ ]:
# ============================================================
# Stage B: local score refinement; Stage C: optional non-overlap HSS route
# ============================================================
CODE13_EVENT_COEFFICIENTS = (0.6, 0.7, 0.8, 0.9, 1.0)
CODE13_UNCERTAINTY_COEFFICIENTS = (0.2, 0.3, 0.4, 0.5, 0.6)
CODE13_DELTA_COEFFICIENTS = (0.0, 0.1, 0.2, 0.3)
CODE13_SCORE_DEFINITIONS = {
    "event_uncertainty": {
        "kind": "event_uncertainty", "event": 1.0,
        "uncertainty": 1.0, "delta": 0.0,
    },
    "event_extreme": {"kind": "event_extreme"},
}


def code13_refined_score(event_coefficient, uncertainty_coefficient, delta_coefficient):
    total = event_coefficient + uncertainty_coefficient + delta_coefficient
    local_mix = (
        event_coefficient * CODE12_EVENT_RISK
        + uncertainty_coefficient * CODE12_U
        + delta_coefficient * CODE12_D
    ) / total
    # Preserve the proven event*risk ranking as the dominant signal and make only
    # a bounded local rank perturbation.  Every term is target-free at inference.
    return (CODE12_EVENT_RISK * CODE12_U) * (0.875 + 0.25 * local_mix)


for event_coefficient in CODE13_EVENT_COEFFICIENTS:
    for uncertainty_coefficient in CODE13_UNCERTAINTY_COEFFICIENTS:
        for delta_coefficient in CODE13_DELTA_COEFFICIENTS:
            score_kind = (
                f"event_uncertainty_refined_e{event_coefficient:.1f}"
                f"_u{uncertainty_coefficient:.1f}_d{delta_coefficient:.1f}"
            )
            CODE13_SCORE_ARRAYS[score_kind] = code13_refined_score(
                event_coefficient, uncertainty_coefficient, delta_coefficient
            )
            CODE13_SCORE_DEFINITIONS[score_kind] = {
                "kind": "event_uncertainty_refined",
                "event": float(event_coefficient),
                "uncertainty": float(uncertainty_coefficient),
                "delta": float(delta_coefficient),
                "base_product_weight": 1.0,
                "local_mix_offset": 0.875,
                "local_mix_scale": 0.25,
            }

# Ten neighborhoods are fixed around the historical CODE12 route before looking at
# CODE13 Validation outcomes.  This keeps Stage-B OOF folds honest: a held-out fold
# does not participate even indirectly in creation of the refinement candidate set.
CODE13_STAGE_B_PARENT_GRID = tuple(
    (coverage, weight)
    for coverage in (0.040, 0.045, 0.050, 0.055, 0.060)
    for weight in (0.30, 0.35)
)
assert len(CODE13_STAGE_B_PARENT_GRID) == 10
stage_b_configs = {}
refined_score_kinds = sorted(
    key for key in CODE13_SCORE_ARRAYS if key.startswith("event_uncertainty_refined_")
)
for parent_coverage, parent_weight in CODE13_STAGE_B_PARENT_GRID:
    for score_kind in refined_score_kinds:
        route = code13_freeze_route(
            CODE13_CH_INDEX, score_kind, "all",
            parent_coverage, parent_weight,
        )
        config = {"kind": "code13_score_refined", "routes": [route]}
        stage_b_configs[code13_config_key(config)] = config
assert len(stage_b_configs) <= 1200
CODE13_STAGE_B_RECORDS = [
    code13_evaluate_config(config) for config in stage_b_configs.values()
]
CODE13_STAGE_B_META = code13_meta_select("stage_b", CODE13_STAGE_B_RECORDS)


def code13_robust_meta(result):
    if result is None:
        return False
    frozen = result["frozen_record"]
    return bool(
        frozen["fold_wins"] >= 4
        and frozen["worst_fold_degradation"] <= 0.06
        and frozen["full_rmse"] <= CODE13_LIVE_CODE12_ROUTE + 1e-12
    )


pre_hss_candidates = [
    result for result in (
        CODE13_REFERENCE_META, CODE13_STAGE_A_META, CODE13_STAGE_B_META
    ) if code13_robust_meta(result)
]
CODE13_PRE_HSS_META = min(pre_hss_candidates, key=lambda result: (
    result["oof_meta_rmse"],
    -result["frozen_record"]["fold_wins"],
    result["frozen_record"]["full_rmse"],
    result["frozen_record"]["worst_fold_degradation"],
    result["frozen_record"]["coverage"],
    -result["parameter_mode_fraction"],
))

# Optional Stage C.  CH always gets first claim.  HSS can affect only the remaining
# elements, with small requested coverage and no overlapping expert application.
CODE13_STAGE_C_RECORDS = []
CODE13_STAGE_C_META = None
if code12_meta_oracle_allows(CODE13_PRE_HSS_META["frozen_record"]["config"], 0):
    ch_route = copy.deepcopy(
        CODE13_PRE_HSS_META["frozen_record"]["config"]["routes"][0]
    )
    stage_c_configs = {}
    for score_kind in ("event_uncertainty", "event_extreme"):
        for coverage in (0.02, 0.03, 0.05):
            for weight in (0.05, 0.10, 0.15, 0.20, 0.25):
                hss_route = code13_freeze_route(
                    CODE13_HSS_INDEX, score_kind, "all", coverage, weight
                )
                config = {
                    "kind": "code13_ch_then_hss_nonoverlap",
                    "routes": [copy.deepcopy(ch_route), hss_route],
                }
                stage_c_configs[code13_config_key(config)] = config
    raw_stage_c = [
        code13_evaluate_config(config) for config in stage_c_configs.values()
    ]
    CODE13_STAGE_C_RECORDS = [
        row for row in raw_stage_c if row["coverage"] <= 0.100001
    ]
    if CODE13_STAGE_C_RECORDS:
        CODE13_STAGE_C_META = code13_meta_select(
            "stage_c", CODE13_STAGE_C_RECORDS
        )

CODE13_ALL_META_RESULTS = [
    result for result in (
        CODE13_REFERENCE_META, CODE13_STAGE_A_META,
        CODE13_STAGE_B_META, CODE13_STAGE_C_META,
    ) if result is not None
]
CODE13_ELIGIBLE_META_RESULTS = [
    result for result in CODE13_ALL_META_RESULTS if code13_robust_meta(result)
]
assert CODE13_ELIGIBLE_META_RESULTS
CODE13_FINAL_META = min(CODE13_ELIGIBLE_META_RESULTS, key=lambda result: (
    result["oof_meta_rmse"],
    -result["frozen_record"]["fold_wins"],
    result["frozen_record"]["full_rmse"],
    result["frozen_record"]["worst_fold_degradation"],
    result["frozen_record"]["coverage"],
    -result["parameter_mode_fraction"],
))
CODE13_FINAL_RECORD = CODE13_FINAL_META["frozen_record"]
CODE13_FINAL_CONFIG = copy.deepcopy(CODE13_FINAL_RECORD["config"])
CODE13_FINAL_VAL_PREDICTION, CODE13_FINAL_MASK = code13_apply_routes(
    CODE13_FINAL_CONFIG, CODE12_P8, CODE13_EXPERT_PREDICTIONS,
    CODE13_SCORE_ARRAYS,
)
assert np.isclose(
    code12_rmse(CODE12_TARGET, CODE13_FINAL_VAL_PREDICTION),
    CODE13_FINAL_RECORD["full_rmse"], atol=1e-10,
)
CODE13_UPGRADE = bool(
    CODE13_FINAL_RECORD["full_rmse"] < CODE13_LIVE_CODE12_ROUTE - 1e-12
)
CODE13_LAST_SHOT = bool(
    CODE13_FINAL_RECORD["full_rmse"] <= 62.80
    and CODE13_FINAL_META["oof_meta_rmse"] <= 62.82
    and CODE13_FINAL_RECORD["fold_wins"] >= 4
    and CODE13_FINAL_RECORD["worst_fold_degradation"] <= 0.06
)
CODE13_TOTAL_CANDIDATES = (
    len(CODE13_STAGE_A_RECORDS)
    + len(CODE13_STAGE_B_RECORDS)
    + len(CODE13_STAGE_C_RECORDS)
)
assert CODE13_TOTAL_CANDIDATES <= 5000

print("\nSTAGE B candidates:", len(CODE13_STAGE_B_RECORDS))
print("STAGE B meta OOF:", f"{CODE13_STAGE_B_META['oof_meta_rmse']:.6f}")
print("STAGE C candidates:", len(CODE13_STAGE_C_RECORDS))
if CODE13_STAGE_C_META is not None:
    print("STAGE C meta OOF:", f"{CODE13_STAGE_C_META['oof_meta_rmse']:.6f}")
print("FINAL stage:", CODE13_FINAL_META["stage"])
print("FINAL config:", CODE13_FINAL_CONFIG)
print("FINAL Full:", f"{CODE13_FINAL_RECORD['full_rmse']:.6f}")
print("FINAL OOF:", f"{CODE13_FINAL_META['oof_meta_rmse']:.6f}")


In [ ]:
# ============================================================
# Freeze final route -> atomic checkpoint -> disk reload -> Test -> report
# ============================================================
def code13_validate_checkpoint(checkpoint):
    required = {
        "architecture_version", "source_code12_sha256",
        "base_model_state_dicts", "base_variants", "base_labels", "base_seeds",
        "code8_horizon_weights", "code8_final_alpha", "routes",
        "expert_members", "event_gate", "routing_scales", "score_definitions",
        "validation_rmse_km_s", "oof_meta_rmse_km_s", "fold_rmse_km_s",
        "fold_wins", "worst_fold_degradation", "parameter_mode_fraction",
        "routing_validation_coverage", "last_shot", "image_size", "channels",
        "wind_feature_dim", "target_horizons", "decoder_config",
        "coronal_feature_config", "coronal_geometry", "prediction_mode",
        "initialization", "external_data_used", "pretrained_weights_used",
        "train_validation_combined", "test_used_for_selection",
        "public_score_used_for_fitting",
    }
    missing = sorted(required - set(checkpoint))
    assert not missing, missing
    assert checkpoint["architecture_version"] == "code13_local_ch_routing_v1"
    assert checkpoint["source_code12_sha256"] == (
        "1262BCFC657C4B26C13331023D3E04FD30772E2705C887ACDBC31EBFDC8F652E"
    )
    assert len(checkpoint["base_model_state_dicts"]) == 3
    assert tuple(checkpoint["base_labels"]) == (
        "V8 Physics 2", "V8 Wide 1", "V8 Wide 2"
    )
    assert np.asarray(checkpoint["code8_horizon_weights"]).shape == (12, 3)
    assert np.asarray(checkpoint["code8_final_alpha"]).shape == (12,)
    assert 1 <= len(checkpoint["routes"]) <= 2
    assert checkpoint["routes"][0]["expert"] == "CH Topology"
    assert len(checkpoint["expert_members"]) == len({
        int(route["expert_index"]) for route in checkpoint["routes"]
    })
    assert checkpoint["event_gate"] is not None
    assert len(checkpoint["event_gate"]["state_dicts"]) == 3
    assert np.asarray(checkpoint["event_gate"]["feature_mean"]).shape == (471,)
    assert np.asarray(checkpoint["event_gate"]["feature_std"]).shape == (471,)
    assert checkpoint["fold_wins"] >= 4
    assert checkpoint["worst_fold_degradation"] <= 0.06
    assert checkpoint["routing_validation_coverage"] <= 0.100001
    assert checkpoint["external_data_used"] is False
    assert checkpoint["pretrained_weights_used"] is False
    assert checkpoint["train_validation_combined"] is False
    assert checkpoint["test_used_for_selection"] is False
    assert checkpoint["public_score_used_for_fitting"] is False
    return checkpoint


final_routes = copy.deepcopy(CODE13_FINAL_CONFIG["routes"])
final_expert_indices = sorted({
    int(route["expert_index"]) for route in final_routes
})
used_score_definitions = {
    route["score_kind"]: copy.deepcopy(CODE13_SCORE_DEFINITIONS[route["score_kind"]])
    for route in final_routes
}
CODE13_CHECKPOINT = {
    "architecture_version": "code13_local_ch_routing_v1",
    "source_code12_sha256": (
        "1262BCFC657C4B26C13331023D3E04FD30772E2705C887ACDBC31EBFDC8F652E"
    ),
    "base_model_state_dicts": CODE12_BASE_STATES,
    "base_variants": list(CODE12_BASE_VARIANTS),
    "base_labels": list(CODE12_BASE_LABELS),
    "base_seeds": list(CODE12_BASE_SEEDS),
    "code8_horizon_weights": CODE8_FROZEN_HORIZON_WEIGHTS.tolist(),
    "code8_final_alpha": CODE8_FROZEN_FINAL_ALPHA.tolist(),
    "routes": final_routes,
    "expert_members": [{
        "expert_index": index,
        "label": MEMBER_LABELS[index],
        "variant": ENSEMBLE_VARIANTS[index],
        "seed": int(MEMBER_SEEDS[index]),
        "state_dict": CODE12_ALL_STATES[index],
    } for index in final_expert_indices],
    "event_gate": {
        "state_dicts": CODE12_GATE_STATES,
        "seeds": list(CODE12_GATE_SEEDS),
        "feature_mean": CODE12_GATE_MEAN.tolist(),
        "feature_std": CODE12_GATE_STD.tolist(),
        "event_q75": CODE12_EVENT_Q75.tolist(),
        "event_q90": CODE12_EVENT_Q90.tolist(),
        "risk_formula": {"large": 0.65, "extreme": 0.35},
    },
    "routing_scales": {
        "uncertainty_q75": CODE12_U_SCALE.tolist(),
        "predicted_delta_q75": CODE12_D_SCALE.tolist(),
        "physics_wide_q75": CODE12_PW_SCALE.tolist(),
    },
    "score_definitions": used_score_definitions,
    "decoder_config": copy.deepcopy(CODE4_ACTIVE_CHECKPOINT["decoder_config"]),
    "coronal_feature_config": copy.deepcopy(
        CODE4_ACTIVE_CHECKPOINT["coronal_feature_config"]
    ),
    "coronal_geometry": copy.deepcopy(CODE4_ACTIVE_CHECKPOINT["coronal_geometry"]),
    "prediction_mode": CODE4_ACTIVE_CHECKPOINT["prediction_mode"],
    "initialization": CODE4_ACTIVE_CHECKPOINT["initialization"],
    "validation_rmse_km_s": float(CODE13_FINAL_RECORD["full_rmse"]),
    "oof_meta_rmse_km_s": float(CODE13_FINAL_META["oof_meta_rmse"]),
    "fold_rmse_km_s": CODE13_FINAL_RECORD["fold_rmse"].tolist(),
    "fold_wins": int(CODE13_FINAL_RECORD["fold_wins"]),
    "worst_fold_degradation": float(
        CODE13_FINAL_RECORD["worst_fold_degradation"]
    ),
    "parameter_mode_fraction": float(
        CODE13_FINAL_META["parameter_mode_fraction"]
    ),
    "routing_validation_coverage": float(CODE13_FINAL_RECORD["coverage"]),
    "last_shot": bool(CODE13_LAST_SHOT),
    "image_size": IMAGE_SIZE,
    "channels": list(CHANNELS),
    "wind_feature_dim": WIND_FEATURE_DIM,
    "target_horizons": TARGET_HORIZONS,
    "external_data_used": False,
    "pretrained_weights_used": False,
    "train_validation_combined": False,
    "test_used_for_selection": False,
    "public_score_used_for_fitting": False,
}
code13_validate_checkpoint(CODE13_CHECKPOINT)
atomic_torch_save(CODE13_CHECKPOINT, CODE7_MODEL_PATH)


def code13_infer_member_states(variants, states, loader):
    predictions, expected_rows = [], None
    assert len(variants) == len(states)
    for variant, state in zip(variants, states):
        model = build_v35_model(variant)
        model.load_state_dict(state, strict=True)
        prediction, rows = predict_single_model(model, loader)
        if expected_rows is None:
            expected_rows = rows
        else:
            assert np.array_equal(rows, expected_rows)
        predictions.append(prediction)
        del model
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    return np.asarray(predictions, dtype=np.float64), expected_rows


def code13_predict_gate_states(states, scaled_features):
    predictions = []
    for state in states:
        model = Code12EventGate().to(DEVICE)
        model.load_state_dict(state, strict=True)
        predictions.append(code12_predict_gate(model, scaled_features))
        del model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return np.mean(predictions, axis=0).astype(np.float64)


def code13_test_score_arrays(checkpoint, base_stack, base_prediction, persistence, gate_probability):
    member_std = base_stack.std(axis=0)
    delta_abs = np.abs(base_prediction - persistence)
    uncertainty = np.clip(
        member_std / np.asarray(
            checkpoint["routing_scales"]["uncertainty_q75"]
        )[None, :], 0.0, 1.0,
    )
    predicted_delta = np.clip(
        delta_abs / np.asarray(
            checkpoint["routing_scales"]["predicted_delta_q75"]
        )[None, :], 0.0, 1.0,
    )
    event_risk = 0.65 * gate_probability[:, :12] + 0.35 * gate_probability[:, 12:]
    arrays = {
        "event_uncertainty": event_risk * uncertainty,
        "event_extreme": gate_probability[:, 12:],
    }
    for score_kind, definition in checkpoint["score_definitions"].items():
        if definition["kind"] != "event_uncertainty_refined":
            continue
        event_coefficient = float(definition["event"])
        uncertainty_coefficient = float(definition["uncertainty"])
        delta_coefficient = float(definition["delta"])
        total = event_coefficient + uncertainty_coefficient + delta_coefficient
        local_mix = (
            event_coefficient * event_risk
            + uncertainty_coefficient * uncertainty
            + delta_coefficient * predicted_delta
        ) / total
        arrays[score_kind] = (event_risk * uncertainty) * (
            float(definition["local_mix_offset"])
            + float(definition["local_mix_scale"]) * local_mix
        )
    return arrays


def code13_reconstruct_test_from_checkpoint(checkpoint, loader):
    checkpoint = code13_validate_checkpoint(checkpoint)
    base_stack, rows = code13_infer_member_states(
        checkpoint["base_variants"], checkpoint["base_model_state_dicts"], loader
    )
    assert base_stack.shape == (3, 3868, 12)
    assert np.array_equal(rows, test_index)
    persistence = np.repeat(
        test_inputs.iloc[test_index][[WIND_COLUMNS[-1]]].to_numpy(np.float64),
        TARGET_HORIZONS, axis=1,
    )
    parent = code8_member_ensemble(
        base_stack, np.asarray(checkpoint["code8_horizon_weights"]),
        persistence, np.ones(TARGET_HORIZONS),
    )
    base_prediction = persistence + np.asarray(
        checkpoint["code8_final_alpha"], dtype=np.float64
    )[None, :] * (parent - persistence)
    test_features = code12_build_gate_features(
        test_inputs, test_index, test_coronal_array, test_coronal_index
    )
    scaled_features = (
        (test_features - np.asarray(
            checkpoint["event_gate"]["feature_mean"], dtype=np.float32
        )) / np.asarray(
            checkpoint["event_gate"]["feature_std"], dtype=np.float32
        )
    ).astype(np.float32)
    gate_probability = code13_predict_gate_states(
        checkpoint["event_gate"]["state_dicts"], scaled_features
    )
    score_arrays = code13_test_score_arrays(
        checkpoint, base_stack, base_prediction, persistence, gate_probability
    )
    expert_predictions = {}
    for member in checkpoint["expert_members"]:
        prediction_stack, expert_rows = code13_infer_member_states(
            [member["variant"]], [member["state_dict"]], loader
        )
        assert np.array_equal(expert_rows, rows)
        expert_predictions[int(member["expert_index"])] = prediction_stack[0]
    prediction, changed = code13_apply_routes(
        {"kind": "frozen", "routes": checkpoint["routes"]},
        base_prediction, expert_predictions, score_arrays,
    )
    del test_features, scaled_features, gate_probability, base_stack
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return prediction, rows, float(changed.mean())


test_dataset_code13, test_loader_code13 = make_stable_test_loader()
reloaded_code13 = code13_validate_checkpoint(
    safe_torch_load(CODE7_MODEL_PATH, map_location="cpu")
)
CODE13_TEST_PREDICTION, CODE13_TEST_ROWS, CODE13_TEST_COVERAGE = (
    code13_reconstruct_test_from_checkpoint(reloaded_code13, test_loader_code13)
)
assert CODE13_TEST_PREDICTION.shape == (3868, 12)
assert np.array_equal(CODE13_TEST_ROWS, test_index)
assert np.isfinite(CODE13_TEST_PREDICTION).all()
code13_submission = pd.DataFrame(CODE13_TEST_PREDICTION, columns=TARGET_COLUMNS)
code13_submission.insert(0, "sample_id", test_ids["sample_id"].tolist())
assert code13_submission.shape == (3868, 13)
assert code13_submission.columns.tolist() == EXPECTED_SUBMISSION_COLUMNS
assert code13_submission["sample_id"].is_unique
assert code13_submission["sample_id"].tolist() == test_ids["sample_id"].tolist()
assert not code13_submission[TARGET_COLUMNS].isna().any().any()
assert np.isfinite(code13_submission[TARGET_COLUMNS].to_numpy(np.float64)).all()
atomic_csv_save(code13_submission, CODE7_SUBMISSION_PATH)
published_code13 = pd.read_csv(CODE7_SUBMISSION_PATH)
assert published_code13.shape == (3868, 13)
assert published_code13.columns.tolist() == EXPECTED_SUBMISSION_COLUMNS
assert published_code13["sample_id"].tolist() == test_ids["sample_id"].tolist()
assert np.allclose(
    published_code13[TARGET_COLUMNS].to_numpy(np.float64),
    CODE13_TEST_PREDICTION, rtol=1e-6, atol=5e-5,
)


def code13_masked_rmse(prediction, mask):
    return float(np.sqrt(np.square(prediction - CODE12_TARGET)[mask].mean()))


CODE13_FINAL_REGIME_ROWS = []
for regime in (
    "change_small", "change_medium", "change_large", "change_extreme",
    "speed_normal", "speed_high", "speed_extreme",
):
    regime_mask = CODE12_REGIME_MASKS[regime]
    CODE13_FINAL_REGIME_ROWS.append({
        "regime": regime,
        "code8_rmse": code13_masked_rmse(CODE12_P8, regime_mask),
        "code12_route_rmse": code13_masked_rmse(
            CODE13_REFERENCE_PREDICTION, regime_mask
        ),
        "code13_rmse": code13_masked_rmse(
            CODE13_FINAL_VAL_PREDICTION, regime_mask
        ),
    })
CODE13_CODE8_HORIZONS = np.sqrt(np.mean(np.square(
    CODE12_P8 - CODE12_TARGET
), axis=0))
CODE13_CODE12_HORIZONS = np.sqrt(np.mean(np.square(
    CODE13_REFERENCE_PREDICTION - CODE12_TARGET
), axis=0))
CODE13_FINAL_HORIZONS = np.sqrt(np.mean(np.square(
    CODE13_FINAL_VAL_PREDICTION - CODE12_TARGET
), axis=0))

if CODE13_FINAL_RECORD["full_rmse"] < 62.75:
    CODE13_DECISION = "VERY STRONG UPGRADE"
elif CODE13_FINAL_RECORD["full_rmse"] < 62.77:
    CODE13_DECISION = "STRONG UPGRADE"
elif CODE13_FINAL_RECORD["full_rmse"] < 62.80:
    CODE13_DECISION = "USEFUL UPGRADE"
elif CODE13_FINAL_RECORD["full_rmse"] < CODE13_LIVE_CODE12_ROUTE:
    CODE13_DECISION = "SMALL UPGRADE"
else:
    CODE13_DECISION = "NO UPGRADE"

stage_a_frozen = CODE13_STAGE_A_META["frozen_record"]
stage_b_frozen = CODE13_STAGE_B_META["frozen_record"]
stage_c_frozen = (
    CODE13_STAGE_C_META["frozen_record"] if CODE13_STAGE_C_META else None
)
print("\n" + "=" * 100)
print("CODE13 FINAL LOCAL ROUTING REPORT")
print("=" * 100)
print("\nBASELINE")
print("CODE8:", f"{CODE12_LIVE_CODE8:.6f}")
print("CODE12 ROUTING:", f"{CODE13_LIVE_CODE12_ROUTE:.6f}")
print("\nSTAGE A — CH TOPOLOGY")
print("best coverage:", f"{stage_a_frozen['coverage']:.6f}")
print("best threshold:", stage_a_frozen["config"]["routes"][0]["threshold"])
print("best weight:", stage_a_frozen["config"]["routes"][0]["weight"])
print("horizon group:", stage_a_frozen["config"]["routes"][0]["horizon_group"])
print("Full RMSE:", f"{stage_a_frozen['full_rmse']:.6f}")
print("OOF:", f"{CODE13_STAGE_A_META['oof_meta_rmse']:.6f}")
print("fold wins:", f"{stage_a_frozen['fold_wins']}/5")
print("worst fold:", f"{stage_a_frozen['worst_fold_degradation']:+.6f}")
print("stability:", f"{CODE13_STAGE_A_META['parameter_mode_fraction']:.2f}")
print("\nSTAGE B — SCORE REFINEMENT")
stage_b_route = stage_b_frozen["config"]["routes"][0]
stage_b_definition = CODE13_SCORE_DEFINITIONS[stage_b_route["score_kind"]]
print("event coefficient:", stage_b_definition.get("event"))
print("uncertainty coefficient:", stage_b_definition.get("uncertainty"))
print("delta coefficient:", stage_b_definition.get("delta"))
print("Full:", f"{stage_b_frozen['full_rmse']:.6f}")
print("OOF:", f"{CODE13_STAGE_B_META['oof_meta_rmse']:.6f}")
print("fold wins:", f"{stage_b_frozen['fold_wins']}/5")
print("worst:", f"{stage_b_frozen['worst_fold_degradation']:+.6f}")
print("\nSTAGE C — HSS OPTIONAL")
print("used:", "yes" if len(CODE13_FINAL_CONFIG["routes"]) == 2 else "no")
if stage_c_frozen is not None:
    print("best candidate coverage:", f"{stage_c_frozen['coverage']:.6f}")
    print("best candidate Full:", f"{stage_c_frozen['full_rmse']:.6f}")
    print("best candidate OOF:", f"{CODE13_STAGE_C_META['oof_meta_rmse']:.6f}")
print("\nFINAL")
print("expert(s):", [route["expert"] for route in final_routes])
print("routing rule:", final_routes)
print("total coverage:", f"{CODE13_FINAL_RECORD['coverage']:.6f}")
print("weights:", [route["weight"] for route in final_routes])
print("Validation RMSE:", f"{CODE13_FINAL_RECORD['full_rmse']:.6f}")
print("Improvement vs CODE8:", f"{CODE12_LIVE_CODE8 - CODE13_FINAL_RECORD['full_rmse']:+.6f}")
print("Improvement vs CODE12:", f"{CODE13_LIVE_CODE12_ROUTE - CODE13_FINAL_RECORD['full_rmse']:+.6f}")
print("OOF META:", f"{CODE13_FINAL_META['oof_meta_rmse']:.6f}")
print("fold wins:", f"{CODE13_FINAL_RECORD['fold_wins']}/5")
print("worst fold:", f"{CODE13_FINAL_RECORD['worst_fold_degradation']:+.6f}")
print("parameter stability:", f"{CODE13_FINAL_META['parameter_mode_fraction']:.2f}")
print("\nREGIME")
print(pd.DataFrame(CODE13_FINAL_REGIME_ROWS).to_string(
    index=False, float_format=lambda value: f"{value:.6f}"
))
print("\nHORIZON")
for horizon, code8_value, code12_value, code13_value in zip(
    range(6, 73, 6), CODE13_CODE8_HORIZONS,
    CODE13_CODE12_HORIZONS, CODE13_FINAL_HORIZONS,
):
    print(
        f"{horizon:2d}h: CODE8 {code8_value:.6f} | "
        f"CODE12 {code12_value:.6f} | CODE13 {code13_value:.6f}"
    )
print("\nLAST_SHOT:", "YES" if CODE13_LAST_SHOT else "NO")
print("DECISION:", CODE13_DECISION)
print("checkpoint disk reload + Test reconstruction: PASS")
print("submission shape/order/finite: PASS")
print("=" * 100)

del test_loader_code13, test_dataset_code13
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


## CODE13 compliance and execution contract

- **No new backbone/model family:** CODE13 reuses the completed CODE8 champion,
  CH Topology, optional HSS Peak, and the Train-only CODE12 event gate.
- **Training source:** Competition Train only.  Official Validation is used only
  for routing/meta-selection and diagnostics; Test is final inference only.
- **No external data, pretrained weights, pseudo-labels, leaderboard correction,
  PCA, tree/linear/kNN/TTA sweep, or backbone retraining is introduced here.**
- The old CODE12 `0.10` gain fallback is intentionally not reused.  If local
  refinement fails, the reproduced `62.820879` CH route is the frozen fallback.
- `model.pth` is reloaded from disk before Test inference, and submission shape,
  columns, ID order, uniqueness, NaN, and Inf are asserted before completion.
